# RFP 기반 RAG 시스템 Generation 파트 모델 및 프롬프트 실험

본 노트북은 RFP 기반 RAG 시스템에서 Generation 파트에 사용할 로컬 LLM 후보 모델과 프롬프트 기준안을 정리하기 위한 실험 노트북입니다.

Generation 파트에서는 검색된 문서 내용을 바탕으로 사용자의 질문에 답변하거나, 질문이 불완전한 경우 필요한 추가 정보를 안내해야 합니다. 또한 비교 요청, 계산 요청, 제안서 작성 요청처럼 질문 유형이 다양한 상황에서도 문서에 근거한 안정적인 응답을 생성해야 합니다.

본 실험의 목적은 다음과 같습니다.

- RFP 문서 기반 QA에 적합한 로컬 LLM 후보 비교
- 정보부족 판단 및 질문 재작성 가능 여부 확인
- 사용자에게 보여줄 수 있는 자연스러운 정보부족 안내 응답 설계
- 모델별 응답 속도와 출력 안정성 비교
- 후처리 함수의 필요성과 한계 확인
- RAG 연동 전 Generation 파트의 입력·출력 기준안 정리

본 노트북은 실험 결과를 다시 실행할 수 있도록 구성하며, 출력 결과를 확인한 뒤 해석과 최종 판단을 작성합니다. 따라서 실행 결과가 이전에 관찰한 결과와 달라질 경우, 최종 해석도 재실행 결과를 기준으로 수정합니다.

## 실험 진행 원칙

본 실험에서는 모델별 비교가 가능하도록 다음 원칙을 적용합니다.

- 동일한 테스트 질문을 여러 모델에 적용합니다.
- 모델 비교 시 프롬프트와 테스트 질문은 고정하고, 모델만 변경합니다.
- 프롬프트 비교 시 모델과 테스트 질문은 고정하고, 프롬프트 구조만 변경합니다.
- `temperature`는 `0.0`으로 고정하여 응답의 무작위성을 줄입니다.
- `success=True`는 모델 호출 성공을 의미하며, 응답 품질이 통과했다는 의미는 아닙니다.
- 응답 품질은 출력 내용을 직접 확인하고 수동 평가로 판단합니다.
- 최종 프롬프트는 실제 RAG 검색 결과 context 형식이 확정된 이후 다시 조정합니다.

In [7]:
import time
import re
import subprocess
import requests
import pandas as pd

In [8]:
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_GENERATE_URL = f"{OLLAMA_BASE_URL}/api/generate"
OLLAMA_TAGS_URL = f"{OLLAMA_BASE_URL}/api/tags"

def check_ollama_server(timeout=5):
    """
    Ollama 서버가 정상적으로 실행 중인지 확인합니다.
    """
    try:
        response = requests.get(OLLAMA_TAGS_URL, timeout=timeout)
        
        if response.status_code == 200:
            print("Ollama 서버 연결: 정상")
            return response.json()
        
        print(f"Ollama 서버 응답 이상: status_code={response.status_code}")
        print(response.text[:300])
        return None
    
    except Exception as e:
        print("Ollama 서버 연결 실패")
        print(e)
        return None

ollama_server_info = check_ollama_server()

Ollama 서버 연결: 정상


In [ ]:
def show_ollama_models():
    """
    현재 로컬 Ollama에 설치된 모델 목록을 확인합니다.
    """
    try:
        result = subprocess.run(
            ["ollama", "list"],
            capture_output=True,
            text=True,
            encoding="utf-8"
        )
        
        print(result.stdout)
        return result.stdout
    
    except Exception as e:
        print("ollama list 실행 실패")
        print(e)
        return ""

ollama_list_text = show_ollama_models()

In [10]:
target_models = [
    "gemma3:4b",
    "qwen2.5:3b",
    "llama3.2:3b",
    "exaone3.5:2.4b",
    "phi3.5"
]

model_check_results = []

for model in target_models:
    installed = model in ollama_list_text
    
    model_check_results.append({
        "model": model,
        "installed": installed
    })

model_check_df = pd.DataFrame(model_check_results)
model_check_df

,model,installed
0,gemma3:4b,True
1,qwen2.5:3b,True
2,llama3.2:3b,True
3,exaone3.5:2.4b,True
4,phi3.5,True


In [11]:
def ask_ollama(prompt, model, temperature=0.0, timeout=120):
    """
    Ollama /api/generate를 사용해 모델 응답을 생성합니다.
    
    success=True는 API 호출 성공을 의미하며,
    응답 품질이 평가 기준을 통과했다는 의미는 아닙니다.
    """
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": temperature
        }
    }
    
    start_time = time.time()
    
    try:
        response = requests.post(
            OLLAMA_GENERATE_URL,
            json=payload,
            timeout=timeout
        )
        
        elapsed = time.time() - start_time
        
        if response.status_code != 200:
            return {
                "success": False,
                "model": model,
                "answer": "",
                "elapsed": elapsed,
                "error": f"status_code={response.status_code}, text={response.text[:300]}"
            }
        
        data = response.json()
        
        return {
            "success": True,
            "model": model,
            "answer": data.get("response", "").strip(),
            "elapsed": elapsed,
            "error": ""
        }
    
    except Exception as e:
        elapsed = time.time() - start_time
        
        return {
            "success": False,
            "model": model,
            "answer": "",
            "elapsed": elapsed,
            "error": str(e)
        }

## 전체 모델 간단 호출 테스트

본 단계에서는 실험 대상 모델들이 현재 로컬 환경에서 정상적으로 호출되는지 확인합니다.

이 테스트는 모델 품질을 평가하기 위한 것이 아니라, 각 모델이 정상적으로 응답을 생성할 수 있는지 확인하는 예비 테스트입니다.

실제 모델 품질은 이후 문서 기반 QA, 정보부족 판단, 질문 재작성, 사용자 응답형 안내 테스트를 통해 별도로 평가합니다.

In [12]:
smoke_test_prompt = """
다음 질문에 한국어 존댓말로 한 문장만 답변하세요.

질문: RFP 문서 기반 QA 실험에서 금액 단위 보존이 중요한 이유는 무엇인가요?
"""

smoke_test_results = []

for model in target_models:
    print(f"테스트 중: {model}")
    
    result = ask_ollama(
        prompt=smoke_test_prompt,
        model=model,
        temperature=0.0,
        timeout=120
    )
    
    smoke_test_results.append({
        "model": model,
        "success": result["success"],
        "elapsed": round(result["elapsed"], 3),
        "answer": result["answer"],
        "error": result["error"]
    })

smoke_test_df = pd.DataFrame(smoke_test_results)
smoke_test_df

테스트 중: gemma3:4b
테스트 중: qwen2.5:3b
테스트 중: llama3.2:3b
테스트 중: exaone3.5:2.4b
테스트 중: phi3.5


,model,success,elapsed,answer,error
0,gemma3:4b,True,6.955,RFP 문서 기반 QA 실험에서는 금액 단위 보존이 정확한 비용 분석 및 평가를 위...,
1,qwen2.5:3b,True,5.363,RFP 문서 기반 QA 실험에서 금액 단위를 보존하는 것은 정확한 정보 제공과 이해...,
2,llama3.2:3b,True,11.425,금액 단위 보존은 RFP 문서 기반 QA 실험에서 중요합니다 because 금액을 ...,
3,exaone3.5:2.4b,True,9.437,RFP 문서 기반 QA 실험에서 금액 단위의 정확한 보존은 예산 관리와 비용 효율성...,
4,phi3.5,True,10.026,"금액 단위를 보존하는 이유는, RFP 문서에서의 QA 품질을 분명히 나타내고, 제공...",


### 전체 모델 간단 호출 테스트 결과 해석

실험 대상 모델 5종을 대상으로 동일한 짧은 질문을 입력하여 Ollama 호출 가능 여부를 확인하였습니다.

확인 결과 `gemma3:4b`, `qwen2.5:3b`, `llama3.2:3b`, `exaone3.5:2.4b`, `phi3.5` 모두 `success=True`가 반환되어 API 호출 자체는 정상적으로 수행되었습니다. 따라서 현재 로컬 환경에서 실험 대상 모델들은 모두 호출 가능한 상태로 확인되었습니다.

응답 시간은 모델별로 차이가 있었습니다. `qwen2.5:3b`가 약 5.363초로 가장 빠르게 응답하였고, `gemma3:4b`는 약 6.955초, `exaone3.5:2.4b`는 약 9.437초, `phi3.5`는 약 10.026초, `llama3.2:3b`는 약 11.425초가 소요되었습니다.

응답 내용 측면에서는 대부분의 모델이 금액 단위 보존의 필요성을 한국어로 설명하였습니다. 다만 `llama3.2:3b`는 응답 중간에 `because`와 같은 영어 표현이 섞여 출력되어, 간단 호출 테스트 단계에서도 외국어 혼입 가능성이 확인되었습니다.

따라서 이 테스트를 통해 모든 모델의 호출 가능 여부는 확인되었지만, 응답 품질은 모델별로 차이가 있을 수 있음을 확인하였습니다. 이후 문서 기반 QA, 정보부족 판단, 질문 재작성, 사용자 응답형 안내 테스트에서 각 모델의 실제 활용 가능성을 별도로 평가합니다.

## 문서 기반 QA / 정보부족 판단 / 질문 재작성 테스트 설계

Generation 파트는 단순히 질문에 답변하는 기능만 담당하지 않습니다. RFP 기반 RAG 시스템에서는 사용자의 질문이 문서에 근거해 답변 가능한지, 추가 정보가 필요한지, 검색에 적합한 질문으로 재작성할 수 있는지를 함께 판단해야 합니다.

따라서 본 실험에서는 다음 세 가지 유형의 테스트를 구성합니다.

| 테스트 유형 | 목적 |
|---|---|
| 문서 기반 QA 테스트 | 제공된 문서 context를 근거로 정확하게 답변하는지 확인 |
| 정보부족 판단 테스트 | 질문만으로 답변하기 어려운 경우 필요한 추가 정보를 안내하는지 확인 |
| 질문 재작성 테스트 | 사용자의 불완전한 질문과 추가 답변을 바탕으로 검색에 적합한 질문으로 재작성할 수 있는지 확인 |

특히 RFP 문서에서는 금액, 기간, 평가 기준, 요구사항 목록과 같은 정보가 중요하므로, 숫자 보존, 항목 누락 방지, 임의 추론 방지를 중점적으로 확인합니다.

In [13]:
qa_prompt = """
너는 RFP 문서를 분석하는 AI입니다.
아래 문서 내용만 근거로 사용자의 질문에 답변하세요.

규칙:
1. 문서에 있는 내용만 사용하세요.
2. 문서에 없는 내용은 추측하지 마세요.
3. 금액, 예산, 비율, 점수, 기간 등 숫자는 원문 표기를 그대로 유지하세요.
4. 억/조 단위로 임의 변환하지 마세요.
5. 목록형 요구사항은 항목을 누락하지 말고 모두 포함하세요.
6. 가격점수 계산은 필요한 산식과 입력값이 모두 있을 때만 계산하세요.
7. 계산에 필요한 정보가 부족하면 계산하지 말고 부족한 정보를 말하세요.
8. 반드시 한국어로 답변하세요.
9. 외국어를 섞지 마세요.

[문서 내용]
{context}

[질문]
{question}

[답변]
"""

In [14]:
clarification_prompt = """
너는 RFP 기반 질의응답 시스템의 질문 완전성 검사기입니다.
사용자의 질문이 현재 정보만으로 답변 가능한지 판단하세요.

규칙:
1. 질문이 너무 모호하면 추가 확인 필요로 판단하세요.
2. 입찰 적합도, 수행 가능성, 제안서 작성, 인력 계획 등은 회사 정보가 필요할 수 있습니다.
3. 필요한 정보가 부족하면 실제로 필요한 정보를 구체적으로 나열하세요.
4. 추가 확인이 필요한데 '없습니다'라고 쓰지 마세요.
5. 문서에 없는 정보를 추측하지 마세요.
6. 반드시 한국어로만 답변하세요.
7. 외국어를 섞지 마세요.

출력 형식:
판단: 답변 가능 / 추가 확인 필요
추가로 확인할 정보:
- 필요한 정보 1
- 필요한 정보 2

[사용자 질문]
{question}
"""

In [15]:
rewrite_prompt = """
너는 사용자의 이전 질문과 추가 답변을 바탕으로 RFP 검색용 질문을 재작성하는 AI입니다.

규칙:
1. 이전 질문과 추가 답변을 합쳐서 더 명확한 질문으로 재작성하세요.
2. 추가 답변만으로도 정보가 부족하면 재작성 가능으로 판단하지 마세요.
3. 비교 질문은 비교 대상이 명확해야 재작성 가능합니다.
4. 입찰 적합도 판단은 회사 정보가 없으면 추가 확인 필요로 판단하세요.
5. 필요한 정보가 부족하면 실제로 필요한 정보를 구체적으로 나열하세요.
6. 반드시 한국어로만 답변하세요.
7. 외국어를 섞지 마세요.

출력 형식:
판단: 재작성 가능 / 추가 확인 필요
재작성 질문:
추가로 확인할 정보:

[이전 질문]
{previous_question}

[추가 답변]
{additional_answer}
"""

In [16]:
qa_test_questions = [
    {
        "case_id": "QA-01",
        "category": "금액 단위 보존",
        "context": """
사업예산: 11,270,000,000원
본 사업의 예산은 부가가치세를 포함한 금액입니다.
""",
        "question": "이 사업의 예산은 얼마인가요?",
        "check_points": ["11,270,000,000원", "억/조 임의 환산 금지"]
    },
    {
        "case_id": "QA-02",
        "category": "항목 누락 방지",
        "context": """
구축 대상 시스템은 다음과 같습니다.
1. 포털 시스템
2. 학사 시스템
3. 수강신청 시스템
4. 전자결재 시스템
5. 규정관리 시스템
6. 검색 시스템
7. 챗봇 시스템
8. 모바일 시스템
9. 통계 시스템
""",
        "question": "구축 대상 시스템 목록을 모두 알려주세요.",
        "check_points": ["9개 항목 모두 포함", "항목 누락 없음"]
    },
    {
        "case_id": "QA-03",
        "category": "가격점수 가드레일",
        "context": """
가격평가는 입찰가격 평점산식에 따라 산정합니다.
다만 현재 문서에는 예정가격, 입찰가격, 평점산식의 구체적인 입력값이 모두 제시되어 있지 않습니다.
""",
        "question": "가격점수를 계산해주세요.",
        "check_points": ["계산하지 않음", "필요 정보 부족 안내"]
    }
]

qa_test_df = pd.DataFrame(qa_test_questions)
qa_test_df

,case_id,category,context,question,check_points
0,QA-01,금액 단위 보존,"\n사업예산: 11,270,000,000원\n본 사업의 예산은 부가가치세를 포함한 ...",이 사업의 예산은 얼마인가요?,"[11,270,000,000원, 억/조 임의 환산 금지]"
1,QA-02,항목 누락 방지,\n구축 대상 시스템은 다음과 같습니다.\n1. 포털 시스템\n2. 학사 시스템\n...,구축 대상 시스템 목록을 모두 알려주세요.,"[9개 항목 모두 포함, 항목 누락 없음]"
2,QA-03,가격점수 가드레일,"\n가격평가는 입찰가격 평점산식에 따라 산정합니다.\n다만 현재 문서에는 예정가격,...",가격점수를 계산해주세요.,"[계산하지 않음, 필요 정보 부족 안내]"


In [17]:
clarification_test_questions = [
    {
        "case_id": "CL-01",
        "category": "입찰 적합도 정보부족",
        "question": "우리 회사가 이 입찰에 적합한지 봐줘.",
        "check_points": ["회사 정보 필요", "수행 실적", "보유 인력", "기술 역량"]
    },
    {
        "case_id": "CL-02",
        "category": "제안서 작성 정보부족",
        "question": "제안서에 쓸 만하게 정리해줘.",
        "check_points": ["사업 범위", "회사 정보", "수행 전략", "실적"]
    },
    {
        "case_id": "CL-03",
        "category": "모호한 요약 요청",
        "question": "이거 핵심만 뽑아줘.",
        "check_points": ["대상 문서 또는 내용 필요"]
    },
    {
        "case_id": "CL-04",
        "category": "가격점수 정보부족",
        "question": "가격점수 계산해줘.",
        "check_points": ["입찰가격", "예정가격", "산식"]
    }
]

clarification_test_df = pd.DataFrame(clarification_test_questions)
clarification_test_df

,case_id,category,question,check_points
0,CL-01,입찰 적합도 정보부족,우리 회사가 이 입찰에 적합한지 봐줘.,"[회사 정보 필요, 수행 실적, 보유 인력, 기술 역량]"
1,CL-02,제안서 작성 정보부족,제안서에 쓸 만하게 정리해줘.,"[사업 범위, 회사 정보, 수행 전략, 실적]"
2,CL-03,모호한 요약 요청,이거 핵심만 뽑아줘.,[대상 문서 또는 내용 필요]
3,CL-04,가격점수 정보부족,가격점수 계산해줘.,"[입찰가격, 예정가격, 산식]"


In [18]:
rewrite_test_cases = [
    {
        "case_id": "RW-01",
        "category": "비교 질문 재작성 가능",
        "previous_question": "이 사업과 비교해주세요.",
        "additional_answer": "경희대 산학협력단 정보시스템 운영 용역과 비교해주세요.",
        "check_points": ["재작성 가능", "비교 대상 포함"]
    },
    {
        "case_id": "RW-02",
        "category": "입찰 적합도 추가 정보 부족",
        "previous_question": "우리 회사가 이 입찰에 적합한지 봐줘.",
        "additional_answer": "그냥 알아서 해줘.",
        "check_points": ["추가 확인 필요", "회사 정보 필요", "없습니다 금지"]
    },
    {
        "case_id": "RW-03",
        "category": "수행 전략 재작성 가능",
        "previous_question": "수행 전략 작성해줘.",
        "additional_answer": "차세대 포털·학사 정보시스템 구축사업 기준으로 작성해줘.",
        "check_points": ["재작성 가능", "사업명 포함"]
    },
    {
        "case_id": "RW-04",
        "category": "비교 대상 부족",
        "previous_question": "이 사업이랑 다른 사업 비교해줘.",
        "additional_answer": "그냥 비슷한 걸로 해줘.",
        "check_points": ["추가 확인 필요", "비교 대상 필요"]
    }
]

rewrite_test_df = pd.DataFrame(rewrite_test_cases)
rewrite_test_df

,case_id,category,previous_question,additional_answer,check_points
0,RW-01,비교 질문 재작성 가능,이 사업과 비교해주세요.,경희대 산학협력단 정보시스템 운영 용역과 비교해주세요.,"[재작성 가능, 비교 대상 포함]"
1,RW-02,입찰 적합도 추가 정보 부족,우리 회사가 이 입찰에 적합한지 봐줘.,그냥 알아서 해줘.,"[추가 확인 필요, 회사 정보 필요, 없습니다 금지]"
2,RW-03,수행 전략 재작성 가능,수행 전략 작성해줘.,차세대 포털·학사 정보시스템 구축사업 기준으로 작성해줘.,"[재작성 가능, 사업명 포함]"
3,RW-04,비교 대상 부족,이 사업이랑 다른 사업 비교해줘.,그냥 비슷한 걸로 해줘.,"[추가 확인 필요, 비교 대상 필요]"


### 테스트 케이스 설계 해석

본 실험에서는 Generation 파트에서 자주 발생할 수 있는 오류를 확인하기 위해 문서 기반 QA, 정보부족 판단, 질문 재작성 테스트 케이스를 구성하였습니다.

문서 기반 QA 테스트에서는 금액 단위 보존, 목록형 항목 누락 방지, 가격점수 계산 가드레일을 확인합니다. 이는 RFP 문서에서 예산, 시스템 목록, 평가 기준과 같은 정보가 잘못 처리될 경우 실제 입찰 분석 결과에 큰 영향을 줄 수 있기 때문입니다.

정보부족 판단 테스트에서는 사용자의 질문만으로 답변하기 어려운 상황에서 모델이 필요한 추가 정보를 구체적으로 안내하는지 확인합니다. 특히 입찰 적합도 판단, 제안서 작성, 가격점수 계산과 같은 요청은 회사 정보나 산식, 입력값이 필요할 수 있으므로 임의로 답변하지 않는 것이 중요합니다.

질문 재작성 테스트에서는 사용자의 이전 질문과 추가 답변을 바탕으로 검색에 적합한 질문을 만들 수 있는지 확인합니다. 비교 대상이 명확하지 않거나 회사 정보가 부족한 상황에서는 재작성 가능으로 판단하지 않고 추가 확인이 필요하다고 판단해야 합니다.

## 모델별 문서 기반 QA 테스트 실행

이 단계에서는 실험 대상 모델들이 제공된 문서 context만을 근거로 질문에 답변하는지 확인합니다.

주요 확인 항목은 다음과 같습니다.

- 금액 단위를 원문 그대로 유지하는지
- 억/조 단위로 임의 환산하지 않는지
- 목록형 항목을 누락하지 않는지
- 계산에 필요한 정보가 부족할 때 임의 계산하지 않는지

In [19]:
qa_results = []

for model in target_models:
    print(f"QA 테스트 실행 중: {model}")
    
    for item in qa_test_questions:
        prompt = qa_prompt.format(
            context=item["context"],
            question=item["question"]
        )
        
        result = ask_ollama(
            prompt=prompt,
            model=model,
            temperature=0.0,
            timeout=120
        )
        
        qa_results.append({
            "model": model,
            "test_type": "QA",
            "case_id": item["case_id"],
            "category": item["category"],
            "question": item["question"],
            "check_points": ", ".join(item["check_points"]),
            "success": result["success"],
            "elapsed": round(result["elapsed"], 3),
            "answer": result["answer"],
            "error": result["error"]
        })

qa_results_df = pd.DataFrame(qa_results)
qa_results_df

QA 테스트 실행 중: gemma3:4b
QA 테스트 실행 중: qwen2.5:3b
QA 테스트 실행 중: llama3.2:3b
QA 테스트 실행 중: exaone3.5:2.4b
QA 테스트 실행 중: phi3.5


,model,test_type,case_id,category,question,check_points,success,elapsed,answer,error
0,gemma3:4b,QA,QA-01,금액 단위 보존,이 사업의 예산은 얼마인가요?,"11,270,000,000원, 억/조 임의 환산 금지",True,8.557,"이 사업의 예산은 부가가치세를 포함한 11,270,000,000원입니다.",
1,gemma3:4b,QA,QA-02,항목 누락 방지,구축 대상 시스템 목록을 모두 알려주세요.,"9개 항목 모두 포함, 항목 누락 없음",True,3.360,구축 대상 시스템은 다음과 같습니다.\n1. 포털 시스템\n2. 학사 시스템\n3....,
2,gemma3:4b,QA,QA-03,가격점수 가드레일,가격점수를 계산해주세요.,"계산하지 않음, 필요 정보 부족 안내",True,3.166,"현재 문서에는 예정가격, 입찰가격, 평점산식의 구체적인 입력값이 모두 제시되어 있지...",
3,qwen2.5:3b,QA,QA-01,금액 단위 보존,이 사업의 예산은 얼마인가요?,"11,270,000,000원, 억/조 임의 환산 금지",True,5.375,"이 사업의 예산은 11,270,000,000원입니다.",
4,qwen2.5:3b,QA,QA-02,항목 누락 방지,구축 대상 시스템 목록을 모두 알려주세요.,"9개 항목 모두 포함, 항목 누락 없음",True,2.919,구축 대상 시스템 목록은 다음과 같습니다.\n\n1. 포털 시스템\n2. 학사 시스...,
5,qwen2.5:3b,QA,QA-03,가격점수 가드레일,가격점수를 계산해주세요.,"계산하지 않음, 필요 정보 부족 안내",True,2.798,"가격점수를 계산하기 위해서는 현재 문서에는 예정가격, 입찰가격, 평점산식의 구체적인...",
6,llama3.2:3b,QA,QA-01,금액 단위 보존,이 사업의 예산은 얼마인가요?,"11,270,000,000원, 억/조 임의 환산 금지",True,6.137,"사업의 예산은 11,270,000,000원입니다.",
7,llama3.2:3b,QA,QA-02,항목 누락 방지,구축 대상 시스템 목록을 모두 알려주세요.,"9개 항목 모두 포함, 항목 누락 없음",True,3.365,구축 대상 시스템 목록은 다음과 같습니다.\n\n1. 포털 시스템\n2. 학사 시스...,
8,llama3.2:3b,QA,QA-03,가격점수 가드레일,가격점수를 계산해주세요.,"계산하지 않음, 필요 정보 부족 안내",True,3.253,"현재 문서에는 가격평가는 입찰가격 평점산식에 따라 산정되며, 입찰가격 평점산식의 구...",
9,exaone3.5:2.4b,QA,QA-01,금액 단위 보존,이 사업의 예산은 얼마인가요?,"11,270,000,000원, 억/조 임의 환산 금지",True,9.266,"이 사업의 예산은 **11,270,000,000원**입니다.",


## 모델별 정보부족 판단 테스트 실행

이 단계에서는 사용자의 질문만으로 답변하기 어려운 상황에서 모델이 추가 확인이 필요한 정보를 적절히 안내하는지 확인합니다.

주요 확인 항목은 다음과 같습니다.

- 모호한 질문을 답변 가능으로 오판하지 않는지
- 회사 정보가 필요한 질문에서 수행 실적, 보유 인력, 기술 역량 등을 요구하는지
- 가격점수 계산 요청에서 입찰가격, 예정가격, 산식 등 핵심 입력값을 요구하는지
- 부족한 정보가 있는데도 “없습니다”라고 출력하지 않는지

In [20]:
clarification_results = []

for model in target_models:
    print(f"정보부족 판단 테스트 실행 중: {model}")
    
    for item in clarification_test_questions:
        prompt = clarification_prompt.format(
            question=item["question"]
        )
        
        result = ask_ollama(
            prompt=prompt,
            model=model,
            temperature=0.0,
            timeout=120
        )
        
        clarification_results.append({
            "model": model,
            "test_type": "Clarification",
            "case_id": item["case_id"],
            "category": item["category"],
            "question": item["question"],
            "check_points": ", ".join(item["check_points"]),
            "success": result["success"],
            "elapsed": round(result["elapsed"], 3),
            "answer": result["answer"],
            "error": result["error"]
        })

clarification_results_df = pd.DataFrame(clarification_results)
clarification_results_df

정보부족 판단 테스트 실행 중: gemma3:4b
정보부족 판단 테스트 실행 중: qwen2.5:3b
정보부족 판단 테스트 실행 중: llama3.2:3b
정보부족 판단 테스트 실행 중: exaone3.5:2.4b
정보부족 판단 테스트 실행 중: phi3.5


,model,test_type,case_id,category,question,check_points,success,elapsed,answer,error
0,gemma3:4b,Clarification,CL-01,입찰 적합도 정보부족,우리 회사가 이 입찰에 적합한지 봐줘.,"회사 정보 필요, 수행 실적, 보유 인력, 기술 역량",True,13.420,판단: 추가 확인 필요\n추가로 확인할 정보:\n- 우리 회사와 관련된 사업 분야 ...,
1,gemma3:4b,Clarification,CL-02,제안서 작성 정보부족,제안서에 쓸 만하게 정리해줘.,"사업 범위, 회사 정보, 수행 전략, 실적",True,3.510,판단: 추가 확인 필요\n추가로 확인할 정보:\n- 제안서를 제출하는 입찰 공고의 ...,
2,gemma3:4b,Clarification,CL-03,모호한 요약 요청,이거 핵심만 뽑아줘.,대상 문서 또는 내용 필요,True,3.573,판단: 추가 확인 필요\n추가로 확인할 정보:\n- 어떤 내용의 핵심을 뽑아달라는 ...,
3,gemma3:4b,Clarification,CL-04,가격점수 정보부족,가격점수 계산해줘.,"입찰가격, 예정가격, 산식",True,3.581,판단: 추가 확인 필요\n추가로 확인할 정보:\n- 가격 점수를 계산하기 위한 기준...,
4,qwen2.5:3b,Clarification,CL-01,입찰 적합도 정보부족,우리 회사가 이 입찰에 적합한지 봐줘.,"회사 정보 필요, 수행 실적, 보유 인력, 기술 역량",True,10.554,판단: 추가 확인 필요\n추가로 확인할 정보:\n- 우리 회사의 기술력과 경험\n-...,
5,qwen2.5:3b,Clarification,CL-02,제안서 작성 정보부족,제안서에 쓸 만하게 정리해줘.,"사업 범위, 회사 정보, 수행 전략, 실적",True,2.693,판단: 추가 확인 필요\n추가로 확인할 정보:\n- 제안서의 주제 또는 분야 (예:...,
6,qwen2.5:3b,Clarification,CL-03,모호한 요약 요청,이거 핵심만 뽑아줘.,대상 문서 또는 내용 필요,True,2.590,판단: 추가 확인 필요\n추가로 확인할 정보:\n- 사용자의 구체적인 질문 내용,
7,qwen2.5:3b,Clarification,CL-04,가격점수 정보부족,가격점수 계산해줘.,"입찰가격, 예정가격, 산식",True,2.794,판단: 추가 확인 필요\n추가로 확인할 정보:\n- 현재 가격 점수 계산 기준이 무...,
8,llama3.2:3b,Clarification,CL-01,입찰 적합도 정보부족,우리 회사가 이 입찰에 적합한지 봐줘.,"회사 정보 필요, 수행 실적, 보유 인력, 기술 역량",True,2.858,"판단: 답변 가능\n\n추가로 확인할 정보:\n- 회사 정보 (회사명, 주소, 사업...",
9,llama3.2:3b,Clarification,CL-02,제안서 작성 정보부족,제안서에 쓸 만하게 정리해줘.,"사업 범위, 회사 정보, 수행 전략, 실적",True,2.900,판단: 답변 가능\n\n추가로 확인할 정보:\n- 제안서의 목적과 내용\n- 제안서...,


## 모델별 질문 재작성 테스트 실행

이 단계에서는 사용자의 이전 질문과 추가 답변을 바탕으로 검색에 적합한 질문을 재작성할 수 있는지 확인합니다.

주요 확인 항목은 다음과 같습니다.

- 비교 대상이 명확할 때 재작성 가능으로 판단하는지
- 비교 대상이 부족할 때 추가 확인 필요로 판단하는지
- 입찰 적합도 판단에 필요한 회사 정보가 부족할 때 재작성 가능으로 오판하지 않는지
- 사업명이 추가 답변에 포함된 경우 이를 재작성 질문에 반영하는지

In [21]:
rewrite_results = []

for model in target_models:
    print(f"질문 재작성 테스트 실행 중: {model}")
    
    for item in rewrite_test_cases:
        prompt = rewrite_prompt.format(
            previous_question=item["previous_question"],
            additional_answer=item["additional_answer"]
        )
        
        result = ask_ollama(
            prompt=prompt,
            model=model,
            temperature=0.0,
            timeout=120
        )
        
        rewrite_results.append({
            "model": model,
            "test_type": "Rewrite",
            "case_id": item["case_id"],
            "category": item["category"],
            "previous_question": item["previous_question"],
            "additional_answer": item["additional_answer"],
            "check_points": ", ".join(item["check_points"]),
            "success": result["success"],
            "elapsed": round(result["elapsed"], 3),
            "answer": result["answer"],
            "error": result["error"]
        })

rewrite_results_df = pd.DataFrame(rewrite_results)
rewrite_results_df

질문 재작성 테스트 실행 중: gemma3:4b
질문 재작성 테스트 실행 중: qwen2.5:3b
질문 재작성 테스트 실행 중: llama3.2:3b
질문 재작성 테스트 실행 중: exaone3.5:2.4b
질문 재작성 테스트 실행 중: phi3.5


,model,test_type,case_id,category,previous_question,additional_answer,check_points,success,elapsed,answer,error
0,gemma3:4b,Rewrite,RW-01,비교 질문 재작성 가능,이 사업과 비교해주세요.,경희대 산학협력단 정보시스템 운영 용역과 비교해주세요.,"재작성 가능, 비교 대상 포함",True,12.736,판단: 재작성 가능\n재작성 질문: 경희대 산학협력단 정보시스템 운영 용역과 비교하...,
1,gemma3:4b,Rewrite,RW-02,입찰 적합도 추가 정보 부족,우리 회사가 이 입찰에 적합한지 봐줘.,그냥 알아서 해줘.,"추가 확인 필요, 회사 정보 필요, 없습니다 금지",True,4.298,판단: 재작성 가능 / 추가 확인 필요\n재작성 질문: 우리 회사(회사명)가 특정 ...,
2,gemma3:4b,Rewrite,RW-03,수행 전략 재작성 가능,수행 전략 작성해줘.,차세대 포털·학사 정보시스템 구축사업 기준으로 작성해줘.,"재작성 가능, 사업명 포함",True,4.177,판단: 재작성 가능\n재작성 질문: 차세대 포털 및 학사 정보 시스템 구축 사업을 ...,
3,gemma3:4b,Rewrite,RW-04,비교 대상 부족,이 사업이랑 다른 사업 비교해줘.,그냥 비슷한 걸로 해줘.,"추가 확인 필요, 비교 대상 필요",True,4.112,판단: 재작성 가능\n재작성 질문: 저희 사업과 유사한 사업들을 비교 분석해주실 수...,
4,qwen2.5:3b,Rewrite,RW-01,비교 질문 재작성 가능,이 사업과 비교해주세요.,경희대 산학협력단 정보시스템 운영 용역과 비교해주세요.,"재작성 가능, 비교 대상 포함",True,10.009,판단: 재작성 가능 / 추가 확인 필요\n\n재작성 질문:\n경희대학교 산학협력단의...,
5,qwen2.5:3b,Rewrite,RW-02,입찰 적합도 추가 정보 부족,우리 회사가 이 입찰에 적합한지 봐줘.,그냥 알아서 해줘.,"추가 확인 필요, 회사 정보 필요, 없습니다 금지",True,3.188,입력하신 내용에서 사용자의 실제 질문과 추가 답변을 파악하기 어렵습니다. 따라서 특...,
6,qwen2.5:3b,Rewrite,RW-03,수행 전략 재작성 가능,수행 전략 작성해줘.,차세대 포털·학사 정보시스템 구축사업 기준으로 작성해줘.,"재작성 가능, 사업명 포함",True,3.213,판단: 재작성 가능\n\n재작성 질문:\n차세대 포털과 학사 정보 시스템 구축 사업...,
7,qwen2.5:3b,Rewrite,RW-04,비교 대상 부족,이 사업이랑 다른 사업 비교해줘.,그냥 비슷한 걸로 해줘.,"추가 확인 필요, 비교 대상 필요",True,2.922,판단: 추가 확인 필요\n재작성 질문: \n이 사업과 비슷한 다른 사업들을 찾아서 ...,
8,llama3.2:3b,Rewrite,RW-01,비교 질문 재작성 가능,이 사업과 비교해주세요.,경희대 산학협력단 정보시스템 운영 용역과 비교해주세요.,"재작성 가능, 비교 대상 포함",True,3.254,판단: 재작성 가능\n\n재작성 질문:\n경희대 산학협력단 정보시스템 운영 용역과 ...,
9,llama3.2:3b,Rewrite,RW-02,입찰 적합도 추가 정보 부족,우리 회사가 이 입찰에 적합한지 봐줘.,그냥 알아서 해줘.,"추가 확인 필요, 회사 정보 필요, 없습니다 금지",True,3.170,판단: 재작성 가능\n\n재작성 질문:\nour company가 이 입찰에 적합한지...,


In [22]:
all_test_results_df = pd.concat(
    [
        qa_results_df,
        clarification_results_df,
        rewrite_results_df
    ],
    ignore_index=True,
    sort=False
)

all_test_results_df[
    ["model", "test_type", "case_id", "category", "success", "elapsed", "answer"]
]

,model,test_type,case_id,category,success,elapsed,answer
0,gemma3:4b,QA,QA-01,금액 단위 보존,True,8.557,"이 사업의 예산은 부가가치세를 포함한 11,270,000,000원입니다."
1,gemma3:4b,QA,QA-02,항목 누락 방지,True,3.360,구축 대상 시스템은 다음과 같습니다.\n1. 포털 시스템\n2. 학사 시스템\n3....
2,gemma3:4b,QA,QA-03,가격점수 가드레일,True,3.166,"현재 문서에는 예정가격, 입찰가격, 평점산식의 구체적인 입력값이 모두 제시되어 있지..."
3,qwen2.5:3b,QA,QA-01,금액 단위 보존,True,5.375,"이 사업의 예산은 11,270,000,000원입니다."
4,qwen2.5:3b,QA,QA-02,항목 누락 방지,True,2.919,구축 대상 시스템 목록은 다음과 같습니다.\n\n1. 포털 시스템\n2. 학사 시스...
5,qwen2.5:3b,QA,QA-03,가격점수 가드레일,True,2.798,"가격점수를 계산하기 위해서는 현재 문서에는 예정가격, 입찰가격, 평점산식의 구체적인..."
6,llama3.2:3b,QA,QA-01,금액 단위 보존,True,6.137,"사업의 예산은 11,270,000,000원입니다."
7,llama3.2:3b,QA,QA-02,항목 누락 방지,True,3.365,구축 대상 시스템 목록은 다음과 같습니다.\n\n1. 포털 시스템\n2. 학사 시스...
8,llama3.2:3b,QA,QA-03,가격점수 가드레일,True,3.253,"현재 문서에는 가격평가는 입찰가격 평점산식에 따라 산정되며, 입찰가격 평점산식의 구..."
9,exaone3.5:2.4b,QA,QA-01,금액 단위 보존,True,9.266,"이 사업의 예산은 **11,270,000,000원**입니다."


In [23]:
model_time_summary_df = all_test_results_df.groupby("model")["elapsed"].agg(
    count="count",
    mean="mean",
    min="min",
    max="max"
).reset_index()

model_time_summary_df = model_time_summary_df.sort_values("mean")
model_time_summary_df

,model,count,mean,min,max
2,llama3.2:3b,11,3.290000,2.600,6.137
0,exaone3.5:2.4b,11,3.710636,2.569,9.266
4,qwen2.5:3b,11,4.459545,2.590,10.554
3,phi3.5,11,5.585091,3.092,10.682
1,gemma3:4b,11,5.862727,3.166,13.420


In [24]:
test_type_time_summary_df = all_test_results_df.groupby(
    ["model", "test_type"]
)["elapsed"].agg(
    count="count",
    mean="mean",
    min="min",
    max="max"
).reset_index()

test_type_time_summary_df.sort_values(["test_type", "mean"])

,model,test_type,count,mean,min,max
6,llama3.2:3b,Clarification,4,2.742750,2.600,2.900
0,exaone3.5:2.4b,Clarification,4,3.191250,2.957,3.447
12,qwen2.5:3b,Clarification,4,4.657750,2.590,10.554
9,phi3.5,Clarification,4,5.505500,3.092,10.120
3,gemma3:4b,Clarification,4,6.021000,3.510,13.420
13,qwen2.5:3b,QA,3,3.697333,2.798,5.375
7,llama3.2:3b,QA,3,4.251667,3.253,6.137
1,exaone3.5:2.4b,QA,3,4.901333,2.569,9.266
4,gemma3:4b,QA,3,5.027667,3.166,8.557
10,phi3.5,QA,3,5.364667,3.256,9.364


In [25]:
call_success_summary_df = all_test_results_df.groupby(
    ["model", "success"]
).size().reset_index(name="count")

call_success_summary_df

,model,success,count
0,exaone3.5:2.4b,True,11
1,gemma3:4b,True,11
2,llama3.2:3b,True,11
3,phi3.5,True,11
4,qwen2.5:3b,True,11


In [27]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

all_test_results_df[
    ["model", "test_type", "case_id", "category", "success", "elapsed", "answer"]
]

,model,test_type,case_id,category,success,elapsed,answer
0,gemma3:4b,QA,QA-01,금액 단위 보존,True,8.557,"이 사업의 예산은 부가가치세를 포함한 11,270,000,000원입니다."
1,gemma3:4b,QA,QA-02,항목 누락 방지,True,3.360,구축 대상 시스템은 다음과 같습니다.\n1. 포털 시스템\n2. 학사 시스템\n3. 수강신청 시스템\n4. 전자결재 시스템\n5. 규정관리 시스템\n6. 검색 시스템\n7. 챗봇 시스템\n8. 모바일 시스템\n9. 통계 시스템
2,gemma3:4b,QA,QA-03,가격점수 가드레일,True,3.166,"현재 문서에는 예정가격, 입찰가격, 평점산식의 구체적인 입력값이 모두 제시되어 있지 않아 가격점수를 계산할 수 없습니다."
3,qwen2.5:3b,QA,QA-01,금액 단위 보존,True,5.375,"이 사업의 예산은 11,270,000,000원입니다."
4,qwen2.5:3b,QA,QA-02,항목 누락 방지,True,2.919,구축 대상 시스템 목록은 다음과 같습니다.\n\n1. 포털 시스템\n2. 학사 시스템\n3. 수강신청 시스템\n4. 전자결재 시스템\n5. 규정관리 시스템\n6. 검색 시스템\n7. 챗봇 시스템\n8. 모바일 시스템\n9. 통계 시스템
5,qwen2.5:3b,QA,QA-03,가격점수 가드레일,True,2.798,"가격점수를 계산하기 위해서는 현재 문서에는 예정가격, 입찰가격, 평점산식의 구체적인 입력값이 제공되지 않았습니다. 따라서 가격점수를 계산할 수 없습니다. 추가 정보가 필요합니다."
6,llama3.2:3b,QA,QA-01,금액 단위 보존,True,6.137,"사업의 예산은 11,270,000,000원입니다."
7,llama3.2:3b,QA,QA-02,항목 누락 방지,True,3.365,구축 대상 시스템 목록은 다음과 같습니다.\n\n1. 포털 시스템\n2. 학사 시스템\n3. 수강신청 시스템\n4. 전자결재 시스템\n5. 규정관리 시스템\n6. 검색 시스템\n7. 챗봇 시스템\n8. 모바일 시스템\n9. 통계 시스템
8,llama3.2:3b,QA,QA-03,가격점수 가드레일,True,3.253,"현재 문서에는 가격평가는 입찰가격 평점산식에 따라 산정되며, 입찰가격 평점산식의 구체적인 입력값이 모두 제시되어 있지 않습니다.因此, 가격점수를 계산할 수 없습니다."
9,exaone3.5:2.4b,QA,QA-01,금액 단위 보존,True,9.266,"이 사업의 예산은 **11,270,000,000원**입니다."


In [28]:
review_cases = all_test_results_df[
    all_test_results_df["case_id"].isin([
        "QA-02", 
        "CL-01", "CL-02", "CL-04",
        "RW-02", "RW-04"
    ])
][
    ["model", "test_type", "case_id", "category", "answer"]
]

review_cases

,model,test_type,case_id,category,answer
1,gemma3:4b,QA,QA-02,항목 누락 방지,구축 대상 시스템은 다음과 같습니다.\n1. 포털 시스템\n2. 학사 시스템\n3. 수강신청 시스템\n4. 전자결재 시스템\n5. 규정관리 시스템\n6. 검색 시스템\n7. 챗봇 시스템\n8. 모바일 시스템\n9. 통계 시스템
4,qwen2.5:3b,QA,QA-02,항목 누락 방지,구축 대상 시스템 목록은 다음과 같습니다.\n\n1. 포털 시스템\n2. 학사 시스템\n3. 수강신청 시스템\n4. 전자결재 시스템\n5. 규정관리 시스템\n6. 검색 시스템\n7. 챗봇 시스템\n8. 모바일 시스템\n9. 통계 시스템
7,llama3.2:3b,QA,QA-02,항목 누락 방지,구축 대상 시스템 목록은 다음과 같습니다.\n\n1. 포털 시스템\n2. 학사 시스템\n3. 수강신청 시스템\n4. 전자결재 시스템\n5. 규정관리 시스템\n6. 검색 시스템\n7. 챗봇 시스템\n8. 모바일 시스템\n9. 통계 시스템
10,exaone3.5:2.4b,QA,QA-02,항목 누락 방지,구축 대상 시스템 목록은 다음과 같습니다:\n\n1. 포털 시스템\n2. 학사 시스템\n3. 수강신청 시스템\n4. 전자결재 시스템\n5. 규정관리 시스템\n6. 검색 시스템\n7. 챗봇 시스템\n8. 모바일 시스템\n9. 통계 시스템
13,phi3.5,QA,QA-02,항목 누락 방지,1. 포털 시스템\n2. 학사 시스템\n3. 수강신청 시스템\n4. 전자결재 시스템\n5. 규정관리 시스템\n6. 검색 시스템\n7. 챕터블러 시스템\n8. 모바일 시스템\n9. 통계 시스템
15,gemma3:4b,Clarification,CL-01,입찰 적합도 정보부족,"판단: 추가 확인 필요\n추가로 확인할 정보:\n- 우리 회사와 관련된 사업 분야 및 주요 제품/서비스 정보\n- 우리 회사의 규모 (매출액, 직원 수 등)\n- 이번 RFP의 상세 내용 (요구 사항, 평가 기준 등)"
16,gemma3:4b,Clarification,CL-02,제안서 작성 정보부족,"판단: 추가 확인 필요\n추가로 확인할 정보:\n- 제안서를 제출하는 입찰 공고의 명칭 및 번호\n- 제안서 작성 목적 (어떤 사업/프로젝트에 대한 제안인가?)\n- 제안서의 구체적인 요구 사항 (예: 분량, 포함되어야 할 내용 등)"
18,gemma3:4b,Clarification,CL-04,가격점수 정보부족,"판단: 추가 확인 필요\n추가로 확인할 정보:\n- 가격 점수를 계산하기 위한 기준 (예: 가중치, 평가 항목 등)\n- 가격 점수를 계산할 대상 (예: 특정 제안서, 여러 제안서 비교 등)\n- 가격 점수 계산에 사용될 데이터 (예: 각 제안사의 가격 정보)"
19,qwen2.5:3b,Clarification,CL-01,입찰 적합도 정보부족,판단: 추가 확인 필요\n추가로 확인할 정보:\n- 우리 회사의 기술력과 경험\n- 이 입찰에서 요구하는 주요 특징이나 기능
20,qwen2.5:3b,Clarification,CL-02,제안서 작성 정보부족,"판단: 추가 확인 필요\n추가로 확인할 정보:\n- 제안서의 주제 또는 분야 (예: 기술 개발, 시스템 구축 등)\n- 사용자의 요구 사항이나 목표"


In [29]:
for _, row in review_cases.iterrows():
    print("=" * 80)
    print(f"model: {row['model']}")
    print(f"test_type: {row['test_type']}")
    print(f"case_id: {row['case_id']}")
    print(f"category: {row['category']}")
    print("-" * 80)
    print(row["answer"])
    print()

model: gemma3:4b
test_type: QA
case_id: QA-02
category: 항목 누락 방지
--------------------------------------------------------------------------------
구축 대상 시스템은 다음과 같습니다.
1. 포털 시스템
2. 학사 시스템
3. 수강신청 시스템
4. 전자결재 시스템
5. 규정관리 시스템
6. 검색 시스템
7. 챗봇 시스템
8. 모바일 시스템
9. 통계 시스템

model: qwen2.5:3b
test_type: QA
case_id: QA-02
category: 항목 누락 방지
--------------------------------------------------------------------------------
구축 대상 시스템 목록은 다음과 같습니다.

1. 포털 시스템
2. 학사 시스템
3. 수강신청 시스템
4. 전자결재 시스템
5. 규정관리 시스템
6. 검색 시스템
7. 챗봇 시스템
8. 모바일 시스템
9. 통계 시스템

model: llama3.2:3b
test_type: QA
case_id: QA-02
category: 항목 누락 방지
--------------------------------------------------------------------------------
구축 대상 시스템 목록은 다음과 같습니다.

1. 포털 시스템
2. 학사 시스템
3. 수강신청 시스템
4. 전자결재 시스템
5. 규정관리 시스템
6. 검색 시스템
7. 챗봇 시스템
8. 모바일 시스템
9. 통계 시스템

model: exaone3.5:2.4b
test_type: QA
case_id: QA-02
category: 항목 누락 방지
--------------------------------------------------------------------------------
구축 대상 시스템 목록은 다음과 같습니다:

1. 포털 시스템
2. 학사 시스

## 모델별 테스트 결과 해석

문서 기반 QA, 정보부족 판단, 질문 재작성 테스트를 전체 모델에 대해 실행한 결과, 모든 모델에서 `success=True`가 반환되어 API 호출 자체는 정상적으로 수행되었습니다. 다만 `success=True`는 호출 성공 여부만 의미하므로, 실제 품질 평가는 각 모델의 응답 내용을 기준으로 별도로 판단하였습니다.

### 1. 문서 기반 QA 테스트 결과

문서 기반 QA 테스트에서는 대부분의 모델이 기본적인 문서 기반 응답을 수행할 수 있음을 확인하였습니다.

금액 단위 보존 테스트에서는 `gemma3:4b`, `qwen2.5:3b`, `llama3.2:3b`, `exaone3.5:2.4b`, `phi3.5` 모두 `11,270,000,000원`을 원문 표기 그대로 유지하였습니다. 억 단위나 조 단위로 임의 환산한 사례는 확인되지 않았습니다.

항목 누락 방지 테스트에서는 `gemma3:4b`, `qwen2.5:3b`, `llama3.2:3b`, `exaone3.5:2.4b`가 포털 시스템, 학사 시스템, 수강신청 시스템, 전자결재 시스템, 규정관리 시스템, 검색 시스템, 챗봇 시스템, 모바일 시스템, 통계 시스템의 9개 항목을 모두 출력하였습니다.

다만 `phi3.5`는 7번 항목인 `챗봇 시스템`을 `챕터블러 시스템`으로 잘못 출력하였습니다. 항목 개수는 9개를 유지했지만, 원문 항목명을 정확히 보존하지 못했기 때문에 항목 누락 방지 테스트에서는 부분 실패로 볼 수 있습니다.

가격점수 가드레일 테스트에서는 `gemma3:4b`와 `qwen2.5:3b`가 예정가격, 입찰가격, 평점산식의 구체적인 입력값 부족을 이유로 가격점수를 계산할 수 없다고 답변하여 안정적인 결과를 보였습니다.

`exaone3.5:2.4b`도 가격점수를 계산하지 않고 필요한 정보가 부족하다고 안내했지만, 필요한 정보로 예정가격과 평점산식만 강조하고 입찰가격을 명확히 포함하지 않았습니다. 따라서 계산을 막는 방향은 적절했지만, 부족 정보 안내의 구체성은 일부 보완이 필요합니다.

`llama3.2:3b`는 가격점수를 계산하지 않은 점은 적절했으나, 응답 중 `因此`라는 중국어 표현이 포함되었습니다. 이는 사용자에게 직접 제공되는 답변으로는 부적절하며, 외국어 혼입 문제가 확인된 사례입니다.

`phi3.5`는 가격점수를 계산할 수 없다는 취지로 답변했지만, `제공하고 있지만l`, `모둌` 같은 깨진 표현이 포함되었고 문장 의미도 불명확했습니다. 따라서 가격점수 가드레일 테스트에서 안정적인 답변으로 보기 어렵습니다.

### 2. 정보부족 판단 테스트 결과

정보부족 판단 테스트에서는 모델별 차이가 크게 나타났습니다.

`gemma3:4b`는 네 개의 정보부족 판단 케이스에서 모두 `추가 확인 필요`로 판단하였습니다. 입찰 적합도, 제안서 작성, 모호한 요약 요청, 가격점수 계산 요청에서 모두 추가 정보가 필요하다는 방향은 적절했습니다.

다만 입찰 적합도 판단에서는 회사의 수행 실적, 보유 인력, 기술 역량보다 사업 분야, 제품/서비스, 회사 규모, RFP 상세 내용 중심으로 답변하였습니다. 가격점수 계산에서도 입찰가격, 예정가격, 산식이라는 핵심 용어를 직접적으로 모두 짚지는 못하고, 가격 점수 계산 기준, 계산 대상, 가격 정보처럼 다소 일반적인 표현을 사용했습니다. 따라서 판단 방향은 안정적이지만, 추가 정보 항목을 더 구체화할 필요가 있습니다.

`qwen2.5:3b`도 네 개의 케이스에서 모두 `추가 확인 필요`로 판단하였습니다. 입찰 적합도 판단에서는 회사의 기술력과 경험을 요구했고, 제안서 작성과 모호한 요약 요청에서도 추가 정보가 필요하다고 판단했습니다. 다만 일부 응답은 필요한 정보가 짧고 일반적으로 제시되어, 실제 사용자 안내용으로는 정보 요구 항목을 더 구체화하는 보완이 필요합니다.

`llama3.2:3b`는 정보부족 판단 테스트에서 가장 큰 문제가 확인되었습니다. 입찰 적합도, 제안서 작성, 모호한 요약 요청, 가격점수 계산 요청 모두에서 `답변 가능`으로 판단하였으며, 일부 케이스에서는 `추가로 확인할 정보: None`이라고 출력했습니다. 특히 모호한 요약 요청과 가격점수 계산 요청은 명확히 추가 정보가 필요한 상황이므로, 정보부족 판단 모델로 사용하기에는 부적합합니다.

`exaone3.5:2.4b`는 네 개의 케이스 모두에서 `추가 확인 필요`로 판단하여 기본 방향은 적절했습니다. 다만 입찰 적합도 판단에서는 회사의 수행 실적, 보유 인력, 기술 역량보다는 입찰 자격 요건, 입찰 범위, 경쟁 업체 분석을 중심으로 안내했습니다. 제안서 작성 요청에서도 회사 정보나 수행 실적보다는 프로젝트 범위, 기술 요구사항, 예산, 일정, 기존 인프라, 리스크 중심으로 답변했습니다. 따라서 정보부족 판단 자체는 가능하지만, 사용자에게 요구해야 할 회사 정보 항목을 더 명확히 포함하도록 보완이 필요합니다.

`phi3.5`는 일부 케이스에서 `추가 확인 필요`로 판단했지만, 표현 품질과 판단 안정성이 부족했습니다. 모호한 요약 요청에서는 `RFP(공개선서)`처럼 부정확한 표현이 나타났고, 가격점수 계산 요청에서는 `판단: 답변 가능 / 추가 확인 필요`처럼 판단이 혼재되었습니다. 또한 `출력:` 같은 내부 라벨을 다시 노출하고, 구매 제품 종류나 상자/유통 방식 등 현재 질문과 맞지 않는 정보를 요구했습니다. 따라서 정보부족 판단 결과의 일관성과 사용자 응답 품질이 모두 부족합니다.

### 3. 질문 재작성 테스트 결과

질문 재작성 테스트에서는 쉬운 재작성 가능 케이스와 정보부족 케이스 사이에서 모델별 차이가 뚜렷하게 나타났습니다.

`gemma3:4b`는 비교 대상이 명확한 케이스와 사업명이 주어진 수행 전략 케이스에서는 재작성 가능으로 판단하고 재작성 질문을 생성했습니다. 그러나 입찰 적합도 정보가 부족한 케이스에서는 `재작성 가능 / 추가 확인 필요`처럼 판단이 혼재되었고, 비교 대상이 부족한 케이스에서도 `재작성 가능`으로 잘못 판단했습니다. 따라서 재작성 문장 생성 능력은 있으나, 정보가 부족한 상황에서 재작성 가능 여부를 보수적으로 판단하는 능력은 보완이 필요합니다.

`qwen2.5:3b`는 수행 전략 재작성 케이스에서는 사업명을 반영하여 재작성 가능으로 판단했습니다. 비교 대상이 부족한 케이스에서는 `추가 확인 필요`로 판단하여 방향은 적절했지만, 동시에 재작성 질문을 함께 출력했습니다. 또한 비교 대상이 명확한 케이스에서도 `재작성 가능 / 추가 확인 필요`처럼 판단이 섞여 출력되었습니다. 따라서 판단 방향은 일부 양호하지만, 출력 형식과 판단값을 하나로 고정하는 보완이 필요합니다.

`llama3.2:3b`는 비교 대상이 명확한 케이스와 사업명이 주어진 케이스에서는 재작성 가능으로 판단했지만, 입찰 적합도 정보가 부족한 케이스와 비교 대상이 부족한 케이스에서도 재작성 가능으로 잘못 판단했습니다. 특히 입찰 적합도 케이스에서는 `our company`라는 영어 표현이 반복되었고, 수행 전략 케이스에서도 `scope`, `timeline`, `deadlines` 같은 영어 표현이 섞였습니다. 따라서 질문 재작성 모델로 사용하기에는 안정성이 낮습니다.

`exaone3.5:2.4b`는 비교 대상이 명확한 케이스와 사업명이 주어진 수행 전략 케이스에서는 비교적 자연스러운 재작성 질문을 생성했습니다. 그러나 입찰 적합도 정보가 부족한 케이스에서는 `추가 확인 필요`로 판단하면서도 재작성 질문을 함께 생성했고, 비교 대상이 부족한 케이스에서는 `재작성 가능`으로 잘못 판단했습니다. 또한 `RFP(Request for Proposal)`처럼 영어 설명이 포함되어 사용자 응답용으로는 추가적인 프롬프트 보완이나 후처리가 필요합니다.

`phi3.5`는 질문 재작성 테스트에서 전반적으로 안정성이 낮았습니다. 비교 대상이 명확한 케이스에서도 재작성 질문이 부자연스럽고, 입찰 적합도 정보부족 케이스와 비교 대상 부족 케이스를 모두 `재작성 가능`으로 잘못 판단했습니다. 또한 `구체하여`, `어떤 특정한 영역에서나?`처럼 부자연스러운 문장이 나타났습니다. 따라서 질문 재작성용 모델로 보기 어렵습니다.

### 4. 모델별 종합 관찰

`gemma3:4b`는 문서 기반 QA와 정보부족 판단에서 비교적 안정적인 결과를 보였습니다. 금액 보존, 항목 나열, 임의 계산 방지 모두 적절하게 수행했으며, 정보부족 판단에서도 전반적으로 추가 확인 필요를 올바르게 판단했습니다. 다만 질문 재작성에서는 정보가 부족한 상황을 재작성 가능으로 오판하는 사례가 있어, 재작성 프롬프트에는 더 엄격한 판단 기준이 필요합니다.

`qwen2.5:3b`는 문서 기반 QA와 정보부족 판단에서 전반적으로 안정적인 결과를 보였고, 응답 속도도 비교적 빠른 편이었습니다. 다만 질문 재작성에서는 판단값이 혼재되거나, 추가 확인 필요와 재작성 질문을 동시에 출력하는 문제가 있었습니다. 따라서 속도 중심 후보로는 가능성이 있지만, 출력 형식 제어가 필요합니다.

`llama3.2:3b`는 문서 기반 QA 일부에서는 답변을 생성할 수 있었지만, 정보부족 판단에서 대부분의 케이스를 `답변 가능`으로 잘못 판단했습니다. 또한 중국어와 영어가 응답에 섞이는 문제가 여러 번 확인되었습니다. 따라서 Generation 파트의 주요 후보로 사용하기에는 위험이 큽니다.

`exaone3.5:2.4b`는 문서 기반 QA와 정보부족 판단에서 기본적인 가능성을 보였습니다. 그러나 부족 정보 안내가 회사 정보보다는 일반적인 프로젝트 정보 중심으로 흐르는 경향이 있었고, 질문 재작성에서는 정보부족 상황을 재작성 가능으로 오판하는 사례가 확인되었습니다. 따라서 보조 후보 또는 추가 검증 후보로 분류하는 것이 적절합니다.

`phi3.5`는 문서 기반 QA에서도 항목명 오류와 깨진 표현이 확인되었고, 정보부족 판단과 질문 재작성에서도 판단 혼재, 내부 라벨 노출, 부자연스러운 문장, 오판이 발생했습니다. 따라서 현재 결과 기준으로는 최종 후보에서 제외하는 것이 적절합니다.

### 5. 현재 단계 판단

현재 출력 결과를 기준으로 보면 문서 기반 QA만 놓고는 여러 모델이 기본적인 답변을 생성할 수 있었지만, 실제 서비스에서 중요한 정보부족 판단과 질문 재작성에서는 모델별 안정성 차이가 크게 나타났습니다.

우선 검토 가능한 모델은 `gemma3:4b`와 `qwen2.5:3b`입니다. `gemma3:4b`는 답변 품질과 정보부족 판단의 안정성이 비교적 높고, `qwen2.5:3b`는 응답 속도와 기본 QA 성능 측면에서 장점이 있습니다.

`exaone3.5:2.4b`는 QA와 정보부족 판단에서 가능성은 있으나, 질문 재작성 판단과 사용자 응답 형식 측면에서 추가 보완이 필요합니다. 따라서 단독 최종 후보보다는 보조 후보 또는 추가 검증 후보로 보는 것이 적절합니다.

`llama3.2:3b`와 `phi3.5`는 정보부족 판단 오류, 외국어 혼입, 표현 깨짐, 판단 혼재 문제가 확인되어 현재 결과 기준으로는 주요 후보에서 제외하는 것이 적절합니다.

## 수동 평가 기준

모델 응답의 품질은 단순히 API 호출 성공 여부로 판단할 수 없습니다. 따라서 각 테스트 케이스에 대해 사람이 직접 응답 내용을 확인하고 `통과`, `부분 통과`, `실패`로 평가합니다.

평가 기준은 다음과 같습니다.

| 평가값 | 의미 |
|---|---|
| 통과 | 핵심 요구사항을 충족하고, 사용자에게 그대로 제공 가능한 수준 |
| 부분 통과 | 큰 방향은 맞지만 일부 누락, 표현 문제, 형식 혼재, 구체성 부족이 있는 경우 |
| 실패 | 핵심 판단 오류, 외국어 혼입, 깨진 표현, 임의 추론, 항목 오류가 있는 경우 |

세부 평가 관점은 다음과 같습니다.

- 문서에 있는 정보만 사용했는가
- 금액과 숫자를 원문 그대로 보존했는가
- 목록형 항목을 누락하거나 왜곡하지 않았는가
- 계산에 필요한 정보가 부족할 때 임의 계산하지 않았는가
- 정보가 부족한 질문을 답변 가능으로 오판하지 않았는가
- 필요한 추가 정보를 구체적으로 안내했는가
- 질문 재작성 가능 여부를 적절히 판단했는가
- 한국어 응답을 유지했는가
- 외국어 혼입, 깨진 표현, 내부 라벨 노출이 없는가

In [30]:
manual_eval_df = all_test_results_df[
    ["model", "test_type", "case_id", "category", "answer"]
].copy()

manual_eval_df["manual_result"] = ""
manual_eval_df["memo"] = ""

manual_eval_df

,model,test_type,case_id,category,answer,manual_result,memo
0,gemma3:4b,QA,QA-01,금액 단위 보존,"이 사업의 예산은 부가가치세를 포함한 11,270,000,000원입니다.",,
1,gemma3:4b,QA,QA-02,항목 누락 방지,구축 대상 시스템은 다음과 같습니다.\n1. 포털 시스템\n2. 학사 시스템\n3. 수강신청 시스템\n4. 전자결재 시스템\n5. 규정관리 시스템\n6. 검색 시스템\n7. 챗봇 시스템\n8. 모바일 시스템\n9. 통계 시스템,,
2,gemma3:4b,QA,QA-03,가격점수 가드레일,"현재 문서에는 예정가격, 입찰가격, 평점산식의 구체적인 입력값이 모두 제시되어 있지 않아 가격점수를 계산할 수 없습니다.",,
3,qwen2.5:3b,QA,QA-01,금액 단위 보존,"이 사업의 예산은 11,270,000,000원입니다.",,
4,qwen2.5:3b,QA,QA-02,항목 누락 방지,구축 대상 시스템 목록은 다음과 같습니다.\n\n1. 포털 시스템\n2. 학사 시스템\n3. 수강신청 시스템\n4. 전자결재 시스템\n5. 규정관리 시스템\n6. 검색 시스템\n7. 챗봇 시스템\n8. 모바일 시스템\n9. 통계 시스템,,
5,qwen2.5:3b,QA,QA-03,가격점수 가드레일,"가격점수를 계산하기 위해서는 현재 문서에는 예정가격, 입찰가격, 평점산식의 구체적인 입력값이 제공되지 않았습니다. 따라서 가격점수를 계산할 수 없습니다. 추가 정보가 필요합니다.",,
6,llama3.2:3b,QA,QA-01,금액 단위 보존,"사업의 예산은 11,270,000,000원입니다.",,
7,llama3.2:3b,QA,QA-02,항목 누락 방지,구축 대상 시스템 목록은 다음과 같습니다.\n\n1. 포털 시스템\n2. 학사 시스템\n3. 수강신청 시스템\n4. 전자결재 시스템\n5. 규정관리 시스템\n6. 검색 시스템\n7. 챗봇 시스템\n8. 모바일 시스템\n9. 통계 시스템,,
8,llama3.2:3b,QA,QA-03,가격점수 가드레일,"현재 문서에는 가격평가는 입찰가격 평점산식에 따라 산정되며, 입찰가격 평점산식의 구체적인 입력값이 모두 제시되어 있지 않습니다.因此, 가격점수를 계산할 수 없습니다.",,
9,exaone3.5:2.4b,QA,QA-01,금액 단위 보존,"이 사업의 예산은 **11,270,000,000원**입니다.",,


## 수동 평가값 입력

아래 셀에서는 앞에서 확인한 전체 응답 원문을 기준으로 수동 평가값을 입력합니다.

평가값은 다음 세 가지로 통일합니다.

- `통과`
- `부분 통과`
- `실패`

평가 기준은 모델 호출 성공 여부가 아니라 실제 응답 내용입니다.

In [31]:
manual_eval_records = [
    # gemma3:4b
    {
        "model": "gemma3:4b",
        "test_type": "QA",
        "case_id": "QA-01",
        "manual_result": "통과",
        "memo": "금액을 11,270,000,000원으로 원문 그대로 유지함"
    },
    {
        "model": "gemma3:4b",
        "test_type": "QA",
        "case_id": "QA-02",
        "manual_result": "통과",
        "memo": "구축 대상 시스템 9개 항목을 모두 출력함"
    },
    {
        "model": "gemma3:4b",
        "test_type": "QA",
        "case_id": "QA-03",
        "manual_result": "통과",
        "memo": "예정가격, 입찰가격, 평점산식 입력값 부족으로 계산하지 않음"
    },
    {
        "model": "gemma3:4b",
        "test_type": "Clarification",
        "case_id": "CL-01",
        "manual_result": "부분 통과",
        "memo": "추가 확인 필요 판단은 적절하나 수행 실적, 보유 인력, 기술 역량이 직접적으로 충분히 제시되지는 않음"
    },
    {
        "model": "gemma3:4b",
        "test_type": "Clarification",
        "case_id": "CL-02",
        "manual_result": "부분 통과",
        "memo": "추가 확인 필요 판단은 적절하나 회사 정보, 수행 전략, 실적 안내가 부족함"
    },
    {
        "model": "gemma3:4b",
        "test_type": "Clarification",
        "case_id": "CL-03",
        "manual_result": "통과",
        "memo": "모호한 요약 요청에 대해 대상 내용과 핵심 기준 확인이 필요하다고 안내함"
    },
    {
        "model": "gemma3:4b",
        "test_type": "Clarification",
        "case_id": "CL-04",
        "manual_result": "부분 통과",
        "memo": "추가 확인 필요 판단은 맞지만 입찰가격, 예정가격, 산식을 직접적으로 모두 짚지는 못함"
    },
    {
        "model": "gemma3:4b",
        "test_type": "Rewrite",
        "case_id": "RW-01",
        "manual_result": "통과",
        "memo": "비교 대상이 명확한 상황에서 재작성 가능으로 판단하고 비교 질문을 생성함"
    },
    {
        "model": "gemma3:4b",
        "test_type": "Rewrite",
        "case_id": "RW-02",
        "manual_result": "부분 통과",
        "memo": "회사 정보 부족을 안내했지만 판단이 재작성 가능 / 추가 확인 필요로 혼재됨"
    },
    {
        "model": "gemma3:4b",
        "test_type": "Rewrite",
        "case_id": "RW-03",
        "manual_result": "통과",
        "memo": "사업명을 반영하여 수행 전략 질문으로 재작성함"
    },
    {
        "model": "gemma3:4b",
        "test_type": "Rewrite",
        "case_id": "RW-04",
        "manual_result": "실패",
        "memo": "비교 대상이 부족한데 재작성 가능으로 오판함"
    },

    # qwen2.5:3b
    {
        "model": "qwen2.5:3b",
        "test_type": "QA",
        "case_id": "QA-01",
        "manual_result": "통과",
        "memo": "금액을 11,270,000,000원으로 원문 그대로 유지함"
    },
    {
        "model": "qwen2.5:3b",
        "test_type": "QA",
        "case_id": "QA-02",
        "manual_result": "통과",
        "memo": "구축 대상 시스템 9개 항목을 모두 출력함"
    },
    {
        "model": "qwen2.5:3b",
        "test_type": "QA",
        "case_id": "QA-03",
        "manual_result": "통과",
        "memo": "예정가격, 입찰가격, 평점산식 입력값 부족으로 계산하지 않음"
    },
    {
        "model": "qwen2.5:3b",
        "test_type": "Clarification",
        "case_id": "CL-01",
        "manual_result": "부분 통과",
        "memo": "추가 확인 필요 판단은 맞지만 수행 실적, 보유 인력 등 구체성이 부족함"
    },
    {
        "model": "qwen2.5:3b",
        "test_type": "Clarification",
        "case_id": "CL-02",
        "manual_result": "부분 통과",
        "memo": "추가 확인 필요 판단은 맞지만 회사 정보, 수행 전략, 실적 안내가 부족함"
    },
    {
        "model": "qwen2.5:3b",
        "test_type": "Clarification",
        "case_id": "CL-03",
        "manual_result": "부분 통과",
        "memo": "추가 확인 필요 판단은 맞지만 대상 문서나 내용 필요성을 충분히 구체화하지 못함"
    },
    {
        "model": "qwen2.5:3b",
        "test_type": "Clarification",
        "case_id": "CL-04",
        "manual_result": "부분 통과",
        "memo": "추가 확인 필요 판단은 맞지만 입찰가격, 예정가격, 산식이라는 핵심 용어가 직접적으로 부족함"
    },
    {
        "model": "qwen2.5:3b",
        "test_type": "Rewrite",
        "case_id": "RW-01",
        "manual_result": "부분 통과",
        "memo": "비교 대상은 반영했지만 판단이 재작성 가능 / 추가 확인 필요로 혼재됨"
    },
    {
        "model": "qwen2.5:3b",
        "test_type": "Rewrite",
        "case_id": "RW-02",
        "manual_result": "부분 통과",
        "memo": "재작성 불가능하다는 방향은 맞지만 출력 형식이 지정 형식과 다름"
    },
    {
        "model": "qwen2.5:3b",
        "test_type": "Rewrite",
        "case_id": "RW-03",
        "manual_result": "통과",
        "memo": "사업명을 반영하여 수행 전략 관련 질문으로 재작성함"
    },
    {
        "model": "qwen2.5:3b",
        "test_type": "Rewrite",
        "case_id": "RW-04",
        "manual_result": "부분 통과",
        "memo": "추가 확인 필요 판단은 맞지만 재작성 질문을 함께 출력하여 형식이 혼재됨"
    },

    # llama3.2:3b
    {
        "model": "llama3.2:3b",
        "test_type": "QA",
        "case_id": "QA-01",
        "manual_result": "통과",
        "memo": "금액을 11,270,000,000원으로 원문 그대로 유지함"
    },
    {
        "model": "llama3.2:3b",
        "test_type": "QA",
        "case_id": "QA-02",
        "manual_result": "통과",
        "memo": "구축 대상 시스템 9개 항목을 모두 출력함"
    },
    {
        "model": "llama3.2:3b",
        "test_type": "QA",
        "case_id": "QA-03",
        "manual_result": "실패",
        "memo": "가격점수를 계산하지 않은 방향은 맞지만 중국어 표현이 혼입됨"
    },
    {
        "model": "llama3.2:3b",
        "test_type": "Clarification",
        "case_id": "CL-01",
        "manual_result": "실패",
        "memo": "추가 확인 필요 상황을 답변 가능으로 오판함"
    },
    {
        "model": "llama3.2:3b",
        "test_type": "Clarification",
        "case_id": "CL-02",
        "manual_result": "실패",
        "memo": "추가 확인 필요 상황을 답변 가능으로 오판함"
    },
    {
        "model": "llama3.2:3b",
        "test_type": "Clarification",
        "case_id": "CL-03",
        "manual_result": "실패",
        "memo": "모호한 요약 요청을 답변 가능으로 판단하고 추가 정보 없음으로 출력함"
    },
    {
        "model": "llama3.2:3b",
        "test_type": "Clarification",
        "case_id": "CL-04",
        "manual_result": "실패",
        "memo": "가격점수 계산 요청을 답변 가능으로 판단하고 추가 정보 없음으로 출력함"
    },
    {
        "model": "llama3.2:3b",
        "test_type": "Rewrite",
        "case_id": "RW-01",
        "manual_result": "부분 통과",
        "memo": "재작성 가능 판단은 맞지만 비교 대상 해석이 부정확함"
    },
    {
        "model": "llama3.2:3b",
        "test_type": "Rewrite",
        "case_id": "RW-02",
        "manual_result": "실패",
        "memo": "회사 정보 부족 상황을 재작성 가능으로 오판하고 영어 표현이 혼입됨"
    },
    {
        "model": "llama3.2:3b",
        "test_type": "Rewrite",
        "case_id": "RW-03",
        "manual_result": "부분 통과",
        "memo": "사업명은 반영했지만 scope, timeline, deadlines 등 영어 표현이 혼입됨"
    },
    {
        "model": "llama3.2:3b",
        "test_type": "Rewrite",
        "case_id": "RW-04",
        "manual_result": "실패",
        "memo": "비교 대상 부족 상황을 재작성 가능으로 오판함"
    },

    # exaone3.5:2.4b
    {
        "model": "exaone3.5:2.4b",
        "test_type": "QA",
        "case_id": "QA-01",
        "manual_result": "통과",
        "memo": "금액을 11,270,000,000원으로 원문 그대로 유지함"
    },
    {
        "model": "exaone3.5:2.4b",
        "test_type": "QA",
        "case_id": "QA-02",
        "manual_result": "통과",
        "memo": "구축 대상 시스템 9개 항목을 모두 출력함"
    },
    {
        "model": "exaone3.5:2.4b",
        "test_type": "QA",
        "case_id": "QA-03",
        "manual_result": "부분 통과",
        "memo": "계산하지 않은 점은 적절하나 부족 정보에서 입찰가격이 명확히 빠짐"
    },
    {
        "model": "exaone3.5:2.4b",
        "test_type": "Clarification",
        "case_id": "CL-01",
        "manual_result": "부분 통과",
        "memo": "추가 확인 필요 판단은 맞지만 회사 수행 실적, 보유 인력, 기술 역량보다 일반 입찰 정보 중심임"
    },
    {
        "model": "exaone3.5:2.4b",
        "test_type": "Clarification",
        "case_id": "CL-02",
        "manual_result": "부분 통과",
        "memo": "추가 확인 필요 판단은 맞지만 회사 정보와 실적보다 프로젝트 정보 중심임"
    },
    {
        "model": "exaone3.5:2.4b",
        "test_type": "Clarification",
        "case_id": "CL-03",
        "manual_result": "통과",
        "memo": "모호한 요약 요청에 대해 구체적인 RFP 요구사항과 응답 범위를 확인함"
    },
    {
        "model": "exaone3.5:2.4b",
        "test_type": "Clarification",
        "case_id": "CL-04",
        "manual_result": "부분 통과",
        "memo": "추가 확인 필요 판단은 맞지만 가격점수 계산에 필요한 입찰가격, 예정가격, 산식을 직접적으로 짚지 못함"
    },
    {
        "model": "exaone3.5:2.4b",
        "test_type": "Rewrite",
        "case_id": "RW-01",
        "manual_result": "부분 통과",
        "memo": "비교 대상은 반영했지만 RFP 영어 설명이 포함되고 추가 정보가 과도함"
    },
    {
        "model": "exaone3.5:2.4b",
        "test_type": "Rewrite",
        "case_id": "RW-02",
        "manual_result": "부분 통과",
        "memo": "추가 확인 필요 판단은 맞지만 재작성 질문을 함께 생성하고 영어 표현이 포함됨"
    },
    {
        "model": "exaone3.5:2.4b",
        "test_type": "Rewrite",
        "case_id": "RW-03",
        "manual_result": "통과",
        "memo": "사업명을 반영하여 수행 전략 질문으로 재작성함"
    },
    {
        "model": "exaone3.5:2.4b",
        "test_type": "Rewrite",
        "case_id": "RW-04",
        "manual_result": "실패",
        "memo": "비교 대상 부족 상황을 재작성 가능으로 오판함"
    },

    # phi3.5
    {
        "model": "phi3.5",
        "test_type": "QA",
        "case_id": "QA-01",
        "manual_result": "통과",
        "memo": "금액 자체는 원문 그대로 유지함"
    },
    {
        "model": "phi3.5",
        "test_type": "QA",
        "case_id": "QA-02",
        "manual_result": "실패",
        "memo": "챗봇 시스템을 챕터블러 시스템으로 잘못 출력함"
    },
    {
        "model": "phi3.5",
        "test_type": "QA",
        "case_id": "QA-03",
        "manual_result": "실패",
        "memo": "깨진 표현과 의미 불명확한 문장이 포함됨"
    },
    {
        "model": "phi3.5",
        "test_type": "Clarification",
        "case_id": "CL-01",
        "manual_result": "부분 통과",
        "memo": "추가 확인 필요 판단은 맞지만 표현이 부자연스럽고 요구 정보가 정돈되지 않음"
    },
    {
        "model": "phi3.5",
        "test_type": "Clarification",
        "case_id": "CL-02",
        "manual_result": "부분 통과",
        "memo": "추가 확인 필요 판단은 맞지만 회사 정보와 수행 실적 안내가 부족함"
    },
    {
        "model": "phi3.5",
        "test_type": "Clarification",
        "case_id": "CL-03",
        "manual_result": "실패",
        "memo": "RFP를 공개선서로 잘못 표현하는 등 응답 품질 문제가 있음"
    },
    {
        "model": "phi3.5",
        "test_type": "Clarification",
        "case_id": "CL-04",
        "manual_result": "실패",
        "memo": "판단이 혼재되고 내부 라벨을 노출하며 질문과 맞지 않는 정보를 요구함"
    },
    {
        "model": "phi3.5",
        "test_type": "Rewrite",
        "case_id": "RW-01",
        "manual_result": "부분 통과",
        "memo": "비교 대상은 일부 반영했지만 질문 문장이 부자연스러움"
    },
    {
        "model": "phi3.5",
        "test_type": "Rewrite",
        "case_id": "RW-02",
        "manual_result": "실패",
        "memo": "회사 정보 부족 상황을 재작성 가능으로 오판하고 문장이 부자연스러움"
    },
    {
        "model": "phi3.5",
        "test_type": "Rewrite",
        "case_id": "RW-03",
        "manual_result": "부분 통과",
        "memo": "사업명은 반영했지만 재작성 질문이 단순하고 구체성이 부족함"
    },
    {
        "model": "phi3.5",
        "test_type": "Rewrite",
        "case_id": "RW-04",
        "manual_result": "실패",
        "memo": "비교 대상 부족 상황을 재작성 가능으로 오판하고 문장이 부자연스러움"
    }
]

manual_eval_result_df = pd.DataFrame(manual_eval_records)
manual_eval_result_df

,model,test_type,case_id,manual_result,memo
0,gemma3:4b,QA,QA-01,통과,"금액을 11,270,000,000원으로 원문 그대로 유지함"
1,gemma3:4b,QA,QA-02,통과,구축 대상 시스템 9개 항목을 모두 출력함
2,gemma3:4b,QA,QA-03,통과,"예정가격, 입찰가격, 평점산식 입력값 부족으로 계산하지 않음"
3,gemma3:4b,Clarification,CL-01,부분 통과,"추가 확인 필요 판단은 적절하나 수행 실적, 보유 인력, 기술 역량이 직접적으로 충분히 제시되지는 않음"
4,gemma3:4b,Clarification,CL-02,부분 통과,"추가 확인 필요 판단은 적절하나 회사 정보, 수행 전략, 실적 안내가 부족함"
5,gemma3:4b,Clarification,CL-03,통과,모호한 요약 요청에 대해 대상 내용과 핵심 기준 확인이 필요하다고 안내함
6,gemma3:4b,Clarification,CL-04,부분 통과,"추가 확인 필요 판단은 맞지만 입찰가격, 예정가격, 산식을 직접적으로 모두 짚지는 못함"
7,gemma3:4b,Rewrite,RW-01,통과,비교 대상이 명확한 상황에서 재작성 가능으로 판단하고 비교 질문을 생성함
8,gemma3:4b,Rewrite,RW-02,부분 통과,회사 정보 부족을 안내했지만 판단이 재작성 가능 / 추가 확인 필요로 혼재됨
9,gemma3:4b,Rewrite,RW-03,통과,사업명을 반영하여 수행 전략 질문으로 재작성함


In [32]:
manual_eval_summary_df = manual_eval_result_df.groupby(
    ["model", "manual_result"]
).size().reset_index(name="count")

manual_eval_summary_df

,model,manual_result,count
0,exaone3.5:2.4b,부분 통과,6
1,exaone3.5:2.4b,실패,1
2,exaone3.5:2.4b,통과,4
3,gemma3:4b,부분 통과,4
4,gemma3:4b,실패,1
5,gemma3:4b,통과,6
6,llama3.2:3b,부분 통과,2
7,llama3.2:3b,실패,7
8,llama3.2:3b,통과,2
9,phi3.5,부분 통과,4


In [33]:
manual_eval_by_type_df = manual_eval_result_df.groupby(
    ["model", "test_type", "manual_result"]
).size().reset_index(name="count")

manual_eval_by_type_df.sort_values(["model", "test_type", "manual_result"])

,model,test_type,manual_result,count
0,exaone3.5:2.4b,Clarification,부분 통과,3
1,exaone3.5:2.4b,Clarification,통과,1
2,exaone3.5:2.4b,QA,부분 통과,1
3,exaone3.5:2.4b,QA,통과,2
4,exaone3.5:2.4b,Rewrite,부분 통과,2
5,exaone3.5:2.4b,Rewrite,실패,1
6,exaone3.5:2.4b,Rewrite,통과,1
7,gemma3:4b,Clarification,부분 통과,3
8,gemma3:4b,Clarification,통과,1
9,gemma3:4b,QA,통과,3


In [34]:
score_map = {
    "통과": 1.0,
    "부분 통과": 0.5,
    "실패": 0.0
}

manual_eval_result_df["score"] = manual_eval_result_df["manual_result"].map(score_map)

model_score_summary_df = manual_eval_result_df.groupby("model")["score"].agg(
    test_count="count",
    total_score="sum",
    average_score="mean"
).reset_index()

model_score_summary_df = model_score_summary_df.sort_values(
    "average_score",
    ascending=False
)

model_score_summary_df

,model,test_count,total_score,average_score
1,gemma3:4b,11,8.0,0.727273
4,qwen2.5:3b,11,7.5,0.681818
0,exaone3.5:2.4b,11,7.0,0.636364
2,llama3.2:3b,11,3.0,0.272727
3,phi3.5,11,3.0,0.272727


## 수동 평가 결과 해석

수동 평가 결과, 전체 11개 테스트 케이스 기준으로 `gemma3:4b`가 평균 점수 0.727로 가장 높은 결과를 보였습니다. 그다음은 `qwen2.5:3b`가 평균 점수 0.682, `exaone3.5:2.4b`가 평균 점수 0.636으로 뒤를 이었습니다. `llama3.2:3b`와 `phi3.5`는 각각 평균 점수 0.273으로 낮은 결과를 보였습니다.

### 1. 전체 평가 요약

`gemma3:4b`는 총 11개 테스트 중 통과 6건, 부분 통과 4건, 실패 1건으로 가장 안정적인 결과를 보였습니다. 특히 문서 기반 QA 테스트에서는 3개 케이스를 모두 통과하였고, 정보부족 판단에서도 대부분의 케이스를 추가 확인 필요로 적절하게 판단했습니다. 다만 질문 재작성 테스트에서는 비교 대상이 부족한 상황을 재작성 가능으로 오판한 사례가 있어 보완이 필요합니다.

`qwen2.5:3b`는 총 11개 테스트 중 통과 4건, 부분 통과 7건, 실패 0건으로 평가되었습니다. 실패로 분류된 케이스는 없었지만, 부분 통과가 많았습니다. 이는 모델이 큰 방향은 대체로 맞게 판단하지만, 출력 형식이 혼재되거나 추가 확인 정보가 충분히 구체적이지 않은 경우가 많았기 때문입니다.

`exaone3.5:2.4b`는 총 11개 테스트 중 통과 4건, 부분 통과 6건, 실패 1건으로 평가되었습니다. QA와 정보부족 판단에서는 기본적인 가능성을 보였지만, 질문 재작성 테스트에서 비교 대상이 부족한 상황을 재작성 가능으로 오판한 문제가 확인되었습니다. 또한 일부 응답에서 영어 표현이나 과도한 추가 정보가 포함되어 사용자 응답용으로는 보완이 필요합니다.

`llama3.2:3b`는 총 11개 테스트 중 통과 2건, 부분 통과 2건, 실패 7건으로 평가되었습니다. 문서 기반 QA 일부에서는 통과했지만, 정보부족 판단 테스트 4개 케이스를 모두 실패했습니다. 특히 추가 확인이 필요한 상황을 `답변 가능`으로 오판하거나, 외국어 표현이 혼입되는 문제가 확인되었습니다.

`phi3.5`는 총 11개 테스트 중 통과 1건, 부분 통과 4건, 실패 6건으로 평가되었습니다. 금액 보존 테스트는 통과했지만, 항목명 오류, 깨진 표현, 판단 혼재, 내부 라벨 노출, 부자연스러운 문장 등이 확인되었습니다. 따라서 안정적인 Generation 모델 후보로 보기 어렵습니다.

### 2. 테스트 유형별 평가

문서 기반 QA 테스트에서는 `gemma3:4b`와 `qwen2.5:3b`가 3개 케이스를 모두 통과했습니다. 두 모델 모두 금액 단위 보존, 항목 나열, 가격점수 계산 가드레일을 안정적으로 처리했습니다.

`exaone3.5:2.4b`도 QA 테스트에서 통과 2건, 부분 통과 1건으로 비교적 양호한 결과를 보였습니다. 가격점수를 임의 계산하지 않은 점은 적절했지만, 부족 정보 안내에서 입찰가격이 명확히 빠진 점이 있어 부분 통과로 평가되었습니다.

`llama3.2:3b`는 QA 테스트에서 통과 2건, 실패 1건을 기록했습니다. 금액 보존과 항목 나열은 수행했지만, 가격점수 가드레일 응답에서 외국어 표현이 혼입되어 실패로 평가되었습니다.

`phi3.5`는 QA 테스트에서 통과 1건, 실패 2건으로 낮은 결과를 보였습니다. 금액 자체는 보존했지만, 항목명을 잘못 출력하거나 깨진 표현이 포함되어 문서 기반 QA 응답 안정성이 부족했습니다.

정보부족 판단 테스트에서는 `gemma3:4b`, `qwen2.5:3b`, `exaone3.5:2.4b`가 모두 추가 확인 필요 방향으로 판단했습니다. 다만 세 모델 모두 일부 케이스에서 필요한 정보가 충분히 구체적으로 제시되지 않아 부분 통과가 많았습니다.

`llama3.2:3b`는 정보부족 판단 테스트 4개 케이스를 모두 실패했습니다. 추가 확인이 필요한 질문을 답변 가능으로 오판하거나, 추가 정보 없음으로 출력한 점이 가장 큰 문제였습니다.

`phi3.5`는 정보부족 판단에서 부분 통과 2건, 실패 2건을 기록했습니다. 일부 케이스에서는 추가 확인 필요 방향은 맞았지만, 표현이 부자연스럽고 질문과 맞지 않는 정보를 요구하거나 내부 라벨을 노출하는 문제가 있었습니다.

질문 재작성 테스트에서는 모든 모델에서 일정 수준의 문제가 확인되었습니다. `gemma3:4b`는 통과 2건, 부분 통과 1건, 실패 1건으로 가장 나은 편이었지만, 비교 대상이 부족한 상황을 재작성 가능으로 판단한 점은 보완이 필요합니다.

`qwen2.5:3b`는 질문 재작성에서 통과 1건, 부분 통과 3건으로 평가되었습니다. 실패는 없었지만, 판단값이 혼재되거나 추가 확인 필요 상황에서도 재작성 질문을 함께 출력하는 문제가 있었습니다.

`exaone3.5:2.4b`는 질문 재작성에서 통과 1건, 부분 통과 2건, 실패 1건으로 평가되었습니다. 사업명이 명확한 수행 전략 질문은 잘 처리했지만, 비교 대상이 부족한 상황을 재작성 가능으로 오판했습니다.

`llama3.2:3b`와 `phi3.5`는 질문 재작성에서 정보부족 상황을 재작성 가능으로 오판하거나 외국어 혼입, 부자연스러운 문장이 나타나 안정성이 낮았습니다.

### 3. 모델별 활용 가능성

현재 수동 평가 결과를 기준으로 하면 `gemma3:4b`는 품질 중심 주력 후보로 볼 수 있습니다. QA 성능이 가장 안정적이고, 정보부족 판단에서도 방향성이 비교적 적절했습니다. 다만 질문 재작성에서 정보가 부족한 상황을 더 엄격하게 판단하도록 프롬프트 보완이 필요합니다.

`qwen2.5:3b`는 속도 중심 대안 후보로 볼 수 있습니다. 실패 케이스가 없고 전반적인 방향성은 안정적이지만, 부분 통과가 많아 출력 형식과 추가 정보 구체성을 개선할 필요가 있습니다.

`exaone3.5:2.4b`는 보조 후보 또는 추가 검증 후보로 볼 수 있습니다. QA와 정보부족 판단에서는 가능성이 있지만, 질문 재작성과 사용자 응답 형식에서 보완이 필요합니다.

`llama3.2:3b`는 정보부족 판단 실패가 많고 외국어 혼입 문제가 확인되어 주요 후보에서 제외하는 것이 적절합니다.

`phi3.5`는 항목명 오류, 깨진 표현, 내부 라벨 노출, 판단 혼재 문제가 확인되어 현재 결과 기준으로는 최종 후보에서 제외하는 것이 적절합니다.

### 4. 현재 단계 결론

수동 평가 결과를 종합하면, 현재 단계에서는 `gemma3:4b`를 품질 중심 주력 후보로, `qwen2.5:3b`를 속도 중심 대안 후보로 정리할 수 있습니다. `exaone3.5:2.4b`는 일부 가능성은 있으나 단독 최종 후보로 보기보다는 보조 후보 또는 추가 검증 후보로 분류하는 것이 적절합니다.

반면 `llama3.2:3b`와 `phi3.5`는 정보부족 판단 오류, 외국어 혼입, 표현 품질 문제, 판단 혼재가 확인되었으므로 현재 결과 기준에서는 주요 후보에서 제외하는 것이 타당합니다.

다만 본 평가는 제한된 테스트 케이스를 기준으로 한 1차 수동 평가이므로, 실제 RAG 검색 결과 context가 연결된 이후에는 동일한 평가 기준으로 다시 검증해야 합니다.

## 사용자 응답형 정보부족 안내 테스트

앞선 테스트에서는 모델이 질문의 완전성을 판단하고, 필요한 경우 추가 확인이 필요하다고 분류할 수 있는지 확인하였습니다.

그러나 실제 서비스에서는 `판단: 추가 확인 필요`과 같은 내부 판단 결과를 그대로 사용자에게 보여주기 어렵습니다. 사용자는 자연스러운 문장으로 현재 답변이 어려운 이유와 추가로 제공해야 할 정보를 안내받아야 합니다.

따라서 본 단계에서는 사용자에게 직접 보여줄 수 있는 정보부족 안내 응답을 생성하는 프롬프트를 테스트합니다.

이 테스트의 목표는 다음과 같습니다.

- 내부 판단 라벨을 노출하지 않는지 확인
- 부족한 정보를 자연스러운 문장으로 안내하는지 확인
- 사용자가 다음에 무엇을 입력해야 하는지 구체적으로 알려주는지 확인
- 외국어 혼입, 깨진 표현, 과도한 전문 용어가 없는지 확인
- 실제 챗봇 응답으로 사용할 수 있는 문체인지 확인

In [35]:
user_facing_clarification_prompt = """
너는 RFP 분석 서비스를 사용하는 사용자에게 답변하는 AI입니다.

사용자의 질문에 답변하기 위해 정보가 부족한 경우,
내부 판단 라벨을 보여주지 말고 사용자에게 자연스럽게 추가 정보를 요청하세요.

규칙:
1. "판단:", "추가 확인 필요", "답변 가능" 같은 내부 라벨을 출력하지 마세요.
2. 사용자의 질문에 바로 답하기 어려운 이유를 간단히 설명하세요.
3. 추가로 필요한 정보를 구체적으로 안내하세요.
4. 사용자가 다음에 입력하면 좋은 예시 질문을 1개 제시하세요.
5. 문장은 한국어 존댓말로 작성하세요.
6. 외국어를 섞지 마세요.
7. 너무 길게 쓰지 말고, 4~6문장 정도로 답변하세요.
8. 문서에 없는 내용은 추측하지 마세요.

[사용자 질문]
{question}

[사용자에게 보여줄 답변]
"""

In [36]:
user_facing_test_questions = [
    {
        "case_id": "UF-01",
        "category": "입찰 적합도 안내",
        "question": "우리 회사가 이 입찰에 적합한지 봐줘.",
        "check_points": [
            "회사 정보 필요 안내",
            "수행 실적 또는 보유 인력 안내",
            "내부 라벨 미노출",
            "자연스러운 존댓말"
        ]
    },
    {
        "case_id": "UF-02",
        "category": "제안서 작성 안내",
        "question": "제안서에 쓸 만하게 정리해줘.",
        "check_points": [
            "사업 정보 필요 안내",
            "회사 정보 필요 안내",
            "작성 방향 안내",
            "내부 라벨 미노출"
        ]
    },
    {
        "case_id": "UF-03",
        "category": "모호한 요약 요청 안내",
        "question": "이거 핵심만 뽑아줘.",
        "check_points": [
            "대상 문서 또는 내용 필요 안내",
            "요약 범위 안내",
            "예시 질문 제시",
            "자연스러운 문체"
        ]
    },
    {
        "case_id": "UF-04",
        "category": "가격점수 계산 안내",
        "question": "가격점수 계산해줘.",
        "check_points": [
            "입찰가격 필요 안내",
            "예정가격 필요 안내",
            "평점산식 필요 안내",
            "임의 계산 금지"
        ]
    }
]

user_facing_test_df = pd.DataFrame(user_facing_test_questions)
user_facing_test_df

,case_id,category,question,check_points
0,UF-01,입찰 적합도 안내,우리 회사가 이 입찰에 적합한지 봐줘.,"[회사 정보 필요 안내, 수행 실적 또는 보유 인력 안내, 내부 라벨 미노출, 자연스러운 존댓말]"
1,UF-02,제안서 작성 안내,제안서에 쓸 만하게 정리해줘.,"[사업 정보 필요 안내, 회사 정보 필요 안내, 작성 방향 안내, 내부 라벨 미노출]"
2,UF-03,모호한 요약 요청 안내,이거 핵심만 뽑아줘.,"[대상 문서 또는 내용 필요 안내, 요약 범위 안내, 예시 질문 제시, 자연스러운 문체]"
3,UF-04,가격점수 계산 안내,가격점수 계산해줘.,"[입찰가격 필요 안내, 예정가격 필요 안내, 평점산식 필요 안내, 임의 계산 금지]"


In [37]:
user_facing_results = []

for model in target_models:
    print(f"사용자 응답형 안내 테스트 실행 중: {model}")
    
    for item in user_facing_test_questions:
        prompt = user_facing_clarification_prompt.format(
            question=item["question"]
        )
        
        result = ask_ollama(
            prompt=prompt,
            model=model,
            temperature=0.0,
            timeout=120
        )
        
        user_facing_results.append({
            "model": model,
            "test_type": "User-Facing Clarification",
            "case_id": item["case_id"],
            "category": item["category"],
            "question": item["question"],
            "check_points": ", ".join(item["check_points"]),
            "success": result["success"],
            "elapsed": round(result["elapsed"], 3),
            "answer": result["answer"],
            "error": result["error"]
        })

user_facing_results_df = pd.DataFrame(user_facing_results)
user_facing_results_df

사용자 응답형 안내 테스트 실행 중: gemma3:4b
사용자 응답형 안내 테스트 실행 중: qwen2.5:3b
사용자 응답형 안내 테스트 실행 중: llama3.2:3b
사용자 응답형 안내 테스트 실행 중: exaone3.5:2.4b
사용자 응답형 안내 테스트 실행 중: phi3.5


,model,test_type,case_id,category,question,check_points,success,elapsed,answer,error
0,gemma3:4b,User-Facing Clarification,UF-01,입찰 적합도 안내,우리 회사가 이 입찰에 적합한지 봐줘.,"회사 정보 필요 안내, 수행 실적 또는 보유 인력 안내, 내부 라벨 미노출, 자연스러운 존댓말",True,8.567,"죄송합니다. 현재 귀사의 정보가 없어, 해당 입찰에 적합한지 판단하기 어렵습니다. \n\n귀사의 사업 분야, 주요 제품/서비스, 그리고 이번 입찰과 관련된 구체적인 요구사항을 파악해야 정확한 분석이 가능합니다. 예를 들어, 입찰 공고서의 상세 내용을 알려주시거나, 귀사의 관련 자료를 공유해주시면 더욱 자세하게 검토할 수 있습니다. \n\n좀 더 정확한 답변을 위해, 귀사의 사업 분야와 이번 입찰과 관련된 주요 내용에 대한 정보를 추가로 제공해 주실 수 있을까요? 예를 들어 “저희 회사는 IT 솔루션 개발 회사이며, 이번 입찰은 정부기관의 클라우드 서비스 구축 프로젝트입니다.” 와 같이 말씀해주시면 감사하겠습니다.",
1,gemma3:4b,User-Facing Clarification,UF-02,제안서 작성 안내,제안서에 쓸 만하게 정리해줘.,"사업 정보 필요 안내, 회사 정보 필요 안내, 작성 방향 안내, 내부 라벨 미노출",True,4.165,"죄송합니다. 제안서 작성에 필요한 내용을 정리해 드리는 데는 몇 가지 추가 정보가 필요합니다. 어떤 종류의 RFP(Request for Proposal, 제안 요청서) 분석을 원하시는지 알려주시면 더욱 정확한 도움을 드릴 수 있습니다.\n\n예를 들어, RFP의 주제, 목표, 주요 평가 기준 등을 구체적으로 말씀해주실 수 있을까요? 또한, 현재까지 준비하신 자료나 관련 문서가 있다면 공유해 주시면 제안서 작성에 도움이 될 만한 정보를 파악하는 데 큰 도움이 될 것입니다. \n\n다음 질문 예시: “이번 RFP는 ‘신규 스마트폰 개발’과 관련된 것으로, 주요 평가 기준은 기술 혁신성, 시장 경쟁력, 가격 등이 있습니다.”",
2,gemma3:4b,User-Facing Clarification,UF-03,모호한 요약 요청 안내,이거 핵심만 뽑아줘.,"대상 문서 또는 내용 필요 안내, 요약 범위 안내, 예시 질문 제시, 자연스러운 문체",True,4.002,"죄송합니다. 현재 요청하신 내용의 핵심을 정확하게 파악하기에는 정보가 부족합니다. 어떤 RFP(제안요청서)에 대한 내용을 요약해 드릴지, 그리고 어떤 부분을 중심으로 핵심을 뽑아드릴지 구체적으로 알려주시면 더욱 정확한 답변을 제공해 드릴 수 있습니다. \n\n예를 들어, “OOO RFP의 기술적 요구사항과 평가 기준만 핵심으로 요약해주세요.” 와 같이 질문해주시면 제가 더 적절하게 도움을 드릴 수 있습니다. 다음 질문 예시: “요청하신 RFP의 주요 내용 중 가장 중요한 부분을 알려주세요.”",
3,gemma3:4b,User-Facing Clarification,UF-04,가격점수 계산 안내,가격점수 계산해줘.,"입찰가격 필요 안내, 예정가격 필요 안내, 평점산식 필요 안내, 임의 계산 금지",True,4.195,"죄송합니다. 가격 점수를 계산하려면 몇 가지 추가 정보가 필요합니다. 현재 제공해주신 내용만으로는 어떤 기준으로 가격을 평가하고, 어떤 데이터를 사용하여 점수를 매기기로 결정했는지 알 수 없어서 정확한 계산이 어렵습니다. \n\n좀 더 자세히 알려주시면 감사하겠습니다. 예를 들어, 각 제안사의 가격과 함께, 가격 평가에 사용될 주요 항목(예: 기술 수준, 서비스 품질, 납기일 등) 및 각 항목별 가중치를 구체적으로 말씀해주시거나, 어떤 방식으로 점수를 계산해야 하는지 설명해주시면 됩니다. \n\n다음 질문 예시는 “각 항목별 가중치와 점수 산정 기준을 자세히 알려주세요.” 입니다.",
4,qwen2.5:3b,User-Facing Clarification,UF-01,입찰 적합도 안내,우리 회사가 이 입찰에 적합한지 봐줘.,"회사 정보 필요 안내, 수행 실적 또는 보유 인력 안내, 내부 라벨 미노출, 자연스러운 존댓말",True,4.641,"판정을 내리는 데 필요한 정보가 부족합니다. 이 입찰에서 요구하는 기술적 능력, 경험이나 전문성을 포함해 우리 회사가 어떤 면에서 적합한지에 대해 자세히 설명해 주시면 감사하겠습니다.",
5,qwen2.5:3b,User-Facing Clarification,UF-02,제안서 작성 안내,제안서에 쓸 만하게 정리해줘.,"사업 정보 필요 안내, 회사 정보 필요 안내, 작성 방향 안내, 내부 라벨 미노출",True,2.839,"판정: 제안서를 정리하는 데 필요한 정보가 아직 부족합니다. 사용자가 어떤 주제나 분야의 제안서를 정리하고 싶으신지, 또는 어떤 요소들을 포함해야 하는지 알려주시면 도와드릴 수 있습니다. 예를 들어, 기술적인 특성, 비용, 시간대, 성과 등이 있을 수 있습니다.",
6,qwen2.5:3b,User-Facing Clarification,UF-03,모호한 요약 요청 안내,이거 핵심만 뽑아줘.,"대상 문서 또는 내용 필요 안내, 요약 범위 안내, 예시 질문 제시, 자연스러운 문체",True,2.839,"죄송합니다, 사용자의 질문을 정확히 이해하고 필요한 정보를 추가로 수집하기 위해 조금 더 구체적인 정보가 필요합니다. 핵심 내용을 뽑아드릴 수 있도록 도와주시면 감사하겠습니다. 예를 들어, RFP 문서의 주요 챕터나 특정 조건에 대해 질문해 주시면 좋겠습니다.",
7,qwen2.5:3b,User-Facing Clarification,UF-04,가격점수 계산 안내,가격점수 계산해줘.,"입찰가격 필요 안내, 예정가격 필요 안내, 평점산식 필요 안내, 임의 계산 금지",True,3.091,"가격점수를 계산하기 위해서는, 현재 사용 중인 가격과 비교 대상의 가격이 필요합니다. 또한, 서비스의 품질이나 성능 등 다른 요소도 고려해야 합니다. 다음에 제공해주시면 감사하겠습니다. 예를 들어, ""현재 사용 중인 서비스의 가격과 비교 대상의 가격을 알려주실 수 있으신가요?""라고 질문드릴게요.",
8,llama3.2:3b,User-Facing Clarification,UF-01,입찰 적합도 안내,우리 회사가 이 입찰에 적합한지 봐줘.,"회사 정보 필요 안내, 수행 실적 또는 보유 인력 안내, 내부 라벨 미노출, 자연스러운 존댓말",True,6.923,"저는 사용자의 질문에 대한 정보가 부족하여 추가적인 정보를 필요로합니다.\n\n우리의 회사는 어떤 산업이나 분야에 있나요? 또한, 이 입찰의 목적과 requirements는 무엇인가요? 이러한 정보가 필요할 때까지 더 많은 details를 알려주세요.",
9,llama3.2:3b,User-Facing Clarification,UF-02,제안서 작성 안내,제안서에 쓸 만하게 정리해줘.,"사업 정보 필요 안내, 회사 정보 필요 안내, 작성 방향 안내, 내부 라벨 미노출",True,6.676,"사용자의 질문에 대한 답변을 제공하기 위해서는 사용자가 제안서의 주요 내용과 목적을 알려주어야 합니다. 사용자가 이 정보를 제공하지 않으면, 사용자가 다음에 입력할 수 있는 좋은 예시 질문으로 ""제안서의 주요 내용은 무엇입니까?""를 제시할 수 있습니다.\n\n사용자의 질문에 대한 답변은 다음과 같습니다.\n\n제안서에 쓸 만한 내용을 정리해줄 수는 없습니다. 사용자가 제안서의 주요 내용과 목적을 알려주어야 합니다. 사용자가 이 정보를 제공하지 않으면, 제안서의 내용을 정리할 수 없습니다. 사용자가 다음에 입력할 수 있는 좋은 예시 질문으로 ""제안서의 주요 내용은 무엇입니까?""를 제시할 수 있습니다.\n\n사용자에게 추가로 필요한 정보를 구체적으로 안내해 줄 수 있습니다. 사용자가 이 정보를 제공하면, 제안서의 내용을 정리해줄 수 있습니다.",


In [38]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

user_facing_results_df[
    ["model", "case_id", "category", "success", "elapsed", "answer"]
]

,model,case_id,category,success,elapsed,answer
0,gemma3:4b,UF-01,입찰 적합도 안내,True,8.567,"죄송합니다. 현재 귀사의 정보가 없어, 해당 입찰에 적합한지 판단하기 어렵습니다. \n\n귀사의 사업 분야, 주요 제품/서비스, 그리고 이번 입찰과 관련된 구체적인 요구사항을 파악해야 정확한 분석이 가능합니다. 예를 들어, 입찰 공고서의 상세 내용을 알려주시거나, 귀사의 관련 자료를 공유해주시면 더욱 자세하게 검토할 수 있습니다. \n\n좀 더 정확한 답변을 위해, 귀사의 사업 분야와 이번 입찰과 관련된 주요 내용에 대한 정보를 추가로 제공해 주실 수 있을까요? 예를 들어 “저희 회사는 IT 솔루션 개발 회사이며, 이번 입찰은 정부기관의 클라우드 서비스 구축 프로젝트입니다.” 와 같이 말씀해주시면 감사하겠습니다."
1,gemma3:4b,UF-02,제안서 작성 안내,True,4.165,"죄송합니다. 제안서 작성에 필요한 내용을 정리해 드리는 데는 몇 가지 추가 정보가 필요합니다. 어떤 종류의 RFP(Request for Proposal, 제안 요청서) 분석을 원하시는지 알려주시면 더욱 정확한 도움을 드릴 수 있습니다.\n\n예를 들어, RFP의 주제, 목표, 주요 평가 기준 등을 구체적으로 말씀해주실 수 있을까요? 또한, 현재까지 준비하신 자료나 관련 문서가 있다면 공유해 주시면 제안서 작성에 도움이 될 만한 정보를 파악하는 데 큰 도움이 될 것입니다. \n\n다음 질문 예시: “이번 RFP는 ‘신규 스마트폰 개발’과 관련된 것으로, 주요 평가 기준은 기술 혁신성, 시장 경쟁력, 가격 등이 있습니다.”"
2,gemma3:4b,UF-03,모호한 요약 요청 안내,True,4.002,"죄송합니다. 현재 요청하신 내용의 핵심을 정확하게 파악하기에는 정보가 부족합니다. 어떤 RFP(제안요청서)에 대한 내용을 요약해 드릴지, 그리고 어떤 부분을 중심으로 핵심을 뽑아드릴지 구체적으로 알려주시면 더욱 정확한 답변을 제공해 드릴 수 있습니다. \n\n예를 들어, “OOO RFP의 기술적 요구사항과 평가 기준만 핵심으로 요약해주세요.” 와 같이 질문해주시면 제가 더 적절하게 도움을 드릴 수 있습니다. 다음 질문 예시: “요청하신 RFP의 주요 내용 중 가장 중요한 부분을 알려주세요.”"
3,gemma3:4b,UF-04,가격점수 계산 안내,True,4.195,"죄송합니다. 가격 점수를 계산하려면 몇 가지 추가 정보가 필요합니다. 현재 제공해주신 내용만으로는 어떤 기준으로 가격을 평가하고, 어떤 데이터를 사용하여 점수를 매기기로 결정했는지 알 수 없어서 정확한 계산이 어렵습니다. \n\n좀 더 자세히 알려주시면 감사하겠습니다. 예를 들어, 각 제안사의 가격과 함께, 가격 평가에 사용될 주요 항목(예: 기술 수준, 서비스 품질, 납기일 등) 및 각 항목별 가중치를 구체적으로 말씀해주시거나, 어떤 방식으로 점수를 계산해야 하는지 설명해주시면 됩니다. \n\n다음 질문 예시는 “각 항목별 가중치와 점수 산정 기준을 자세히 알려주세요.” 입니다."
4,qwen2.5:3b,UF-01,입찰 적합도 안내,True,4.641,"판정을 내리는 데 필요한 정보가 부족합니다. 이 입찰에서 요구하는 기술적 능력, 경험이나 전문성을 포함해 우리 회사가 어떤 면에서 적합한지에 대해 자세히 설명해 주시면 감사하겠습니다."
5,qwen2.5:3b,UF-02,제안서 작성 안내,True,2.839,"판정: 제안서를 정리하는 데 필요한 정보가 아직 부족합니다. 사용자가 어떤 주제나 분야의 제안서를 정리하고 싶으신지, 또는 어떤 요소들을 포함해야 하는지 알려주시면 도와드릴 수 있습니다. 예를 들어, 기술적인 특성, 비용, 시간대, 성과 등이 있을 수 있습니다."
6,qwen2.5:3b,UF-03,모호한 요약 요청 안내,True,2.839,"죄송합니다, 사용자의 질문을 정확히 이해하고 필요한 정보를 추가로 수집하기 위해 조금 더 구체적인 정보가 필요합니다. 핵심 내용을 뽑아드릴 수 있도록 도와주시면 감사하겠습니다. 예를 들어, RFP 문서의 주요 챕터나 특정 조건에 대해 질문해 주시면 좋겠습니다."
7,qwen2.5:3b,UF-04,가격점수 계산 안내,True,3.091,"가격점수를 계산하기 위해서는, 현재 사용 중인 가격과 비교 대상의 가격이 필요합니다. 또한, 서비스의 품질이나 성능 등 다른 요소도 고려해야 합니다. 다음에 제공해주시면 감사하겠습니다. 예를 들어, ""현재 사용 중인 서비스의 가격과 비교 대상의 가격을 알려주실 수 있으신가요?""라고 질문드릴게요."
8,llama3.2:3b,UF-01,입찰 적합도 안내,True,6.923,"저는 사용자의 질문에 대한 정보가 부족하여 추가적인 정보를 필요로합니다.\n\n우리의 회사는 어떤 산업이나 분야에 있나요? 또한, 이 입찰의 목적과 requirements는 무엇인가요? 이러한 정보가 필요할 때까지 더 많은 details를 알려주세요."
9,llama3.2:3b,UF-02,제안서 작성 안내,True,6.676,"사용자의 질문에 대한 답변을 제공하기 위해서는 사용자가 제안서의 주요 내용과 목적을 알려주어야 합니다. 사용자가 이 정보를 제공하지 않으면, 사용자가 다음에 입력할 수 있는 좋은 예시 질문으로 ""제안서의 주요 내용은 무엇입니까?""를 제시할 수 있습니다.\n\n사용자의 질문에 대한 답변은 다음과 같습니다.\n\n제안서에 쓸 만한 내용을 정리해줄 수는 없습니다. 사용자가 제안서의 주요 내용과 목적을 알려주어야 합니다. 사용자가 이 정보를 제공하지 않으면, 제안서의 내용을 정리할 수 없습니다. 사용자가 다음에 입력할 수 있는 좋은 예시 질문으로 ""제안서의 주요 내용은 무엇입니까?""를 제시할 수 있습니다.\n\n사용자에게 추가로 필요한 정보를 구체적으로 안내해 줄 수 있습니다. 사용자가 이 정보를 제공하면, 제안서의 내용을 정리해줄 수 있습니다."


## 사용자 응답형 안내 테스트 결과 해석

사용자 응답형 정보부족 안내 테스트에서는 내부 판단 결과를 그대로 출력하지 않고, 실제 사용자에게 보여줄 수 있는 자연스러운 안내 문장을 생성할 수 있는지 확인하였습니다.

전체 모델 모두 `success=True`가 반환되어 API 호출 자체는 정상적으로 수행되었습니다. 그러나 실제 응답 품질에서는 모델별 차이가 크게 나타났습니다.

### 1. 내부 라벨 노출 여부

`gemma3:4b`는 대부분의 응답에서 `판단:`, `추가 확인 필요`와 같은 내부 라벨을 노출하지 않았습니다. 사용자에게 정보가 부족한 이유를 설명하고, 추가로 제공해야 할 정보를 자연스럽게 요청하는 형태로 답변했습니다.

`qwen2.5:3b`는 일부 응답에서 내부 라벨에 가까운 표현이 남았습니다. 특히 제안서 작성 안내 케이스에서 `판정:`이라는 표현을 출력했기 때문에, 사용자 응답형 프롬프트의 “내부 라벨 미노출” 규칙을 완전히 지키지는 못했습니다.

`llama3.2:3b`는 명시적인 `판단:` 라벨은 적게 노출했지만, 응답이 시스템 설명처럼 장황하게 흐르거나 “사용자의 질문에 대한 답변은 다음과 같습니다.” 같은 불필요한 메타 문장이 포함되었습니다. 사용자에게 바로 보여주기에는 자연스럽지 않습니다.

`exaone3.5:2.4b`는 내부 라벨 노출은 거의 없었고, 전반적으로 사용자 안내문 형태를 유지했습니다. 다만 일부 응답에서 `RFP(Request for Proposal)`과 같은 영어 설명이 포함되었고, 답변 길이가 다소 장황한 경향이 있었습니다.

`phi3.5`는 대부분의 응답에서 `판단:`, `추가 확인 필요`, `답변:`, `문장화:` 같은 내부 라벨 또는 메타 표현을 그대로 노출했습니다. 따라서 사용자 응답형 안내에는 적합하지 않은 결과를 보였습니다.

### 2. 추가 정보 안내의 구체성

입찰 적합도 안내에서는 `gemma3:4b`가 회사의 사업 분야, 주요 제품/서비스, 입찰 관련 요구사항을 요청했습니다. 다만 수행 실적이나 보유 인력까지 직접적으로 충분히 언급하지는 않아 일부 보완 여지가 있습니다.

`qwen2.5:3b`는 기술적 능력, 경험, 전문성을 요구하여 방향은 적절했지만, 응답이 짧고 예시 질문이 충분히 구체적이지 않았습니다.

`exaone3.5:2.4b`는 회사의 역량과 경험, 예산 및 자원, 목표와 기대치를 요구하여 비교적 구체적인 편이었습니다. 다만 입찰 적합도 판단에서 필요한 수행 실적, 보유 인력, 기술 역량을 더 직접적으로 안내하면 더 적절할 것으로 보입니다.

제안서 작성 안내에서는 `gemma3:4b`가 RFP의 주제, 목표, 주요 평가 기준, 준비 자료를 요청했습니다. 그러나 회사의 강점, 수행 실적, 수행 전략과 같은 제안서 작성 핵심 정보는 상대적으로 약하게 제시되었습니다.

`exaone3.5:2.4b`는 프로젝트 목표와 범위, 현재 상황 분석, 제안 내용, 예상 효과, 실행 계획 등을 안내하여 내용은 풍부했습니다. 다만 답변이 길고 일반 컨설팅 문장처럼 흐르는 경향이 있어, 실제 챗봇 응답으로는 더 짧게 정리하는 것이 좋습니다.

모호한 요약 요청에서는 `gemma3:4b`와 `exaone3.5:2.4b`가 대상 RFP 문서와 요약 범위가 필요하다는 점을 비교적 명확히 안내했습니다. `qwen2.5:3b`도 추가 정보가 필요하다고 답했지만, 표현이 다소 일반적이었습니다.

가격점수 계산 안내에서는 전반적으로 아쉬움이 있었습니다. `gemma3:4b`는 가격 평가 기준과 가중치, 제안사 가격을 요청했지만, 입찰가격·예정가격·평점산식이라는 핵심 표현을 직접적으로 모두 포함하지는 않았습니다. `qwen2.5:3b`는 현재 가격과 비교 대상 가격을 요청했지만, RFP의 가격점수 계산 맥락과는 다소 맞지 않았습니다. `exaone3.5:2.4b` 역시 요구사항, 경쟁사 분석, 프로젝트 규모를 중심으로 안내하여 가격점수 계산에 필요한 입찰가격, 예정가격, 평점산식을 직접적으로 요구하지 못했습니다.

### 3. 사용자 응답 품질

`gemma3:4b`는 한국어 존댓말 문체가 자연스럽고, 사용자에게 어떤 정보를 추가로 제공해야 하는지 비교적 친절하게 안내했습니다. 다만 일부 응답에 `RFP(Request for Proposal, 제안 요청서)`처럼 영어 설명이 들어갔고, 답변이 조금 길어지는 경향이 있었습니다.

`qwen2.5:3b`는 응답 속도가 빠르고 문장이 비교적 짧다는 장점이 있습니다. 그러나 안내 정보가 다소 부족하고, 일부 케이스에서 `판정:`이라는 내부 라벨이 노출되었습니다. 따라서 사용자 응답용으로 사용하려면 내부 라벨 제거와 안내 항목 구체화가 필요합니다.

`llama3.2:3b`는 사용자 응답형 안내에서 외국어 혼입 문제가 뚜렷하게 나타났습니다. `requirements`, `details`, `tool`, `price score` 같은 영어 표현뿐만 아니라, 일부 케이스에서는 일본어와 힌디어로 보이는 문자도 섞였습니다. 또한 가격점수 계산 안내에서는 임의 계산 방식 예시를 제시하여, 임의 계산 금지 관점에서도 위험합니다.

`exaone3.5:2.4b`는 사용자에게 보여줄 수 있는 안내문 형태를 비교적 잘 유지했습니다. 문체도 자연스럽고 정보 안내도 풍부했습니다. 다만 전체적으로 답변이 길고, RFP 영어 설명이 반복적으로 포함되며, 가격점수 계산에 필요한 핵심 입력값을 직접적으로 짚지 못한 점은 보완이 필요합니다.

`phi3.5`는 사용자 응답형 안내 테스트에서 가장 큰 문제가 확인되었습니다. 내부 라벨이 그대로 출력되었고, 외국어와 깨진 표현이 다수 포함되었으며, 가격점수 계산 안내에서는 응답이 매우 길게 반복되었습니다. 특히 일본어, 영어, 깨진 한국어가 섞이고 동일한 문장이 반복되어 실제 사용자 응답으로 사용할 수 없습니다.

### 4. 현재 단계 판단

사용자 응답형 안내 테스트 기준으로는 `gemma3:4b`가 가장 안정적인 후보입니다. 내부 라벨 노출이 거의 없고, 한국어 존댓말 문체가 자연스러우며, 사용자에게 추가 정보를 요청하는 방식도 비교적 적절했습니다.

`exaone3.5:2.4b`도 사용자 응답형 안내문 생성 능력은 양호한 편입니다. 다만 답변이 길고 영어 설명이 포함되는 경향이 있어, 최종 사용자 응답용으로 사용하려면 길이 제한과 외국어 제거 규칙이 필요합니다.

`qwen2.5:3b`는 응답 속도와 간결성은 장점이지만, 정보 안내가 다소 부족하고 내부 라벨이 일부 노출되었습니다. 따라서 속도 중심 대안 후보로는 검토할 수 있으나, 사용자 응답형 프롬프트를 더 강하게 제어해야 합니다.

`llama3.2:3b`는 외국어 혼입, 장황한 설명, 임의 계산 방식 제시 문제가 있어 사용자 응답형 안내에는 부적합합니다.

`phi3.5`는 내부 라벨 노출, 깨진 표현, 외국어 혼입, 반복 출력, 과도한 응답 길이 문제가 모두 확인되어 사용자 응답형 안내 모델로 사용하기 어렵습니다.

따라서 사용자 응답형 안내까지 포함한 현재 단계에서는 `gemma3:4b`를 품질 중심 주력 후보로 유지하고, `qwen2.5:3b`는 속도 중심 대안 후보로 보되 출력 형식 보완이 필요합니다. `exaone3.5:2.4b`는 보조 후보 또는 추가 검증 후보로 분류하는 것이 적절합니다.

In [39]:
user_facing_manual_eval_records = [
    # gemma3:4b
    {
        "model": "gemma3:4b",
        "case_id": "UF-01",
        "manual_result": "부분 통과",
        "memo": "자연스러운 존댓말 응답이며 내부 라벨은 없지만 수행 실적, 보유 인력 안내가 부족함"
    },
    {
        "model": "gemma3:4b",
        "case_id": "UF-02",
        "manual_result": "부분 통과",
        "memo": "제안서 작성에 필요한 추가 정보를 안내했지만 회사 정보와 수행 전략 안내가 부족하고 영어 설명이 포함됨"
    },
    {
        "model": "gemma3:4b",
        "case_id": "UF-03",
        "manual_result": "통과",
        "memo": "대상 문서와 요약 범위가 필요하다는 점을 자연스럽게 안내함"
    },
    {
        "model": "gemma3:4b",
        "case_id": "UF-04",
        "manual_result": "부분 통과",
        "memo": "임의 계산은 하지 않았지만 입찰가격, 예정가격, 평점산식이라는 핵심 입력값을 직접적으로 모두 안내하지 못함"
    },

    # qwen2.5:3b
    {
        "model": "qwen2.5:3b",
        "case_id": "UF-01",
        "manual_result": "부분 통과",
        "memo": "필요 정보 부족을 안내했지만 응답이 짧고 수행 실적, 보유 인력 안내가 부족함"
    },
    {
        "model": "qwen2.5:3b",
        "case_id": "UF-02",
        "manual_result": "실패",
        "memo": "판정이라는 내부 라벨이 노출되고 회사 정보 안내가 부족함"
    },
    {
        "model": "qwen2.5:3b",
        "case_id": "UF-03",
        "manual_result": "부분 통과",
        "memo": "추가 정보 필요성을 안내했지만 대상 문서와 요약 범위 안내가 다소 일반적임"
    },
    {
        "model": "qwen2.5:3b",
        "case_id": "UF-04",
        "manual_result": "실패",
        "memo": "입찰가격, 예정가격, 평점산식을 직접 안내하지 못하고 현재 사용 중인 가격/비교 대상 가격으로 일반화함"
    },

    # llama3.2:3b
    {
        "model": "llama3.2:3b",
        "case_id": "UF-01",
        "manual_result": "실패",
        "memo": "requirements, details 등 영어 표현이 혼입되고 문체가 부자연스러움"
    },
    {
        "model": "llama3.2:3b",
        "case_id": "UF-02",
        "manual_result": "실패",
        "memo": "응답이 장황하고 반복적이며 사용자 응답형 문장으로 부적절함"
    },
    {
        "model": "llama3.2:3b",
        "case_id": "UF-03",
        "manual_result": "실패",
        "memo": "영어, 일본어, 힌디어로 보이는 문자가 혼입되어 사용자 응답으로 부적합함"
    },
    {
        "model": "llama3.2:3b",
        "case_id": "UF-04",
        "manual_result": "실패",
        "memo": "임의 계산 방식 예시를 제시하고 price score 등 영어 표현이 혼입됨"
    },

    # exaone3.5:2.4b
    {
        "model": "exaone3.5:2.4b",
        "case_id": "UF-01",
        "manual_result": "통과",
        "memo": "회사 역량과 경험, 자원 정보를 자연스럽게 요구함"
    },
    {
        "model": "exaone3.5:2.4b",
        "case_id": "UF-02",
        "manual_result": "부분 통과",
        "memo": "제안서 작성에 필요한 정보를 풍부하게 안내했지만 답변이 다소 길고 회사 정보 안내가 약함"
    },
    {
        "model": "exaone3.5:2.4b",
        "case_id": "UF-03",
        "manual_result": "부분 통과",
        "memo": "요약 대상과 범위를 안내했지만 RFP 영어 설명이 포함되고 답변이 다소 장황함"
    },
    {
        "model": "exaone3.5:2.4b",
        "case_id": "UF-04",
        "manual_result": "부분 통과",
        "memo": "가격점수 계산에 추가 정보가 필요하다고 안내했지만 입찰가격, 예정가격, 평점산식을 직접 짚지 못함"
    },

    # phi3.5
    {
        "model": "phi3.5",
        "case_id": "UF-01",
        "manual_result": "실패",
        "memo": "내부 라벨이 노출되고 깨진 표현 및 외국어 혼입이 있음"
    },
    {
        "model": "phi3.5",
        "case_id": "UF-02",
        "manual_result": "실패",
        "memo": "내부 라벨이 노출되고 문장이 부자연스러움"
    },
    {
        "model": "phi3.5",
        "case_id": "UF-03",
        "manual_result": "실패",
        "memo": "내부 라벨, 외국어 혼입, 부자연스러운 문장이 포함됨"
    },
    {
        "model": "phi3.5",
        "case_id": "UF-04",
        "manual_result": "실패",
        "memo": "응답이 과도하게 반복되고 일본어/영어/깨진 표현이 혼입되어 사용 불가"
    }
]

user_facing_manual_eval_result_df = pd.DataFrame(user_facing_manual_eval_records)
user_facing_manual_eval_result_df

,model,case_id,manual_result,memo
0,gemma3:4b,UF-01,부분 통과,"자연스러운 존댓말 응답이며 내부 라벨은 없지만 수행 실적, 보유 인력 안내가 부족함"
1,gemma3:4b,UF-02,부분 통과,제안서 작성에 필요한 추가 정보를 안내했지만 회사 정보와 수행 전략 안내가 부족하고 영어 설명이 포함됨
2,gemma3:4b,UF-03,통과,대상 문서와 요약 범위가 필요하다는 점을 자연스럽게 안내함
3,gemma3:4b,UF-04,부분 통과,"임의 계산은 하지 않았지만 입찰가격, 예정가격, 평점산식이라는 핵심 입력값을 직접적으로 모두 안내하지 못함"
4,qwen2.5:3b,UF-01,부분 통과,"필요 정보 부족을 안내했지만 응답이 짧고 수행 실적, 보유 인력 안내가 부족함"
5,qwen2.5:3b,UF-02,실패,판정이라는 내부 라벨이 노출되고 회사 정보 안내가 부족함
6,qwen2.5:3b,UF-03,부분 통과,추가 정보 필요성을 안내했지만 대상 문서와 요약 범위 안내가 다소 일반적임
7,qwen2.5:3b,UF-04,실패,"입찰가격, 예정가격, 평점산식을 직접 안내하지 못하고 현재 사용 중인 가격/비교 대상 가격으로 일반화함"
8,llama3.2:3b,UF-01,실패,"requirements, details 등 영어 표현이 혼입되고 문체가 부자연스러움"
9,llama3.2:3b,UF-02,실패,응답이 장황하고 반복적이며 사용자 응답형 문장으로 부적절함


In [40]:
user_facing_eval_summary_df = user_facing_manual_eval_result_df.groupby(
    ["model", "manual_result"]
).size().reset_index(name="count")

user_facing_eval_summary_df

,model,manual_result,count
0,exaone3.5:2.4b,부분 통과,3
1,exaone3.5:2.4b,통과,1
2,gemma3:4b,부분 통과,3
3,gemma3:4b,통과,1
4,llama3.2:3b,실패,4
5,phi3.5,실패,4
6,qwen2.5:3b,부분 통과,2
7,qwen2.5:3b,실패,2


In [41]:
user_facing_score_map = {
    "통과": 1.0,
    "부분 통과": 0.5,
    "실패": 0.0
}

user_facing_manual_eval_result_df["score"] = user_facing_manual_eval_result_df["manual_result"].map(user_facing_score_map)

user_facing_score_summary_df = user_facing_manual_eval_result_df.groupby("model")["score"].agg(
    test_count="count",
    total_score="sum",
    average_score="mean"
).reset_index()

user_facing_score_summary_df = user_facing_score_summary_df.sort_values(
    "average_score",
    ascending=False
)

user_facing_score_summary_df

,model,test_count,total_score,average_score
0,exaone3.5:2.4b,4,2.5,0.625
1,gemma3:4b,4,2.5,0.625
4,qwen2.5:3b,4,1.0,0.250
2,llama3.2:3b,4,0.0,0.000
3,phi3.5,4,0.0,0.000


## 사용자 응답형 안내 수동 평가 결과 해석

사용자 응답형 정보부족 안내 테스트를 수동 평가한 결과, `gemma3:4b`와 `exaone3.5:2.4b`가 각각 평균 점수 0.625로 가장 높은 결과를 보였습니다. 두 모델 모두 4개 테스트 중 통과 1건, 부분 통과 3건으로 평가되었습니다.

`qwen2.5:3b`는 평균 점수 0.250으로, 부분 통과 2건과 실패 2건을 기록했습니다. 기본적인 정보부족 안내는 가능했지만, 내부 라벨 노출과 가격점수 계산 안내의 부정확성이 확인되었습니다.

`llama3.2:3b`와 `phi3.5`는 각각 4개 테스트 모두 실패로 평가되어 평균 점수 0.000을 기록했습니다. 두 모델 모두 사용자에게 직접 제공하기 어려운 수준의 응답 품질 문제가 확인되었습니다.

### 1. 상위 모델 비교

`gemma3:4b`는 사용자에게 보여줄 수 있는 자연스러운 한국어 존댓말 응답을 생성하는 데 강점이 있었습니다. 내부 라벨 노출이 거의 없었고, 정보가 부족한 이유를 설명한 뒤 추가 정보를 요청하는 흐름도 비교적 자연스러웠습니다. 다만 입찰 적합도나 제안서 작성 안내에서 수행 실적, 보유 인력, 회사 강점, 수행 전략과 같은 항목이 충분히 직접적으로 제시되지는 않았습니다. 또한 가격점수 계산 안내에서는 입찰가격, 예정가격, 평점산식이라는 핵심 입력값을 명확히 모두 요구하지 못했습니다.

`exaone3.5:2.4b`는 사용자 응답형 안내문 생성 능력이 비교적 양호했습니다. 특히 입찰 적합도 안내에서는 회사의 역량과 경험, 예산 및 자원 등 필요한 정보를 자연스럽게 요청했습니다. 다만 답변이 전반적으로 길고, 일부 응답에서 `RFP(Request for Proposal)`처럼 영어 설명이 포함되었습니다. 가격점수 계산 안내에서도 입찰가격, 예정가격, 평점산식이라는 핵심 정보를 직접적으로 요구하지 못하고, 프로젝트 요구사항이나 경쟁사 분석 등으로 다소 일반화되는 경향이 있었습니다.

따라서 사용자 응답형 안내 기준에서는 `gemma3:4b`와 `exaone3.5:2.4b`가 가장 우수한 후보로 볼 수 있습니다. 다만 `gemma3:4b`는 정보 요구 항목을 더 구체화해야 하고, `exaone3.5:2.4b`는 답변 길이와 영어 표현을 줄이는 보완이 필요합니다.

### 2. qwen2.5:3b 평가

`qwen2.5:3b`는 응답 속도가 빠르고 일부 케이스에서는 간결한 안내가 가능했습니다. 그러나 제안서 작성 안내에서 `판정:`이라는 내부 라벨이 노출되었고, 가격점수 계산 안내에서는 입찰가격, 예정가격, 평점산식이 아니라 현재 사용 중인 가격과 비교 대상 가격처럼 일반적인 가격 비교 문제로 흐르는 문제가 있었습니다.

즉, `qwen2.5:3b`는 기본 QA나 내부 판단 테스트에서는 속도 중심 대안 후보로 볼 수 있었지만, 사용자에게 직접 보여줄 안내 응답에서는 품질이 다소 떨어졌습니다. 따라서 사용자 응답형 안내까지 고려하면 단독 사용자 응답 모델로 쓰기보다는, 후처리나 프롬프트 보완을 전제로 한 대안 후보로 보는 것이 적절합니다.

### 3. 제외 후보 평가

`llama3.2:3b`는 사용자 응답형 안내 테스트에서 4개 케이스 모두 실패로 평가되었습니다. 응답에 `requirements`, `details`, `tool`, `price score` 같은 영어 표현이 섞였고, 일부 응답에서는 일본어와 힌디어로 보이는 문자도 포함되었습니다. 또한 가격점수 계산 안내에서 임의 계산 방식 예시를 제시하여, 임의 계산 금지 규칙에도 맞지 않았습니다.

`phi3.5` 역시 4개 케이스 모두 실패로 평가되었습니다. `판단:`, `추가 확인 필요`, `답변:`, `문장화:` 같은 내부 라벨이 그대로 노출되었고, 외국어 혼입, 반복 출력, 깨진 문장, 과도하게 긴 응답이 확인되었습니다. 특히 가격점수 계산 안내에서는 응답이 비정상적으로 반복되어 실제 사용자 응답으로 사용할 수 없는 수준이었습니다.

따라서 `llama3.2:3b`와 `phi3.5`는 사용자 응답형 안내 모델 후보에서 제외하는 것이 적절합니다.

### 4. 현재 단계 판단

사용자 응답형 안내 테스트까지 반영하면, `gemma3:4b`는 여전히 품질 중심 주력 후보로 유지할 수 있습니다. 전체 테스트에서 가장 높은 종합 점수를 보였고, 사용자 응답형 안내에서도 공동 최고 점수를 기록했습니다.

`exaone3.5:2.4b`는 사용자 응답형 안내에서는 `gemma3:4b`와 같은 평균 점수를 기록했지만, 앞선 전체 수동 평가에서는 `gemma3:4b`, `qwen2.5:3b`보다 낮았습니다. 따라서 단독 최종 후보보다는 보조 후보 또는 추가 검증 후보로 분류하는 것이 적절합니다.

`qwen2.5:3b`는 속도와 기본 QA 측면에서는 장점이 있지만, 사용자 응답형 안내에서는 내부 라벨 노출과 정보 요구의 부정확성이 확인되었습니다. 따라서 속도 중심 대안 후보로 유지하되, 실제 사용자 응답에 사용하려면 프롬프트와 후처리 보완이 필요합니다.

결론적으로 현재 단계에서는 `gemma3:4b`를 품질 중심 주력 후보, `qwen2.5:3b`를 속도 중심 대안 후보, `exaone3.5:2.4b`를 보조 후보 또는 추가 검증 후보로 정리할 수 있습니다. `llama3.2:3b`와 `phi3.5`는 사용자 응답 품질 문제로 제외하는 것이 타당합니다.

## 최종 모델 비교

앞선 실험에서는 문서 기반 QA, 정보부족 판단, 질문 재작성, 사용자 응답형 정보부족 안내를 기준으로 로컬 LLM 후보 모델을 비교하였습니다.

이 단계에서는 각 모델의 주요 강점, 한계, 활용 가능성을 종합하여 최종 후보군을 정리합니다.

In [42]:
final_model_comparison_df = pd.DataFrame([
    {
        "model": "gemma3:4b",
        "role": "품질 중심 주력 후보",
        "strength": "문서 기반 QA 안정성, 자연스러운 한국어 존댓말 응답, 정보부족 판단 방향성",
        "limitation": "질문 재작성에서 정보부족 상황을 재작성 가능으로 오판한 사례가 있음. 일부 안내 항목의 구체성 보완 필요",
        "final_decision": "주력 후보"
    },
    {
        "model": "qwen2.5:3b",
        "role": "속도 중심 대안 후보",
        "strength": "빠른 응답 속도, 문서 기반 QA 안정성, 전체 수동 평가에서 실패 케이스 없음",
        "limitation": "사용자 응답형 안내에서 내부 라벨 노출과 정보 요구 부정확성 확인. 출력 형식 제어 필요",
        "final_decision": "대안 후보"
    },
    {
        "model": "exaone3.5:2.4b",
        "role": "보조 후보 / 추가 검증 후보",
        "strength": "사용자 응답형 안내문 생성 능력, QA 및 정보부족 판단의 기본 가능성",
        "limitation": "답변이 장황하고 영어 설명이 포함됨. 질문 재작성에서 정보부족 상황 오판 사례 있음",
        "final_decision": "추가 검증 필요"
    },
    {
        "model": "llama3.2:3b",
        "role": "제외 후보",
        "strength": "일부 단순 QA에서는 금액 보존 및 항목 나열 가능",
        "limitation": "정보부족 판단 실패, 외국어 혼입, 사용자 응답형 안내 품질 문제",
        "final_decision": "제외"
    },
    {
        "model": "phi3.5",
        "role": "제외 후보",
        "strength": "금액 보존과 일부 단순 응답 가능",
        "limitation": "항목명 오류, 깨진 표현, 내부 라벨 노출, 반복 출력, 외국어 혼입",
        "final_decision": "제외"
    }
])

final_model_comparison_df

,model,role,strength,limitation,final_decision
0,gemma3:4b,품질 중심 주력 후보,"문서 기반 QA 안정성, 자연스러운 한국어 존댓말 응답, 정보부족 판단 방향성",질문 재작성에서 정보부족 상황을 재작성 가능으로 오판한 사례가 있음. 일부 안내 항목의 구체성 보완 필요,주력 후보
1,qwen2.5:3b,속도 중심 대안 후보,"빠른 응답 속도, 문서 기반 QA 안정성, 전체 수동 평가에서 실패 케이스 없음",사용자 응답형 안내에서 내부 라벨 노출과 정보 요구 부정확성 확인. 출력 형식 제어 필요,대안 후보
2,exaone3.5:2.4b,보조 후보 / 추가 검증 후보,"사용자 응답형 안내문 생성 능력, QA 및 정보부족 판단의 기본 가능성",답변이 장황하고 영어 설명이 포함됨. 질문 재작성에서 정보부족 상황 오판 사례 있음,추가 검증 필요
3,llama3.2:3b,제외 후보,일부 단순 QA에서는 금액 보존 및 항목 나열 가능,"정보부족 판단 실패, 외국어 혼입, 사용자 응답형 안내 품질 문제",제외
4,phi3.5,제외 후보,금액 보존과 일부 단순 응답 가능,"항목명 오류, 깨진 표현, 내부 라벨 노출, 반복 출력, 외국어 혼입",제외


In [43]:
base_score_df = model_score_summary_df.rename(
    columns={
        "average_score": "base_average_score",
        "total_score": "base_total_score",
        "test_count": "base_test_count"
    }
)

user_facing_score_df = user_facing_score_summary_df.rename(
    columns={
        "average_score": "user_facing_average_score",
        "total_score": "user_facing_total_score",
        "test_count": "user_facing_test_count"
    }
)

combined_score_df = pd.merge(
    base_score_df,
    user_facing_score_df,
    on="model",
    how="left"
)

combined_score_df["combined_average_score"] = (
    combined_score_df["base_average_score"] + combined_score_df["user_facing_average_score"]
) / 2

combined_score_df = combined_score_df.sort_values(
    "combined_average_score",
    ascending=False
)

combined_score_df

,model,base_test_count,base_total_score,base_average_score,user_facing_test_count,user_facing_total_score,user_facing_average_score,combined_average_score
0,gemma3:4b,11,8.0,0.727273,4,2.5,0.625,0.676136
2,exaone3.5:2.4b,11,7.0,0.636364,4,2.5,0.625,0.630682
1,qwen2.5:3b,11,7.5,0.681818,4,1.0,0.250,0.465909
3,llama3.2:3b,11,3.0,0.272727,4,0.0,0.000,0.136364
4,phi3.5,11,3.0,0.272727,4,0.0,0.000,0.136364


## 최종 모델 비교 해석

기본 테스트와 사용자 응답형 안내 테스트를 결합하여 종합 점수를 계산한 결과, `gemma3:4b`가 가장 높은 종합 평균 점수 0.676을 기록하였습니다. 그다음은 `exaone3.5:2.4b`가 0.631, `qwen2.5:3b`가 0.466으로 뒤를 이었습니다. `llama3.2:3b`와 `phi3.5`는 각각 0.136으로 낮은 점수를 기록했습니다.

### 1. gemma3:4b

`gemma3:4b`는 기본 테스트 평균 점수 0.727, 사용자 응답형 안내 평균 점수 0.625를 기록하여 전체 모델 중 가장 높은 종합 점수를 보였습니다.

문서 기반 QA에서는 금액 보존, 항목 나열, 임의 계산 방지 측면에서 안정적인 결과를 보였고, 정보부족 판단에서도 대체로 추가 확인이 필요한 상황을 올바르게 판단했습니다. 사용자 응답형 안내에서도 내부 라벨 노출이 거의 없고, 한국어 존댓말 문체가 자연스러웠습니다.

다만 질문 재작성 테스트에서 비교 대상이 부족한 상황을 재작성 가능으로 오판한 사례가 있었고, 사용자 안내 응답에서도 수행 실적, 보유 인력, 입찰가격, 예정가격, 평점산식 같은 핵심 요구 정보를 더 직접적으로 제시할 필요가 있었습니다.

따라서 `gemma3:4b`는 현재 단계에서 품질 중심 주력 후보로 정리할 수 있습니다.

### 2. exaone3.5:2.4b

`exaone3.5:2.4b`는 기본 테스트 평균 점수 0.636, 사용자 응답형 안내 평균 점수 0.625를 기록하여 종합 평균 점수 0.631로 두 번째로 높은 결과를 보였습니다.

특히 사용자 응답형 안내에서는 `gemma3:4b`와 같은 평균 점수를 기록하여, 실제 사용자에게 보여줄 수 있는 안내문 생성 능력은 비교적 양호한 편으로 확인되었습니다. 입찰 적합도 안내에서는 회사의 역량과 경험, 자원 정보 등을 자연스럽게 요구했고, 제안서 작성이나 요약 요청에서도 비교적 풍부한 안내를 생성했습니다.

하지만 기본 테스트에서는 `gemma3:4b`, `qwen2.5:3b`보다 낮은 점수를 보였고, 질문 재작성에서 정보가 부족한 상황을 재작성 가능으로 오판한 사례가 있었습니다. 또한 답변이 장황해지는 경향이 있고, `RFP(Request for Proposal)`과 같은 영어 설명이 포함되는 문제가 있었습니다.

따라서 `exaone3.5:2.4b`는 단독 최종 후보보다는 보조 후보 또는 추가 검증 후보로 분류하는 것이 적절합니다.

### 3. qwen2.5:3b

`qwen2.5:3b`는 기본 테스트에서는 평균 점수 0.682로 비교적 높은 결과를 보였지만, 사용자 응답형 안내 테스트에서는 평균 점수 0.250으로 크게 낮아졌습니다. 최종 종합 평균 점수는 0.466입니다.

문서 기반 QA에서는 금액 보존, 항목 나열, 임의 계산 방지 측면에서 안정적인 결과를 보였고, 기본 테스트 수동 평가에서도 실패 케이스는 없었습니다. 또한 전체적으로 응답 속도가 빠른 편이라는 장점이 있습니다.

그러나 사용자 응답형 안내에서는 `판정:`과 같은 내부 라벨이 노출되었고, 가격점수 계산 안내에서 입찰가격, 예정가격, 평점산식이 아니라 일반적인 가격 비교 문제로 흐르는 문제가 있었습니다. 즉, 내부 판단이나 문서 기반 QA에는 어느 정도 활용 가능성이 있지만, 사용자에게 직접 보여줄 응답 생성에는 보완이 필요합니다.

따라서 `qwen2.5:3b`는 속도 중심 대안 후보로 유지하되, 실제 사용자 응답에 사용하려면 프롬프트 제어와 후처리 보완이 필요합니다.

### 4. llama3.2:3b

`llama3.2:3b`는 기본 테스트 평균 점수 0.273, 사용자 응답형 안내 평균 점수 0.000을 기록하여 종합 평균 점수 0.136에 그쳤습니다.

단순 QA 일부에서는 금액 보존과 항목 나열이 가능했지만, 정보부족 판단 테스트에서 대부분의 케이스를 답변 가능으로 오판했습니다. 또한 사용자 응답형 안내에서는 영어, 일본어, 힌디어로 보이는 문자 혼입과 장황한 설명, 임의 계산 방식 제시 문제가 확인되었습니다.

따라서 `llama3.2:3b`는 현재 결과 기준으로 주요 후보에서 제외하는 것이 적절합니다.

### 5. phi3.5

`phi3.5` 역시 기본 테스트 평균 점수 0.273, 사용자 응답형 안내 평균 점수 0.000을 기록하여 종합 평균 점수 0.136에 머물렀습니다.

금액 보존과 일부 단순 응답은 가능했지만, 항목명 오류, 깨진 표현, 내부 라벨 노출, 외국어 혼입, 반복 출력 문제가 확인되었습니다. 특히 사용자 응답형 안내에서는 모든 케이스가 실패로 평가되었고, 가격점수 안내에서는 비정상적으로 긴 반복 응답이 생성되었습니다.

따라서 `phi3.5`는 현재 결과 기준으로 최종 후보에서 제외하는 것이 적절합니다.

### 6. 최종 후보 정리

현재 종합 평가 결과를 기준으로 하면 다음과 같이 정리할 수 있습니다.

| 모델 | 최종 역할 | 판단 |
|---|---|---|
| `gemma3:4b` | 품질 중심 주력 후보 | 전체 종합 점수가 가장 높고, QA·정보부족 판단·사용자 응답형 안내가 비교적 안정적임 |
| `qwen2.5:3b` | 속도 중심 대안 후보 | 기본 테스트와 속도 측면에서는 장점이 있으나 사용자 응답형 안내 보완 필요 |
| `exaone3.5:2.4b` | 보조 후보 / 추가 검증 후보 | 사용자 응답형 안내는 양호하지만 재작성 판단과 답변 길이, 영어 표현 보완 필요 |
| `llama3.2:3b` | 제외 후보 | 정보부족 판단 오류와 외국어 혼입 문제가 큼 |
| `phi3.5` | 제외 후보 | 깨진 표현, 내부 라벨 노출, 반복 출력, 외국어 혼입 문제가 큼 |

따라서 현재 단계에서는 `gemma3:4b`를 주력 후보로 설정하고, `qwen2.5:3b`를 속도 중심 대안 후보로 유지하는 것이 적절합니다. `exaone3.5:2.4b`는 사용자 응답형 안내에서는 가능성이 있으므로 보조 후보로 두고 추가 검증할 수 있습니다.

다만 이 종합 점수는 기본 테스트 평균과 사용자 응답형 안내 평균을 단순 평균한 값입니다. 실제 RAG 시스템에 연결한 이후에는 검색 context 품질, chunk 구성, 문서명 및 페이지 정보 포함 여부에 따라 모델 응답이 달라질 수 있으므로, 최종 모델 선택 전에는 실제 검색 결과를 연결한 상태에서 재평가가 필요합니다.

## 최종 후보 압축

기본 QA, 정보부족 판단, 질문 재작성, 사용자 응답형 안내 테스트를 종합한 결과, `llama3.2:3b`와 `phi3.5`는 최종 후보군에서 제외하였습니다.

`llama3.2:3b`는 정보부족 판단에서 추가 확인이 필요한 상황을 답변 가능으로 오판하는 사례가 많았고, 사용자 응답형 안내에서 영어, 일본어, 힌디어로 보이는 문자 혼입 문제가 반복적으로 확인되었습니다.

`phi3.5`는 항목명 오류, 깨진 표현, 내부 라벨 노출, 반복 출력, 외국어 혼입 문제가 확인되어 사용자에게 직접 제공할 응답 생성 모델로 사용하기 어렵다고 판단하였습니다.

따라서 이후 최종 후보 검토는 다음 3개 모델을 중심으로 진행합니다.

- `gemma3:4b`
- `qwen2.5:3b`
- `exaone3.5:2.4b`

In [44]:
final_candidate_models = [
    "gemma3:4b",
    "qwen2.5:3b",
    "exaone3.5:2.4b"
]

final_candidate_models

['gemma3:4b', 'qwen2.5:3b', 'exaone3.5:2.4b']

## 후처리 함수 설계

모델 응답을 그대로 사용자에게 제공할 경우, 일부 응답에서 내부 라벨, 외국어 혼입, 깨진 표현, 과도하게 짧은 응답이 노출될 수 있습니다.

따라서 사용자에게 최종 응답을 보여주기 전, 최소한의 후처리 함수를 적용합니다.

후처리 함수의 목적은 다음과 같습니다.

- 빈 응답 또는 지나치게 짧은 응답을 기본 안내 문구로 대체
- `판단:`, `추가 확인 필요`, `답변 가능` 같은 내부 라벨 제거
- 한국어 응답에 부적절한 외국어 문자 혼입 감지
- 깨진 응답이 사용자에게 그대로 노출되는 것을 방지
- 의미를 훼손할 수 있는 과도한 문장 자르기는 수행하지 않음

특히 번호 목록이나 긴 안내 응답은 문장 단위로 잘라버릴 경우 의미가 깨질 수 있으므로, 본 후처리 함수에서는 응답 길이를 임의로 자르지 않습니다.

In [45]:
def postprocess_user_response(answer):
    """
    LLM 원본 응답을 사용자에게 보여주기 전에 검수하는 함수입니다.
    
    처리 내용:
    - 빈 응답 또는 지나치게 짧은 응답 대체
    - 내부 라벨 제거
    - 외국어 문자 혼입 감지
    - 의미가 깨질 수 있는 문장 단위 자르기는 수행하지 않음
    """
    
    fallback_answer = (
        "현재 질문만으로는 정확한 답변을 드리기 어렵습니다. "
        "관련 RFP 문서나 추가 정보를 제공해주시면 확인 후 안내해 드리겠습니다."
    )
    
    if answer is None:
        return fallback_answer
    
    processed = str(answer).strip()
    
    if len(processed) < 10:
        return fallback_answer
    
    # 한자, 일본어, 키릴 문자 등 한국어 응답에 부적절한 외국어 문자 감지
    foreign_pattern = r"[\u4e00-\u9fff\u3040-\u30ff\u0400-\u04FF]"
    
    if re.search(foreign_pattern, processed):
        return fallback_answer
    
    banned_labels = [
        "판단:",
        "판정:",
        "추가 확인 필요:",
        "추가 확인 필요",
        "답변 가능",
        "재작성 질문:",
        "추가로 확인할 정보:",
        "출력:",
        "답변:",
        "문장화:",
        "[최종 답변]",
        "[응답]"
    ]
    
    for label in banned_labels:
        processed = processed.replace(label, "").strip()
    
    if len(processed) < 10:
        return fallback_answer
    
    return processed

In [46]:
postprocess_test_samples = [
    "판단: 추가 확인 필요\n추가로 확인할 정보:\n- 입찰가격\n- 예정가격\n- 평점산식",
    "현재 질문만으로는 가격점수를 계산하기 어렵습니다. 입찰가격, 예정가격, 평점산식을 제공해 주세요.",
    "현재 문서에는 가격점수를 계산할 수 없습니다. 因此 추가 정보가 필요합니다.",
    "",
    "짧음"
]

for i, sample in enumerate(postprocess_test_samples, start=1):
    print("=" * 80)
    print(f"샘플 {i}")
    print("[원본]")
    print(sample)
    print("\n[후처리 결과]")
    print(postprocess_user_response(sample))
    print()

샘플 1
[원본]
판단: 추가 확인 필요
추가로 확인할 정보:
- 입찰가격
- 예정가격
- 평점산식

[후처리 결과]
- 입찰가격
- 예정가격
- 평점산식

샘플 2
[원본]
현재 질문만으로는 가격점수를 계산하기 어렵습니다. 입찰가격, 예정가격, 평점산식을 제공해 주세요.

[후처리 결과]
현재 질문만으로는 가격점수를 계산하기 어렵습니다. 입찰가격, 예정가격, 평점산식을 제공해 주세요.

샘플 3
[원본]
현재 문서에는 가격점수를 계산할 수 없습니다. 因此 추가 정보가 필요합니다.

[후처리 결과]
현재 질문만으로는 정확한 답변을 드리기 어렵습니다. 관련 RFP 문서나 추가 정보를 제공해주시면 확인 후 안내해 드리겠습니다.

샘플 4
[원본]


[후처리 결과]
현재 질문만으로는 정확한 답변을 드리기 어렵습니다. 관련 RFP 문서나 추가 정보를 제공해주시면 확인 후 안내해 드리겠습니다.

샘플 5
[원본]
짧음

[후처리 결과]
현재 질문만으로는 정확한 답변을 드리기 어렵습니다. 관련 RFP 문서나 추가 정보를 제공해주시면 확인 후 안내해 드리겠습니다.



In [47]:
user_facing_postprocessed_df = user_facing_results_df.copy()

user_facing_postprocessed_df["postprocessed_answer"] = user_facing_postprocessed_df["answer"].apply(
    postprocess_user_response
)

user_facing_postprocessed_df["changed"] = (
    user_facing_postprocessed_df["answer"] != user_facing_postprocessed_df["postprocessed_answer"]
)

user_facing_postprocessed_df[
    [
        "model",
        "case_id",
        "category",
        "changed",
        "answer",
        "postprocessed_answer"
    ]
]

,model,case_id,category,changed,answer,postprocessed_answer
0,gemma3:4b,UF-01,입찰 적합도 안내,False,"죄송합니다. 현재 귀사의 정보가 없어, 해당 입찰에 적합한지 판단하기 어렵습니다. \n\n귀사의 사업 분야, 주요 제품/서비스, 그리고 이번 입찰과 관련된 구체적인 요구사항을 파악해야 정확한 분석이 가능합니다. 예를 들어, 입찰 공고서의 상세 내용을 알려주시거나, 귀사의 관련 자료를 공유해주시면 더욱 자세하게 검토할 수 있습니다. \n\n좀 더 정확한 답변을 위해, 귀사의 사업 분야와 이번 입찰과 관련된 주요 내용에 대한 정보를 추가로 제공해 주실 수 있을까요? 예를 들어 “저희 회사는 IT 솔루션 개발 회사이며, 이번 입찰은 정부기관의 클라우드 서비스 구축 프로젝트입니다.” 와 같이 말씀해주시면 감사하겠습니다.","죄송합니다. 현재 귀사의 정보가 없어, 해당 입찰에 적합한지 판단하기 어렵습니다. \n\n귀사의 사업 분야, 주요 제품/서비스, 그리고 이번 입찰과 관련된 구체적인 요구사항을 파악해야 정확한 분석이 가능합니다. 예를 들어, 입찰 공고서의 상세 내용을 알려주시거나, 귀사의 관련 자료를 공유해주시면 더욱 자세하게 검토할 수 있습니다. \n\n좀 더 정확한 답변을 위해, 귀사의 사업 분야와 이번 입찰과 관련된 주요 내용에 대한 정보를 추가로 제공해 주실 수 있을까요? 예를 들어 “저희 회사는 IT 솔루션 개발 회사이며, 이번 입찰은 정부기관의 클라우드 서비스 구축 프로젝트입니다.” 와 같이 말씀해주시면 감사하겠습니다."
1,gemma3:4b,UF-02,제안서 작성 안내,False,"죄송합니다. 제안서 작성에 필요한 내용을 정리해 드리는 데는 몇 가지 추가 정보가 필요합니다. 어떤 종류의 RFP(Request for Proposal, 제안 요청서) 분석을 원하시는지 알려주시면 더욱 정확한 도움을 드릴 수 있습니다.\n\n예를 들어, RFP의 주제, 목표, 주요 평가 기준 등을 구체적으로 말씀해주실 수 있을까요? 또한, 현재까지 준비하신 자료나 관련 문서가 있다면 공유해 주시면 제안서 작성에 도움이 될 만한 정보를 파악하는 데 큰 도움이 될 것입니다. \n\n다음 질문 예시: “이번 RFP는 ‘신규 스마트폰 개발’과 관련된 것으로, 주요 평가 기준은 기술 혁신성, 시장 경쟁력, 가격 등이 있습니다.”","죄송합니다. 제안서 작성에 필요한 내용을 정리해 드리는 데는 몇 가지 추가 정보가 필요합니다. 어떤 종류의 RFP(Request for Proposal, 제안 요청서) 분석을 원하시는지 알려주시면 더욱 정확한 도움을 드릴 수 있습니다.\n\n예를 들어, RFP의 주제, 목표, 주요 평가 기준 등을 구체적으로 말씀해주실 수 있을까요? 또한, 현재까지 준비하신 자료나 관련 문서가 있다면 공유해 주시면 제안서 작성에 도움이 될 만한 정보를 파악하는 데 큰 도움이 될 것입니다. \n\n다음 질문 예시: “이번 RFP는 ‘신규 스마트폰 개발’과 관련된 것으로, 주요 평가 기준은 기술 혁신성, 시장 경쟁력, 가격 등이 있습니다.”"
2,gemma3:4b,UF-03,모호한 요약 요청 안내,False,"죄송합니다. 현재 요청하신 내용의 핵심을 정확하게 파악하기에는 정보가 부족합니다. 어떤 RFP(제안요청서)에 대한 내용을 요약해 드릴지, 그리고 어떤 부분을 중심으로 핵심을 뽑아드릴지 구체적으로 알려주시면 더욱 정확한 답변을 제공해 드릴 수 있습니다. \n\n예를 들어, “OOO RFP의 기술적 요구사항과 평가 기준만 핵심으로 요약해주세요.” 와 같이 질문해주시면 제가 더 적절하게 도움을 드릴 수 있습니다. 다음 질문 예시: “요청하신 RFP의 주요 내용 중 가장 중요한 부분을 알려주세요.”","죄송합니다. 현재 요청하신 내용의 핵심을 정확하게 파악하기에는 정보가 부족합니다. 어떤 RFP(제안요청서)에 대한 내용을 요약해 드릴지, 그리고 어떤 부분을 중심으로 핵심을 뽑아드릴지 구체적으로 알려주시면 더욱 정확한 답변을 제공해 드릴 수 있습니다. \n\n예를 들어, “OOO RFP의 기술적 요구사항과 평가 기준만 핵심으로 요약해주세요.” 와 같이 질문해주시면 제가 더 적절하게 도움을 드릴 수 있습니다. 다음 질문 예시: “요청하신 RFP의 주요 내용 중 가장 중요한 부분을 알려주세요.”"
3,gemma3:4b,UF-04,가격점수 계산 안내,False,"죄송합니다. 가격 점수를 계산하려면 몇 가지 추가 정보가 필요합니다. 현재 제공해주신 내용만으로는 어떤 기준으로 가격을 평가하고, 어떤 데이터를 사용하여 점수를 매기기로 결정했는지 알 수 없어서 정확한 계산이 어렵습니다. \n\n좀 더 자세히 알려주시면 감사하겠습니다. 예를 들어, 각 제안사의 가격과 함께, 가격 평가에 사용될 주요 항목(예: 기술 수준, 서비스 품질, 납기일 등) 및 각 항목별 가중치를 구체적으로 말씀해주시거나, 어떤 방식으로 점수를 계산해야 하는지 설명해주시면 됩니다. \n\n다음 질문 예시는 “각 항목별 가중치와 점수 산정 기준을 자세히 알려주세요.” 입니다.","죄송합니다. 가격 점수를 계산하려면 몇 가지 추가 정보가 필요합니다. 현재 제공해주신 내용만으로는 어떤 기준으로 가격을 평가하고, 어떤 데이터를 사용하여 점수를 매기기로 결정했는지 알 수 없어서 정확한 계산이 어렵습니다. \n\n좀 더 자세히 알려주시면 감사하겠습니다. 예를 들어, 각 제안사의 가격과 함께, 가격 평가에 사용될 주요 항목(예: 기술 수준, 서비스 품질, 납기일 등) 및 각 항목별 가중치를 구체적으로 말씀해주시거나, 어떤 방식으로 점수를 계산해야 하는지 설명해주시면 됩니다. \n\n다음 질문 예시는 “각 항목별 가중치와 점수 산정 기준을 자세히 알려주세요.” 입니다."
4,qwen2.5:3b,UF-01,입찰 적합도 안내,False,"판정을 내리는 데 필요한 정보가 부족합니다. 이 입찰에서 요구하는 기술적 능력, 경험이나 전문성을 포함해 우리 회사가 어떤 면에서 적합한지에 대해 자세히 설명해 주시면 감사하겠습니다.","판정을 내리는 데 필요한 정보가 부족합니다. 이 입찰에서 요구하는 기술적 능력, 경험이나 전문성을 포함해 우리 회사가 어떤 면에서 적합한지에 대해 자세히 설명해 주시면 감사하겠습니다."
5,qwen2.5:3b,UF-02,제안서 작성 안내,True,"판정: 제안서를 정리하는 데 필요한 정보가 아직 부족합니다. 사용자가 어떤 주제나 분야의 제안서를 정리하고 싶으신지, 또는 어떤 요소들을 포함해야 하는지 알려주시면 도와드릴 수 있습니다. 예를 들어, 기술적인 특성, 비용, 시간대, 성과 등이 있을 수 있습니다.","제안서를 정리하는 데 필요한 정보가 아직 부족합니다. 사용자가 어떤 주제나 분야의 제안서를 정리하고 싶으신지, 또는 어떤 요소들을 포함해야 하는지 알려주시면 도와드릴 수 있습니다. 예를 들어, 기술적인 특성, 비용, 시간대, 성과 등이 있을 수 있습니다."
6,qwen2.5:3b,UF-03,모호한 요약 요청 안내,False,"죄송합니다, 사용자의 질문을 정확히 이해하고 필요한 정보를 추가로 수집하기 위해 조금 더 구체적인 정보가 필요합니다. 핵심 내용을 뽑아드릴 수 있도록 도와주시면 감사하겠습니다. 예를 들어, RFP 문서의 주요 챕터나 특정 조건에 대해 질문해 주시면 좋겠습니다.","죄송합니다, 사용자의 질문을 정확히 이해하고 필요한 정보를 추가로 수집하기 위해 조금 더 구체적인 정보가 필요합니다. 핵심 내용을 뽑아드릴 수 있도록 도와주시면 감사하겠습니다. 예를 들어, RFP 문서의 주요 챕터나 특정 조건에 대해 질문해 주시면 좋겠습니다."
7,qwen2.5:3b,UF-04,가격점수 계산 안내,False,"가격점수를 계산하기 위해서는, 현재 사용 중인 가격과 비교 대상의 가격이 필요합니다. 또한, 서비스의 품질이나 성능 등 다른 요소도 고려해야 합니다. 다음에 제공해주시면 감사하겠습니다. 예를 들어, ""현재 사용 중인 서비스의 가격과 비교 대상의 가격을 알려주실 수 있으신가요?""라고 질문드릴게요.","가격점수를 계산하기 위해서는, 현재 사용 중인 가격과 비교 대상의 가격이 필요합니다. 또한, 서비스의 품질이

In [48]:
candidate_postprocessed_df = user_facing_postprocessed_df[
    user_facing_postprocessed_df["model"].isin(final_candidate_models)
].copy()

candidate_postprocessed_df[
    [
        "model",
        "case_id",
        "category",
        "changed",
        "answer",
        "postprocessed_answer"
    ]
]

,model,case_id,category,changed,answer,postprocessed_answer
0,gemma3:4b,UF-01,입찰 적합도 안내,False,"죄송합니다. 현재 귀사의 정보가 없어, 해당 입찰에 적합한지 판단하기 어렵습니다. \n\n귀사의 사업 분야, 주요 제품/서비스, 그리고 이번 입찰과 관련된 구체적인 요구사항을 파악해야 정확한 분석이 가능합니다. 예를 들어, 입찰 공고서의 상세 내용을 알려주시거나, 귀사의 관련 자료를 공유해주시면 더욱 자세하게 검토할 수 있습니다. \n\n좀 더 정확한 답변을 위해, 귀사의 사업 분야와 이번 입찰과 관련된 주요 내용에 대한 정보를 추가로 제공해 주실 수 있을까요? 예를 들어 “저희 회사는 IT 솔루션 개발 회사이며, 이번 입찰은 정부기관의 클라우드 서비스 구축 프로젝트입니다.” 와 같이 말씀해주시면 감사하겠습니다.","죄송합니다. 현재 귀사의 정보가 없어, 해당 입찰에 적합한지 판단하기 어렵습니다. \n\n귀사의 사업 분야, 주요 제품/서비스, 그리고 이번 입찰과 관련된 구체적인 요구사항을 파악해야 정확한 분석이 가능합니다. 예를 들어, 입찰 공고서의 상세 내용을 알려주시거나, 귀사의 관련 자료를 공유해주시면 더욱 자세하게 검토할 수 있습니다. \n\n좀 더 정확한 답변을 위해, 귀사의 사업 분야와 이번 입찰과 관련된 주요 내용에 대한 정보를 추가로 제공해 주실 수 있을까요? 예를 들어 “저희 회사는 IT 솔루션 개발 회사이며, 이번 입찰은 정부기관의 클라우드 서비스 구축 프로젝트입니다.” 와 같이 말씀해주시면 감사하겠습니다."
1,gemma3:4b,UF-02,제안서 작성 안내,False,"죄송합니다. 제안서 작성에 필요한 내용을 정리해 드리는 데는 몇 가지 추가 정보가 필요합니다. 어떤 종류의 RFP(Request for Proposal, 제안 요청서) 분석을 원하시는지 알려주시면 더욱 정확한 도움을 드릴 수 있습니다.\n\n예를 들어, RFP의 주제, 목표, 주요 평가 기준 등을 구체적으로 말씀해주실 수 있을까요? 또한, 현재까지 준비하신 자료나 관련 문서가 있다면 공유해 주시면 제안서 작성에 도움이 될 만한 정보를 파악하는 데 큰 도움이 될 것입니다. \n\n다음 질문 예시: “이번 RFP는 ‘신규 스마트폰 개발’과 관련된 것으로, 주요 평가 기준은 기술 혁신성, 시장 경쟁력, 가격 등이 있습니다.”","죄송합니다. 제안서 작성에 필요한 내용을 정리해 드리는 데는 몇 가지 추가 정보가 필요합니다. 어떤 종류의 RFP(Request for Proposal, 제안 요청서) 분석을 원하시는지 알려주시면 더욱 정확한 도움을 드릴 수 있습니다.\n\n예를 들어, RFP의 주제, 목표, 주요 평가 기준 등을 구체적으로 말씀해주실 수 있을까요? 또한, 현재까지 준비하신 자료나 관련 문서가 있다면 공유해 주시면 제안서 작성에 도움이 될 만한 정보를 파악하는 데 큰 도움이 될 것입니다. \n\n다음 질문 예시: “이번 RFP는 ‘신규 스마트폰 개발’과 관련된 것으로, 주요 평가 기준은 기술 혁신성, 시장 경쟁력, 가격 등이 있습니다.”"
2,gemma3:4b,UF-03,모호한 요약 요청 안내,False,"죄송합니다. 현재 요청하신 내용의 핵심을 정확하게 파악하기에는 정보가 부족합니다. 어떤 RFP(제안요청서)에 대한 내용을 요약해 드릴지, 그리고 어떤 부분을 중심으로 핵심을 뽑아드릴지 구체적으로 알려주시면 더욱 정확한 답변을 제공해 드릴 수 있습니다. \n\n예를 들어, “OOO RFP의 기술적 요구사항과 평가 기준만 핵심으로 요약해주세요.” 와 같이 질문해주시면 제가 더 적절하게 도움을 드릴 수 있습니다. 다음 질문 예시: “요청하신 RFP의 주요 내용 중 가장 중요한 부분을 알려주세요.”","죄송합니다. 현재 요청하신 내용의 핵심을 정확하게 파악하기에는 정보가 부족합니다. 어떤 RFP(제안요청서)에 대한 내용을 요약해 드릴지, 그리고 어떤 부분을 중심으로 핵심을 뽑아드릴지 구체적으로 알려주시면 더욱 정확한 답변을 제공해 드릴 수 있습니다. \n\n예를 들어, “OOO RFP의 기술적 요구사항과 평가 기준만 핵심으로 요약해주세요.” 와 같이 질문해주시면 제가 더 적절하게 도움을 드릴 수 있습니다. 다음 질문 예시: “요청하신 RFP의 주요 내용 중 가장 중요한 부분을 알려주세요.”"
3,gemma3:4b,UF-04,가격점수 계산 안내,False,"죄송합니다. 가격 점수를 계산하려면 몇 가지 추가 정보가 필요합니다. 현재 제공해주신 내용만으로는 어떤 기준으로 가격을 평가하고, 어떤 데이터를 사용하여 점수를 매기기로 결정했는지 알 수 없어서 정확한 계산이 어렵습니다. \n\n좀 더 자세히 알려주시면 감사하겠습니다. 예를 들어, 각 제안사의 가격과 함께, 가격 평가에 사용될 주요 항목(예: 기술 수준, 서비스 품질, 납기일 등) 및 각 항목별 가중치를 구체적으로 말씀해주시거나, 어떤 방식으로 점수를 계산해야 하는지 설명해주시면 됩니다. \n\n다음 질문 예시는 “각 항목별 가중치와 점수 산정 기준을 자세히 알려주세요.” 입니다.","죄송합니다. 가격 점수를 계산하려면 몇 가지 추가 정보가 필요합니다. 현재 제공해주신 내용만으로는 어떤 기준으로 가격을 평가하고, 어떤 데이터를 사용하여 점수를 매기기로 결정했는지 알 수 없어서 정확한 계산이 어렵습니다. \n\n좀 더 자세히 알려주시면 감사하겠습니다. 예를 들어, 각 제안사의 가격과 함께, 가격 평가에 사용될 주요 항목(예: 기술 수준, 서비스 품질, 납기일 등) 및 각 항목별 가중치를 구체적으로 말씀해주시거나, 어떤 방식으로 점수를 계산해야 하는지 설명해주시면 됩니다. \n\n다음 질문 예시는 “각 항목별 가중치와 점수 산정 기준을 자세히 알려주세요.” 입니다."
4,qwen2.5:3b,UF-01,입찰 적합도 안내,False,"판정을 내리는 데 필요한 정보가 부족합니다. 이 입찰에서 요구하는 기술적 능력, 경험이나 전문성을 포함해 우리 회사가 어떤 면에서 적합한지에 대해 자세히 설명해 주시면 감사하겠습니다.","판정을 내리는 데 필요한 정보가 부족합니다. 이 입찰에서 요구하는 기술적 능력, 경험이나 전문성을 포함해 우리 회사가 어떤 면에서 적합한지에 대해 자세히 설명해 주시면 감사하겠습니다."
5,qwen2.5:3b,UF-02,제안서 작성 안내,True,"판정: 제안서를 정리하는 데 필요한 정보가 아직 부족합니다. 사용자가 어떤 주제나 분야의 제안서를 정리하고 싶으신지, 또는 어떤 요소들을 포함해야 하는지 알려주시면 도와드릴 수 있습니다. 예를 들어, 기술적인 특성, 비용, 시간대, 성과 등이 있을 수 있습니다.","제안서를 정리하는 데 필요한 정보가 아직 부족합니다. 사용자가 어떤 주제나 분야의 제안서를 정리하고 싶으신지, 또는 어떤 요소들을 포함해야 하는지 알려주시면 도와드릴 수 있습니다. 예를 들어, 기술적인 특성, 비용, 시간대, 성과 등이 있을 수 있습니다."
6,qwen2.5:3b,UF-03,모호한 요약 요청 안내,False,"죄송합니다, 사용자의 질문을 정확히 이해하고 필요한 정보를 추가로 수집하기 위해 조금 더 구체적인 정보가 필요합니다. 핵심 내용을 뽑아드릴 수 있도록 도와주시면 감사하겠습니다. 예를 들어, RFP 문서의 주요 챕터나 특정 조건에 대해 질문해 주시면 좋겠습니다.","죄송합니다, 사용자의 질문을 정확히 이해하고 필요한 정보를 추가로 수집하기 위해 조금 더 구체적인 정보가 필요합니다. 핵심 내용을 뽑아드릴 수 있도록 도와주시면 감사하겠습니다. 예를 들어, RFP 문서의 주요 챕터나 특정 조건에 대해 질문해 주시면 좋겠습니다."
7,qwen2.5:3b,UF-04,가격점수 계산 안내,False,"가격점수를 계산하기 위해서는, 현재 사용 중인 가격과 비교 대상의 가격이 필요합니다. 또한, 서비스의 품질이나 성능 등 다른 요소도 고려해야 합니다. 다음에 제공해주시면 감사하겠습니다. 예를 들어, ""현재 사용 중인 서비스의 가격과 비교 대상의 가격을 알려주실 수 있으신가요?""라고 질문드릴게요.","가격점수를 계산하기 위해서는, 현재 사용 중인 가격과 비교 대상의 가격이 필요합니다. 또한, 서비스의 품질이

In [49]:
postprocess_change_summary_df = user_facing_postprocessed_df.groupby(
    ["model", "changed"]
).size().reset_index(name="count")

postprocess_change_summary_df

,model,changed,count
0,exaone3.5:2.4b,False,4
1,gemma3:4b,False,4
2,llama3.2:3b,False,2
3,llama3.2:3b,True,2
4,phi3.5,True,4
5,qwen2.5:3b,False,3
6,qwen2.5:3b,True,1


In [50]:
candidate_postprocess_change_summary_df = candidate_postprocessed_df.groupby(
    ["model", "changed"]
).size().reset_index(name="count")

candidate_postprocess_change_summary_df

,model,changed,count
0,exaone3.5:2.4b,False,4
1,gemma3:4b,False,4
2,qwen2.5:3b,False,3
3,qwen2.5:3b,True,1


## 후처리 적용 결과 해석

후처리 함수를 사용자 응답형 안내 결과에 적용한 결과, 전체 모델 기준으로는 일부 응답이 변경되었지만, 최종 후보군 3개 모델에서는 변경이 많지 않았습니다.

전체 모델 기준으로 보면 `gemma3:4b`와 `exaone3.5:2.4b`는 각각 4개 응답 모두 후처리 전후 결과가 동일했습니다. 이는 두 모델의 사용자 응답형 안내 결과에서 외국어 문자 혼입, 내부 라벨 노출, 빈 응답 등 후처리 함수가 감지하는 문제가 발생하지 않았다는 의미입니다.

`qwen2.5:3b`는 4개 응답 중 1개가 후처리로 변경되었습니다. 제안서 작성 안내 케이스에서 원본 응답에 포함된 `판정:` 라벨이 제거되었고, 나머지 문장은 유지되었습니다. 이는 후처리 함수가 내부 라벨 제거에는 일부 효과가 있음을 보여줍니다.

반면 `llama3.2:3b`와 `phi3.5`는 제외 후보로 분류된 모델답게 후처리 변경이 많이 발생했습니다. `llama3.2:3b`는 4개 중 2개 응답이 변경되었고, `phi3.5`는 4개 응답 모두 변경되었습니다. 특히 외국어 문자 혼입이나 비정상적인 응답이 감지된 경우 기본 안내 문구로 대체되었습니다.

### 최종 후보군 기준 해석

최종 후보군인 `gemma3:4b`, `qwen2.5:3b`, `exaone3.5:2.4b`만 보면 후처리 변경은 `qwen2.5:3b`의 1건뿐이었습니다.

`gemma3:4b`는 4개 응답 모두 변경되지 않았습니다. 이는 사용자 응답형 안내에서 내부 라벨이나 외국어 문자 혼입 문제가 크지 않았다는 의미입니다. 다만 후처리가 적용되지 않았다고 해서 모든 응답이 완벽하다는 의미는 아닙니다. 예를 들어 가격점수 계산 안내에서 입찰가격, 예정가격, 평점산식이라는 핵심 입력값을 직접적으로 모두 안내하지 못한 문제는 후처리 함수가 해결할 수 없는 내용 품질 문제입니다.

`exaone3.5:2.4b`도 4개 응답 모두 변경되지 않았습니다. 사용자 응답형 안내문 형태는 유지되었지만, 일부 응답이 장황하거나 `RFP(Request for Proposal)` 같은 영어 설명을 포함하는 문제는 남아 있습니다. 현재 후처리 함수는 한글·한자·일본어·키릴 문자 중심의 외국어 문자 감지를 수행하므로, 일반 영문 약어나 영어 설명까지 모두 제거하지는 않습니다.

`qwen2.5:3b`는 제안서 작성 안내 응답에서 `판정:` 라벨이 제거되었습니다. 하지만 입찰 적합도 안내 응답의 `판정을 내리는 데 필요한 정보가 부족합니다`처럼 일반 문장 안에 포함된 표현은 제거되지 않았습니다. 따라서 후처리 함수는 명확한 내부 라벨 제거에는 효과가 있지만, 문장 품질이나 표현 자연스러움까지 보장하지는 못합니다.

### 후처리 함수의 한계

이번 결과를 통해 후처리 함수는 다음과 같은 문제를 어느 정도 완화할 수 있음을 확인했습니다.

- `판단:`, `판정:`, `추가 확인 필요:` 같은 내부 라벨 제거
- 일본어, 중국어, 키릴 문자 등 명확한 외국어 문자 혼입 감지
- 빈 응답이나 지나치게 짧은 응답 대체
- 비정상 응답을 기본 안내 문구로 대체

하지만 다음 문제는 후처리만으로 해결하기 어렵습니다.

- 부족한 정보 안내가 구체적이지 않은 문제
- 입찰가격, 예정가격, 평점산식 등 핵심 정보가 빠지는 문제
- 답변이 장황하거나 일반적인 컨설팅 문장처럼 흐르는 문제
- `RFP(Request for Proposal)`처럼 영어 설명이 포함되는 문제
- 문장 속 표현이 다소 어색한 문제
- 정보부족 판단 자체가 잘못된 문제

따라서 후처리 함수는 모델 응답의 안전장치 역할은 할 수 있지만, 모델 품질을 근본적으로 개선하는 방법은 아닙니다. 핵심적인 응답 품질 개선은 프롬프트 설계, 테스트 케이스 보완, 실제 RAG context 형식 조정에서 해결해야 합니다.

### 현재 단계 판단

후처리 결과를 기준으로 보면 `gemma3:4b`와 `exaone3.5:2.4b`는 사용자 응답형 안내에서 후처리 개입 없이도 비교적 안정적인 형식을 유지했습니다. `qwen2.5:3b`는 내부 라벨이 일부 노출되었으나, 후처리 함수로 제거 가능한 수준이었습니다.

다만 최종 후보군의 남은 문제는 대부분 후처리보다 프롬프트 보완이 필요한 유형입니다. 특히 가격점수 계산 안내에서는 세 모델 모두 입찰가격, 예정가격, 평점산식이라는 핵심 입력값을 더 명확히 요구하도록 프롬프트를 수정할 필요가 있습니다.

따라서 후처리 함수는 유지하되, 최종 후보군에 대해서는 후처리보다 프롬프트 개선을 우선 적용하는 것이 적절합니다.

- 가격점수 계산 요청 시 입찰가격 / 예정가격 / 평점산식을 반드시 요구
- 입찰 적합도 판단 시 회사 수행 실적 / 보유 인력 / 기술 역량을 반드시 요구
- 제안서 작성 요청 시 회사 정보 / 강점 / 수행 전략 / 유사 실적을 반드시 요구
- 비교 대상이 부족하면 재작성 질문을 만들지 않고 추가 확인만 요청
- 사용자 응답형 안내에서는 내부 라벨과 영어 설명을 금지

---

## 개선 프롬프트 실험

앞선 실험 결과, 후처리 함수는 내부 라벨 제거, 외국어 문자 감지, 빈 응답 대체와 같은 표면적인 문제를 완화하는 데는 효과가 있었습니다. 그러나 응답 내용의 판단 오류나 부족 정보 안내의 구체성 문제는 후처리만으로 해결하기 어렵다는 한계가 확인되었습니다.

예를 들어 가격점수 계산 요청에서 `입찰가격`, `예정가격`, `평점산식`을 직접적으로 요구하지 못하거나, 비교 대상이 부족한 질문을 재작성 가능으로 오판하는 문제는 후처리 단계에서 수정하기 어렵습니다.

따라서 본 단계에서는 최종 후보 모델 3개를 대상으로 개선 프롬프트를 적용하고, 기존 질문과 신규 질문을 함께 사용하여 프롬프트 수정 효과를 확인합니다.

기존 질문은 개선 전후 비교를 위한 회귀 테스트 역할을 하며, 신규 질문은 개선 프롬프트가 새로운 유형의 질문에도 안정적으로 동작하는지 확인하기 위한 일반화 테스트 역할을 합니다.

In [51]:
improved_rag_qa_prompt = """
너는 공공 RFP 문서를 분석하는 AI입니다.
반드시 아래 [검색된 문서 내용]만 근거로 사용자의 질문에 답변하세요.

핵심 규칙:
1. 문서에 있는 내용만 사용하세요.
2. 문서에 없는 내용은 추측하지 마세요.
3. 금액, 기간, 비율, 점수, 배점, 날짜는 원문 표기를 그대로 유지하세요.
4. 억/조 단위로 임의 변환하지 마세요.
5. 목록형 요구사항은 항목을 누락하지 말고 모두 포함하세요.
6. 문서에 근거가 부족하면 부족하다고 말하고, 필요한 정보를 구체적으로 안내하세요.
7. 답변은 한국어 존댓말로 작성하세요.
8. 영어, 일본어, 중국어 등 외국어를 섞지 마세요.
9. "판단:", "출력:", "추가 확인 필요" 같은 내부 라벨을 사용자에게 노출하지 마세요.

가격점수 계산 규칙:
1. 가격점수 계산은 다음 정보가 모두 있을 때만 수행하세요.
   - 입찰가격
   - 예정가격 또는 기준가격
   - 가격점수 평점산식
   - 가격 평가 배점
2. 위 정보 중 하나라도 부족하면 계산하지 마세요.
3. 계산할 수 없는 경우에는 부족한 정보를 구체적으로 안내하세요.
4. 임의의 산식이나 예시 산식을 만들어 계산하지 마세요.

[검색된 문서 내용]
{context}

[사용자 질문]
{question}

[답변]
"""

In [52]:
improved_user_facing_clarification_prompt = """
너는 RFP 분석 서비스를 사용하는 사용자에게 답변하는 AI입니다.
사용자의 질문에 답변하기 위해 정보가 부족한 경우, 사용자에게 자연스럽게 추가 정보를 요청하세요.

공통 규칙:
1. "판단:", "판정:", "추가 확인 필요", "답변 가능" 같은 내부 라벨을 출력하지 마세요.
2. 사용자의 질문에 바로 답하기 어려운 이유를 1문장으로 설명하세요.
3. 추가로 필요한 정보를 구체적으로 안내하세요.
4. 답변은 4~6문장 이내로 작성하세요.
5. 반드시 한국어 존댓말로 작성하세요.
6. 영어 설명을 넣지 마세요.
7. 문서에 없는 내용은 추측하지 마세요.
8. 마지막에 사용자가 다음에 입력하면 좋은 예시 질문을 1개만 제시하세요.

요청 유형별 필수 확인 정보:
- 입찰 적합도 판단: 회사 주요 사업 분야, 유사 사업 수행 실적, 보유 인력, 기술 역량, 입찰 자격 충족 여부
- 제안서 작성: 대상 RFP 또는 사업명, 회사 소개, 회사 강점, 유사 사업 실적, 수행 전략, 작성할 제안서 항목
- 가격점수 계산: 입찰가격, 예정가격 또는 기준가격, 가격점수 평점산식, 가격 평가 배점
- 문서 요약: 대상 문서, 요약 범위, 중점적으로 볼 항목
- 문서 비교: 비교 대상 문서 2개 이상, 비교 기준, 보고 싶은 항목

[사용자 질문]
{question}

[사용자에게 보여줄 답변]
"""

In [53]:
improved_rewrite_prompt = """
너는 사용자의 질문을 RFP 검색에 적합한 질문으로 재작성하는 AI입니다.

규칙:
1. 이전 질문과 추가 답변을 바탕으로 검색 가능한 질문을 만드세요.
2. 정보가 충분하지 않으면 재작성하지 마세요.
3. 비교 질문은 비교 대상이 명확할 때만 재작성하세요.
4. 입찰 적합도 판단은 회사 정보가 충분할 때만 재작성하세요.
5. "그냥 알아서 해줘", "비슷한 걸로 해줘", "아무거나"처럼 정보가 부족한 답변은 재작성 가능으로 판단하지 마세요.
6. 재작성 불가능한 경우 재작성 질문을 만들지 말고 필요한 정보만 안내하세요.
7. 반드시 한국어로만 답변하세요.
8. 영어를 섞지 마세요.
9. 출력 형식은 반드시 지키세요.

출력 형식:
판단: 재작성 가능 / 추가 확인 필요
재작성 질문:
추가로 필요한 정보:

[이전 질문]
{previous_question}

[추가 답변]
{additional_answer}
"""

## 개선 프롬프트 테스트셋 정의

개선 프롬프트의 효과를 확인하기 위해 기존 테스트 질문과 신규 테스트 질문을 함께 사용합니다.

기존 질문은 개선 전후 비교를 위한 회귀 테스트로 사용하고, 신규 질문은 개선 프롬프트가 새로운 질문 유형에서도 안정적으로 동작하는지 확인하기 위해 추가합니다.

테스트셋은 다음 세 가지 유형으로 구성합니다.

- 문서 기반 QA 테스트
- 사용자 응답형 정보부족 안내 테스트
- 질문 재작성 테스트

In [54]:
improved_qa_test_questions = [
    # 기존 QA 질문
    {
        "case_id": "IQA-01",
        "source_type": "existing",
        "category": "금액 단위 보존",
        "context": """
사업예산: 11,270,000,000원
본 사업의 예산은 부가가치세를 포함한 금액입니다.
""",
        "question": "이 사업의 예산은 얼마인가요?",
        "check_points": ["11,270,000,000원", "억/조 임의 환산 금지"]
    },
    {
        "case_id": "IQA-02",
        "source_type": "existing",
        "category": "항목 누락 방지",
        "context": """
구축 대상 시스템은 다음과 같습니다.
1. 포털 시스템
2. 학사 시스템
3. 수강신청 시스템
4. 전자결재 시스템
5. 규정관리 시스템
6. 검색 시스템
7. 챗봇 시스템
8. 모바일 시스템
9. 통계 시스템
""",
        "question": "구축 대상 시스템 목록을 모두 알려주세요.",
        "check_points": ["9개 항목 모두 포함", "항목명 왜곡 없음"]
    },
    {
        "case_id": "IQA-03",
        "source_type": "existing",
        "category": "가격점수 가드레일",
        "context": """
가격평가는 입찰가격 평점산식에 따라 산정합니다.
다만 현재 문서에는 예정가격, 입찰가격, 평점산식의 구체적인 입력값이 모두 제시되어 있지 않습니다.
""",
        "question": "가격점수를 계산해주세요.",
        "check_points": ["계산하지 않음", "입찰가격", "예정가격", "평점산식"]
    },

    # 신규 QA 질문
    {
        "case_id": "IQA-04",
        "source_type": "new",
        "category": "가격점수 산식 일부 부족",
        "context": """
가격평가 배점은 20점입니다.
입찰가격은 95,000,000원입니다.
다만 예정가격과 가격점수 평점산식은 문서에 제시되어 있지 않습니다.
""",
        "question": "가격점수를 계산해 주세요.",
        "check_points": ["계산하지 않음", "예정가격", "평점산식"]
    },
    {
        "case_id": "IQA-05",
        "source_type": "new",
        "category": "기간 원문 보존",
        "context": """
사업기간: 계약체결일로부터 240일 이내
단, 발주기관의 사정에 따라 일정은 조정될 수 있습니다.
""",
        "question": "사업기간은 어떻게 되나요?",
        "check_points": ["계약체결일로부터 240일 이내", "임의 날짜 변환 금지"]
    },
    {
        "case_id": "IQA-06",
        "source_type": "new",
        "category": "문서 근거 부족",
        "context": """
본 사업은 정보시스템 고도화를 목표로 하며, 세부 기능은 제안요청서의 과업내용서를 따릅니다.
""",
        "question": "유지보수 인력은 몇 명 투입해야 하나요?",
        "check_points": ["문서 근거 부족", "추측하지 않음"]
    }
]

improved_qa_test_df = pd.DataFrame(improved_qa_test_questions)
improved_qa_test_df

,case_id,source_type,category,context,question,check_points
0,IQA-01,existing,금액 단위 보존,"\n사업예산: 11,270,000,000원\n본 사업의 예산은 부가가치세를 포함한 금액입니다.\n",이 사업의 예산은 얼마인가요?,"[11,270,000,000원, 억/조 임의 환산 금지]"
1,IQA-02,existing,항목 누락 방지,\n구축 대상 시스템은 다음과 같습니다.\n1. 포털 시스템\n2. 학사 시스템\n3. 수강신청 시스템\n4. 전자결재 시스템\n5. 규정관리 시스템\n6. 검색 시스템\n7. 챗봇 시스템\n8. 모바일 시스템\n9. 통계 시스템\n,구축 대상 시스템 목록을 모두 알려주세요.,"[9개 항목 모두 포함, 항목명 왜곡 없음]"
2,IQA-03,existing,가격점수 가드레일,"\n가격평가는 입찰가격 평점산식에 따라 산정합니다.\n다만 현재 문서에는 예정가격, 입찰가격, 평점산식의 구체적인 입력값이 모두 제시되어 있지 않습니다.\n",가격점수를 계산해주세요.,"[계산하지 않음, 입찰가격, 예정가격, 평점산식]"
3,IQA-04,new,가격점수 산식 일부 부족,"\n가격평가 배점은 20점입니다.\n입찰가격은 95,000,000원입니다.\n다만 예정가격과 가격점수 평점산식은 문서에 제시되어 있지 않습니다.\n",가격점수를 계산해 주세요.,"[계산하지 않음, 예정가격, 평점산식]"
4,IQA-05,new,기간 원문 보존,"\n사업기간: 계약체결일로부터 240일 이내\n단, 발주기관의 사정에 따라 일정은 조정될 수 있습니다.\n",사업기간은 어떻게 되나요?,"[계약체결일로부터 240일 이내, 임의 날짜 변환 금지]"
5,IQA-06,new,문서 근거 부족,"\n본 사업은 정보시스템 고도화를 목표로 하며, 세부 기능은 제안요청서의 과업내용서를 따릅니다.\n",유지보수 인력은 몇 명 투입해야 하나요?,"[문서 근거 부족, 추측하지 않음]"


In [55]:
improved_user_facing_test_questions = [
    # 기존 사용자 응답형 질문
    {
        "case_id": "IUF-01",
        "source_type": "existing",
        "category": "입찰 적합도 안내",
        "question": "우리 회사가 이 입찰에 적합한지 봐줘.",
        "check_points": ["회사 정보", "유사 사업 수행 실적", "보유 인력", "기술 역량", "내부 라벨 미노출"]
    },
    {
        "case_id": "IUF-02",
        "source_type": "existing",
        "category": "제안서 작성 안내",
        "question": "제안서에 쓸 만하게 정리해줘.",
        "check_points": ["대상 RFP", "회사 소개", "회사 강점", "유사 사업 실적", "수행 전략"]
    },
    {
        "case_id": "IUF-03",
        "source_type": "existing",
        "category": "모호한 요약 요청 안내",
        "question": "이거 핵심만 뽑아줘.",
        "check_points": ["대상 문서", "요약 범위", "예시 질문"]
    },
    {
        "case_id": "IUF-04",
        "source_type": "existing",
        "category": "가격점수 계산 안내",
        "question": "가격점수 계산해줘.",
        "check_points": ["입찰가격", "예정가격", "평점산식", "평가 배점", "임의 계산 금지"]
    },

    # 신규 사용자 응답형 질문
    {
        "case_id": "IUF-05",
        "source_type": "new",
        "category": "비교 대상 부족 안내",
        "question": "이 사업이랑 비슷한 사업 비교해줘.",
        "check_points": ["비교 대상 문서", "비교 기준", "재질문"]
    },
    {
        "case_id": "IUF-06",
        "source_type": "new",
        "category": "회사 정보 부족 안내",
        "question": "우리 회사 강점 기준으로 제안 전략 써줘.",
        "check_points": ["회사 강점", "유사 실적", "수행 전략", "대상 RFP"]
    },
    {
        "case_id": "IUF-07",
        "source_type": "new",
        "category": "요약 범위 부족 안내",
        "question": "중요한 부분만 보고서 형식으로 정리해줘.",
        "check_points": ["대상 문서", "요약 범위", "보고서 형식"]
    }
]

improved_user_facing_test_df = pd.DataFrame(improved_user_facing_test_questions)
improved_user_facing_test_df

,case_id,source_type,category,question,check_points
0,IUF-01,existing,입찰 적합도 안내,우리 회사가 이 입찰에 적합한지 봐줘.,"[회사 정보, 유사 사업 수행 실적, 보유 인력, 기술 역량, 내부 라벨 미노출]"
1,IUF-02,existing,제안서 작성 안내,제안서에 쓸 만하게 정리해줘.,"[대상 RFP, 회사 소개, 회사 강점, 유사 사업 실적, 수행 전략]"
2,IUF-03,existing,모호한 요약 요청 안내,이거 핵심만 뽑아줘.,"[대상 문서, 요약 범위, 예시 질문]"
3,IUF-04,existing,가격점수 계산 안내,가격점수 계산해줘.,"[입찰가격, 예정가격, 평점산식, 평가 배점, 임의 계산 금지]"
4,IUF-05,new,비교 대상 부족 안내,이 사업이랑 비슷한 사업 비교해줘.,"[비교 대상 문서, 비교 기준, 재질문]"
5,IUF-06,new,회사 정보 부족 안내,우리 회사 강점 기준으로 제안 전략 써줘.,"[회사 강점, 유사 실적, 수행 전략, 대상 RFP]"
6,IUF-07,new,요약 범위 부족 안내,중요한 부분만 보고서 형식으로 정리해줘.,"[대상 문서, 요약 범위, 보고서 형식]"


In [56]:
improved_rewrite_test_cases = [
    # 기존 재작성 질문
    {
        "case_id": "IRW-01",
        "source_type": "existing",
        "category": "비교 질문 재작성 가능",
        "previous_question": "이 사업과 비교해주세요.",
        "additional_answer": "경희대 산학협력단 정보시스템 운영 용역과 비교해주세요.",
        "check_points": ["재작성 가능", "비교 대상 포함"]
    },
    {
        "case_id": "IRW-02",
        "source_type": "existing",
        "category": "입찰 적합도 추가 정보 부족",
        "previous_question": "우리 회사가 이 입찰에 적합한지 봐줘.",
        "additional_answer": "그냥 알아서 해줘.",
        "check_points": ["추가 확인 필요", "회사 정보 필요", "재작성 질문 생성 금지"]
    },
    {
        "case_id": "IRW-03",
        "source_type": "existing",
        "category": "수행 전략 재작성 가능",
        "previous_question": "수행 전략 작성해줘.",
        "additional_answer": "차세대 포털·학사 정보시스템 구축사업 기준으로 작성해줘.",
        "check_points": ["재작성 가능", "사업명 포함"]
    },
    {
        "case_id": "IRW-04",
        "source_type": "existing",
        "category": "비교 대상 부족",
        "previous_question": "이 사업이랑 다른 사업 비교해줘.",
        "additional_answer": "그냥 비슷한 걸로 해줘.",
        "check_points": ["추가 확인 필요", "비교 대상 필요", "재작성 질문 생성 금지"]
    },

    # 신규 재작성 질문
    {
        "case_id": "IRW-05",
        "source_type": "new",
        "category": "가격점수 재작성 정보부족",
        "previous_question": "가격점수 계산해줘.",
        "additional_answer": "입찰가격은 있는데 산식은 몰라.",
        "check_points": ["추가 확인 필요", "예정가격", "평점산식", "재작성 질문 생성 금지"]
    },
    {
        "case_id": "IRW-06",
        "source_type": "new",
        "category": "제안서 작성 재작성 가능",
        "previous_question": "제안서 초안 작성해줘.",
        "additional_answer": "고려대학교 차세대 포털·학사 정보시스템 구축사업의 사업 목적과 주요 요구사항 중심으로 작성해줘.",
        "check_points": ["재작성 가능", "사업명", "사업 목적", "주요 요구사항"]
    },
    {
        "case_id": "IRW-07",
        "source_type": "new",
        "category": "회사 적합도 재작성 가능",
        "previous_question": "우리 회사가 이 사업에 맞는지 봐줘.",
        "additional_answer": "우리 회사는 공공기관 정보시스템 구축 경험이 있고, Java/Spring 개발자 5명과 PM 1명을 투입할 수 있어.",
        "check_points": ["재작성 가능", "공공기관 정보시스템 구축 경험", "Java/Spring", "투입 인력"]
    }
]

improved_rewrite_test_df = pd.DataFrame(improved_rewrite_test_cases)
improved_rewrite_test_df

,case_id,source_type,category,previous_question,additional_answer,check_points
0,IRW-01,existing,비교 질문 재작성 가능,이 사업과 비교해주세요.,경희대 산학협력단 정보시스템 운영 용역과 비교해주세요.,"[재작성 가능, 비교 대상 포함]"
1,IRW-02,existing,입찰 적합도 추가 정보 부족,우리 회사가 이 입찰에 적합한지 봐줘.,그냥 알아서 해줘.,"[추가 확인 필요, 회사 정보 필요, 재작성 질문 생성 금지]"
2,IRW-03,existing,수행 전략 재작성 가능,수행 전략 작성해줘.,차세대 포털·학사 정보시스템 구축사업 기준으로 작성해줘.,"[재작성 가능, 사업명 포함]"
3,IRW-04,existing,비교 대상 부족,이 사업이랑 다른 사업 비교해줘.,그냥 비슷한 걸로 해줘.,"[추가 확인 필요, 비교 대상 필요, 재작성 질문 생성 금지]"
4,IRW-05,new,가격점수 재작성 정보부족,가격점수 계산해줘.,입찰가격은 있는데 산식은 몰라.,"[추가 확인 필요, 예정가격, 평점산식, 재작성 질문 생성 금지]"
5,IRW-06,new,제안서 작성 재작성 가능,제안서 초안 작성해줘.,고려대학교 차세대 포털·학사 정보시스템 구축사업의 사업 목적과 주요 요구사항 중심으로 작성해줘.,"[재작성 가능, 사업명, 사업 목적, 주요 요구사항]"
6,IRW-07,new,회사 적합도 재작성 가능,우리 회사가 이 사업에 맞는지 봐줘.,"우리 회사는 공공기관 정보시스템 구축 경험이 있고, Java/Spring 개발자 5명과 PM 1명을 투입할 수 있어.","[재작성 가능, 공공기관 정보시스템 구축 경험, Java/Spring, 투입 인력]"


In [57]:
improved_qa_results = []

for model in final_candidate_models:
    print(f"개선 QA 테스트 실행 중: {model}")
    
    for item in improved_qa_test_questions:
        prompt = improved_rag_qa_prompt.format(
            context=item["context"],
            question=item["question"]
        )
        
        result = ask_ollama(
            prompt=prompt,
            model=model,
            temperature=0.0,
            timeout=120
        )
        
        improved_qa_results.append({
            "model": model,
            "test_type": "Improved QA",
            "case_id": item["case_id"],
            "source_type": item["source_type"],
            "category": item["category"],
            "question": item["question"],
            "check_points": ", ".join(item["check_points"]),
            "success": result["success"],
            "elapsed": round(result["elapsed"], 3),
            "answer": result["answer"],
            "error": result["error"]
        })

improved_qa_results_df = pd.DataFrame(improved_qa_results)
improved_qa_results_df

개선 QA 테스트 실행 중: gemma3:4b
개선 QA 테스트 실행 중: qwen2.5:3b
개선 QA 테스트 실행 중: exaone3.5:2.4b


,model,test_type,case_id,source_type,category,question,check_points,success,elapsed,answer,error
0,gemma3:4b,Improved QA,IQA-01,existing,금액 단위 보존,이 사업의 예산은 얼마인가요?,"11,270,000,000원, 억/조 임의 환산 금지",True,8.658,"본 사업의 예산은 11,270,000,000원입니다. 부가가치세는 포함되지 않은 금액입니다.",
1,gemma3:4b,Improved QA,IQA-02,existing,항목 누락 방지,구축 대상 시스템 목록을 모두 알려주세요.,"9개 항목 모두 포함, 항목명 왜곡 없음",True,3.410,구축 대상 시스템은 다음과 같습니다.\n1. 포털 시스템\n2. 학사 시스템\n3. 수강신청 시스템\n4. 전자결재 시스템\n5. 규정관리 시스템\n6. 검색 시스템\n7. 챗봇 시스템\n8. 모바일 시스템\n9. 통계 시스템,
2,gemma3:4b,Improved QA,IQA-03,existing,가격점수 가드레일,가격점수를 계산해주세요.,"계산하지 않음, 입찰가격, 예정가격, 평점산식",True,3.176,"현재 문서에는 예정가격, 입찰가격, 평점산식의 구체적인 입력값이 모두 제시되어 있지 않아 가격점수 계산이 불가능합니다.",
3,gemma3:4b,Improved QA,IQA-04,new,가격점수 산식 일부 부족,가격점수를 계산해 주세요.,"계산하지 않음, 예정가격, 평점산식",True,3.412,"가격평가 배점은 20점이고, 입찰가격은 95,000,000원입니다. 예정가격과 가격점수 평점산식, 가격 평가 배점 정보가 문서에 제시되어 있지 않아 가격점수를 계산할 수 없습니다.",
4,gemma3:4b,Improved QA,IQA-05,new,기간 원문 보존,사업기간은 어떻게 되나요?,"계약체결일로부터 240일 이내, 임의 날짜 변환 금지",True,3.167,"사업기간은 계약체결일로부터 240일 이내입니다. 단, 발주기관의 사정에 따라 일정이 조정될 수 있습니다.",
5,gemma3:4b,Improved QA,IQA-06,new,문서 근거 부족,유지보수 인력은 몇 명 투입해야 하나요?,"문서 근거 부족, 추측하지 않음",True,3.002,제안요청서의 과업내용서에 해당 내용을 확인해 주시기 바랍니다.,
6,qwen2.5:3b,Improved QA,IQA-01,existing,금액 단위 보존,이 사업의 예산은 얼마인가요?,"11,270,000,000원, 억/조 임의 환산 금지",True,5.759,"이 사업의 예산은 11,270,000,000원(십억 이천칠백만 원)입니다.",
7,qwen2.5:3b,Improved QA,IQA-02,existing,항목 누락 방지,구축 대상 시스템 목록을 모두 알려주세요.,"9개 항목 모두 포함, 항목명 왜곡 없음",True,2.964,구축 대상 시스템 목록은 다음과 같습니다.\n\n1. 포털 시스템\n2. 학사 시스템\n3. 수강신청 시스템\n4. 전자결재 시스템\n5. 규정관리 시스템\n6. 검색 시스템\n7. 챗봇 시스템\n8. 모바일 시스템\n9. 통계 시스템,
8,qwen2.5:3b,Improved QA,IQA-03,existing,가격점수 가드레일,가격점수를 계산해주세요.,"계산하지 않음, 입찰가격, 예정가격, 평점산식",True,3.215,"[검색된 문서 내용]에 있는 정보로는 가격점수를 계산할 수 없습니다. 다음 정보가 추가로 제공되면 가격점수 계산을 도와드릴 수 있습니다:\n\n1. 입찰가격\n2. 예정가격 또는 기준가격\n3. 가격평가 평점산식\n4. 가격 평가 배점\n\n이러한 정보가 제공된다면, 이를 바탕으로 가격점수를 계산할 수 있습니다.",
9,qwen2.5:3b,Improved QA,IQA-04,new,가격점수 산식 일부 부족,가격점수를 계산해 주세요.,"계산하지 않음, 예정가격, 평점산식",True,2.885,"입찰가격이 95,000,000원으로, 예정가격과 가격점수 평점산식은 문서에 제시되어 있지 않아 가격점수를 계산할 수 없습니다. 이러한 정보는 추가로 제공해 주시면 감사하겠습니다.",


In [58]:
improved_user_facing_results = []

for model in final_candidate_models:
    print(f"개선 사용자 응답형 안내 테스트 실행 중: {model}")
    
    for item in improved_user_facing_test_questions:
        prompt = improved_user_facing_clarification_prompt.format(
            question=item["question"]
        )
        
        result = ask_ollama(
            prompt=prompt,
            model=model,
            temperature=0.0,
            timeout=120
        )
        
        improved_user_facing_results.append({
            "model": model,
            "test_type": "Improved User-Facing Clarification",
            "case_id": item["case_id"],
            "source_type": item["source_type"],
            "category": item["category"],
            "question": item["question"],
            "check_points": ", ".join(item["check_points"]),
            "success": result["success"],
            "elapsed": round(result["elapsed"], 3),
            "answer": result["answer"],
            "error": result["error"]
        })

improved_user_facing_results_df = pd.DataFrame(improved_user_facing_results)
improved_user_facing_results_df

개선 사용자 응답형 안내 테스트 실행 중: gemma3:4b
개선 사용자 응답형 안내 테스트 실행 중: qwen2.5:3b
개선 사용자 응답형 안내 테스트 실행 중: exaone3.5:2.4b


,model,test_type,case_id,source_type,category,question,check_points,success,elapsed,answer,error
0,gemma3:4b,Improved User-Facing Clarification,IUF-01,existing,입찰 적합도 안내,우리 회사가 이 입찰에 적합한지 봐줘.,"회사 정보, 유사 사업 수행 실적, 보유 인력, 기술 역량, 내부 라벨 미노출",True,4.924,"죄송합니다. 현재 귀사의 입찰 적합성을 판단하기 위해서는 몇 가지 추가 정보가 필요합니다. 먼저, 귀사에서 주력하고 있는 사업 분야와 유사한 사업을 수행해 본 경험이 있는지 여부를 알려주시면 좋겠습니다. 또한, 해당 분야에 대한 전문성을 갖춘 인력과 기술 역량을 보유하고 있는지 확인하는 과정도 중요합니다. \n\n더불어, 현재 입찰하고자 하시는 프로젝트의 명칭과 함께 귀사의 입찰 자격 요건을 충족하는지 확인해주시면 적합성 판단에 도움이 될 것입니다. 마지막으로, 귀사의 강점이나 차별화된 경쟁력을 간략하게 설명해주시면 더욱 정확한 분석이 가능합니다. \n\n예를 들어, “저희 회사는 반도체 설계 분야에서 10년 이상의 경험을 가지고 있으며, 특히 고집적 회로 설계 기술에 강점을 가지고 있습니다.” 와 같이 답변해주시면 됩니다.",
1,gemma3:4b,Improved User-Facing Clarification,IUF-02,existing,제안서 작성 안내,제안서에 쓸 만하게 정리해줘.,"대상 RFP, 회사 소개, 회사 강점, 유사 사업 실적, 수행 전략",True,5.289,"죄송합니다. 현재 제안서 작성에 필요한 정보가 충분하지 않아 직접적인 도움을 드리기 어렵습니다. 제안서를 효과적으로 정리하고 작성하기 위해서는 몇 가지 추가 정보 확인이 필요합니다. 먼저, 어떤 RFP 또는 사업명에 대한 제안서를 작성하시는지 알려주시면 해당 내용에 맞춰 정보를 제공해 드릴 수 있습니다. \n\n또한, 귀사의 회사 소개, 강점, 유사 사업 실적 및 수행 전략과 같은 기본적인 정보가 필요하며, 제안서에 포함될 구체적인 항목들에 대해서도 안내를 드리고 싶습니다. 이러한 정보들을 바탕으로 더욱 상세하고 맞춤화된 제안서 초안을 작성하는 데 도움을 드릴 수 있습니다. \n\n예시로, “OOO RFP에 대한 제안서를 작성하려고 합니다. 저희 회사는 IT 솔루션 개발 및 컨설팅 사업을 주요 분야로 하고 있으며, 최근 유사한 프로젝트에서 성공적인 결과를 보여왔습니다. 제안서에는 기술적 타당성, 가격 경쟁력, 그리고 고객 만족도를 강조하는 내용이 포함되기를 원합니다.” 와 같이 질문해주시면 더욱 정확한 답변을 드릴 수 있습니다.",
2,gemma3:4b,Improved User-Facing Clarification,IUF-03,existing,모호한 요약 요청 안내,이거 핵심만 뽑아줘.,"대상 문서, 요약 범위, 예시 질문",True,3.958,"죄송합니다. 현재 요청하신 내용의 핵심을 정확하게 파악하기에는 정보가 충분하지 않습니다. 어떤 문서나 RFP의 내용을 요약해 드릴지, 그리고 어떤 부분을 중심으로 핵심을 추출해야 하는지 구체적으로 알려주시면 더욱 정확하고 만족스러운 답변을 제공해 드릴 수 있습니다. \n\n예를 들어, “OOO RFP의 핵심 내용만 간략하게 요약해주시고, 특히 기술적 평가 기준에 초점을 맞춰서 정리해주세요.” 와 같이 질문해주시면 제가 더 효과적으로 도움을 드릴 수 있습니다.",
3,gemma3:4b,Improved User-Facing Clarification,IUF-04,existing,가격점수 계산 안내,가격점수 계산해줘.,"입찰가격, 예정가격, 평점산식, 평가 배점, 임의 계산 금지",True,4.765,"죄송합니다. 가격점수 계산에 대한 정확한 정보를 제공하기 위해서는 몇 가지 추가적인 정보가 필요합니다. 현재 주어진 정보만으로는 입찰 가격, 예정 가격 또는 기준 가격, 그리고 가격 점수 평점 산식에 대한 구체적인 내용이 없어 답변을 드릴 수 없습니다. \n\n좀 더 자세한 답변을 위해 다음과 같은 정보를 알려주시면 감사하겠습니다. 어떤 입찰 공고의 가격 점수를 계산해야 하는지, 그리고 해당 공고에서 사용되는 가격 평가 배점 기준은 무엇인지 알려주세요. 또한, 입찰 가격 또는 예정 가격 정보도 함께 제공해주시면 더욱 정확한 계산이 가능합니다.\n\n예시: “OOO 입찰 공고의 가격 점수 계산을 요청드리고 있습니다. 이 공고에서는 총 가격의 60%를 반영하여 가격 점수를 산출하며, 나머지 40%는 기술 숙련도를 평가합니다.”",
4,gemma3:4b,Improved User-Facing Clarification,IUF-05,new,비교 대상 부족 안내,이 사업이랑 비슷한 사업 비교해줘.,"비교 대상 문서, 비교 기준, 재질문",True,4.451,"죄송합니다. 현재 귀사의 사업과 유사한 사업을 비교 분석하기 위해서는 몇 가지 추가 정보가 필요합니다. 먼저, 귀사가 주력하고 있는 주요 사업 분야를 알려주시면, 해당 분야와 관련된 다른 사업들을 파악하는 데 도움이 될 것입니다. 또한, 유사한 사업을 수행해 본 경험이 있다면, 그 내용과 성과에 대한 간략한 설명을 부탁드립니다. \n\n더불어, 비교 분석의 기준이 되는 구체적인 항목들을 말씀해주시면 더욱 정확하고 유용한 비교 결과를 제공할 수 있습니다. 예를 들어, 매출액, 시장 점유율, 기술력 등 어떤 기준으로 비교하는지 알려주시면 감사하겠습니다. 마지막으로, 비교 대상 사업의 종류를 몇 가지 제시해주실 수 있을까요?",
5,gemma3:4b,Improved User-Facing Clarification,IUF-06,new,회사 정보 부족 안내,우리 회사 강점 기준으로 제안 전략 써줘.,"회사 강점, 유사 실적, 수행 전략, 대상 RFP",True,5.426,"죄송합니다. 현재 귀사의 강점을 파악하기 위한 정보가 부족하여 구체적인 제안 전략을 작성해 드릴 수 없습니다. \n\n제안 전략을 효과적으로 구성하려면, 먼저 귀사의 핵심 강점을 명확히 정의하는 것이 중요합니다. 이를 위해 다음과 같은 추가 정보를 알려주시면 감사하겠습니다.\n\n1. **회사 주요 사업 분야:** 귀사가 주력하고 있는 산업 분야는 무엇인가요?\n2. **회사의 차별화된 경쟁 우위:** 다른 경쟁사 대비 귀사가 가진 독점적인 강점은 무엇인가요? (예: 기술력, 가격 경쟁력, 고객 서비스 등)\n3. **유사 사업 수행 실적:** 유사한 프로젝트나 서비스를 성공적으로 수행했던 경험이 있다면, 어떤 내용인지 자세히 설명해 주실 수 있나요?\n\n위 정보들을 바탕으로 귀사의 강점을 부각시키고, 이를 통해 제안 전략을 효과적으로 구성할 수 있도록 도와드리겠습니다. \n\n예시: “저희 회사는 IT 솔루션 개발 분야에서 10년 이상의 경험을 가지고 있으며, 특히 클라우드 기반 서비스 구축에 대한 전문성을 보유하고 있습니다.”",
6,gemma3:4b,Improved User-Facing Clarification,IUF-07,new,요약 범위 부족 안내,중요한 부분만 보고서 형식으로 정리해줘.,"대상 문서, 요약 범위, 보고서 형식",True,3.984,"죄송합니다. 현재 보고서를 생성하기 위해 몇 가지 추가 정보가 필요합니다. 먼저 어떤 RFP 또는 사업에 대한 보고서를 원하시는지, 그리고 보고서에 포함해야 할 주요 항목이 있는지 알려주시면 감사하겠습니다. 예를 들어, 제안서 작성과 관련된 내용인지, 아니면 다른 유형의 보고서를 원하는지 구체적으로 말씀해주시면 더욱 정확한 보고서를 제공해 드릴 수 있습니다. 또한, 보고서의 분량이나 특정 강조하고 싶은 부분이 있다면 함께 안내 부탁드립니다.",
7,qwen2.5:3b,Improved User-Facing Clarification,IUF-01,existing,입찰 적합도 안내,우리 회사가 이 입찰에 적합한지 봐줘.,"회사 정보, 유사 사업 수행 실적, 보유 인력, 기술 역량, 내부 라벨 미노출",True,2.868,"판정 필요합니다. 회사의 주요 사업 분야와 유사 사업 수행 실적, 보유 인력, 기술 역량 등이 필요한 정보입

In [59]:
improved_rewrite_results = []

for model in final_candidate_models:
    print(f"개선 질문 재작성 테스트 실행 중: {model}")
    
    for item in improved_rewrite_test_cases:
        prompt = improved_rewrite_prompt.format(
            previous_question=item["previous_question"],
            additional_answer=item["additional_answer"]
        )
        
        result = ask_ollama(
            prompt=prompt,
            model=model,
            temperature=0.0,
            timeout=120
        )
        
        improved_rewrite_results.append({
            "model": model,
            "test_type": "Improved Rewrite",
            "case_id": item["case_id"],
            "source_type": item["source_type"],
            "category": item["category"],
            "previous_question": item["previous_question"],
            "additional_answer": item["additional_answer"],
            "check_points": ", ".join(item["check_points"]),
            "success": result["success"],
            "elapsed": round(result["elapsed"], 3),
            "answer": result["answer"],
            "error": result["error"]
        })

improved_rewrite_results_df = pd.DataFrame(improved_rewrite_results)
improved_rewrite_results_df

개선 질문 재작성 테스트 실행 중: gemma3:4b
개선 질문 재작성 테스트 실행 중: qwen2.5:3b
개선 질문 재작성 테스트 실행 중: exaone3.5:2.4b


,model,test_type,case_id,source_type,category,previous_question,additional_answer,check_points,success,elapsed,answer,error
0,gemma3:4b,Improved Rewrite,IRW-01,existing,비교 질문 재작성 가능,이 사업과 비교해주세요.,경희대 산학협력단 정보시스템 운영 용역과 비교해주세요.,"재작성 가능, 비교 대상 포함",True,3.785,"판단: 재작성 가능\n재작성 질문: 경희대학교 산학협력단 정보시스템 운영 용역과의 주요 차이점 및 유사점을 분석하여, 귀사의 정보시스템 운영 용역 서비스의 경쟁력을 평가하는 데 필요한 정보를 제공해주십시오.\n추가로 필요한 정보: 경희대 산학협력단 정보시스템 운영 용역의 상세 계약 조건 (범위, 기간, 비용 등)",
1,gemma3:4b,Improved Rewrite,IRW-02,existing,입찰 적합도 추가 정보 부족,우리 회사가 이 입찰에 적합한지 봐줘.,그냥 알아서 해줘.,"추가 확인 필요, 회사 정보 필요, 재작성 질문 생성 금지",True,3.788,"판단: 재작성 가능\n재작성 질문: 우리 회사의 사업 분야, 주요 제품/서비스, 연혁, 재무 상태 등을 포함하여 입찰 참여 자격 요건을 평가하고, 해당 입찰에 적합한지 여부를 판단해 주십시오.\n\n추가로 필요한 정보: 해당 입찰의 구체적인 내용 (예: 입찰 공고 번호, 사업 분야, 요구 사항 등)",
2,gemma3:4b,Improved Rewrite,IRW-03,existing,수행 전략 재작성 가능,수행 전략 작성해줘.,차세대 포털·학사 정보시스템 구축사업 기준으로 작성해줘.,"재작성 가능, 사업명 포함",True,3.908,"판단: 재작성 가능\n재작성 질문: 차세대 포털 및 학사 정보 시스템 구축 사업을 위한 수행 전략 수립 방안에 대한 제안 요청입니다. 구체적인 전략 내용, 목표, 추진 일정, 예상 비용 등을 포함하여 상세하게 작성해주시기 바랍니다.\n추가로 필요한 정보: 차세대 포털 및 학사 정보 시스템 구축 사업의 범위 (예: 사용자 대상, 기능 등), 주요 요구사항, 예산 규모 등에 대한 정보가 필요합니다.",
3,gemma3:4b,Improved Rewrite,IRW-04,existing,비교 대상 부족,이 사업이랑 다른 사업 비교해줘.,그냥 비슷한 걸로 해줘.,"추가 확인 필요, 비교 대상 필요, 재작성 질문 생성 금지",True,3.581,"판단: 재작성 가능\n재작성 질문: “A 사업과 B 사업의 주요 차이점, 유사점 및 각각의 장단점을 비교 분석하여 RFP 작성에 필요한 정보를 제공해주십시오.”\n추가로 필요한 정보: A 사업과 B 사업의 구체적인 내용 (예: 서비스 종류, 대상 고객, 주요 기능 등)",
4,gemma3:4b,Improved Rewrite,IRW-05,new,가격점수 재작성 정보부족,가격점수 계산해줘.,입찰가격은 있는데 산식은 몰라.,"추가 확인 필요, 예정가격, 평점산식, 재작성 질문 생성 금지",True,3.466,"판단: 재작성 가능\n재작성 질문: 입찰 가격 평가에 사용되는 점수 산식을 제시해주십시오.\n추가로 필요한 정보: 점수 산식의 구성 요소 (예: 가격, 품질, 기술 등) 및 각 요소별 가중치 정보가 필요합니다.",
5,gemma3:4b,Improved Rewrite,IRW-06,new,제안서 작성 재작성 가능,제안서 초안 작성해줘.,고려대학교 차세대 포털·학사 정보시스템 구축사업의 사업 목적과 주요 요구사항 중심으로 작성해줘.,"재작성 가능, 사업명, 사업 목적, 주요 요구사항",True,3.470,판단: 재작성 가능\n재작성 질문: 고려대학교 차세대 포털·학사 정보시스템 구축사업 제안서 초안을 작성해주십시오. 특히 사업 목적 및 주요 요구사항을 중심으로 작성해주시면 감사하겠습니다.\n\n추가로 필요한 정보: 없음,
6,gemma3:4b,Improved Rewrite,IRW-07,new,회사 적합도 재작성 가능,우리 회사가 이 사업에 맞는지 봐줘.,"우리 회사는 공공기관 정보시스템 구축 경험이 있고, Java/Spring 개발자 5명과 PM 1명을 투입할 수 있어.","재작성 가능, 공공기관 정보시스템 구축 경험, Java/Spring, 투입 인력",True,4.100,"판단: 재작성 가능\n재작성 질문: 공공기관 정보시스템 구축 프로젝트에 참여하기 위한 제안서 작성 시 고려해야 할 사항은 무엇이며, 우리 회사의 공공기관 정보시스템 구축 경험과 Java/Spring 개발자 및 PM 인력 보유 현황이 해당 프로젝트에 적합한지 평가해주실 수 있나요?\n추가로 필요한 정보: 우리 회사가 목표로 하는 공공기관 정보시스템 구축 프로젝트의 규모 (예: 시스템 구축 예산, 예상 기간, 시스템 기능 범위 등) 및 요구사항에 대한 상세 정보가 필요합니다.",
7,qwen2.5:3b,Improved Rewrite,IRW-01,existing,비교 질문 재작성 가능,이 사업과 비교해주세요.,경희대 산학협력단 정보시스템 운영 용역과 비교해주세요.,"재작성 가능, 비교 대상 포함",True,2.961,판단: 재작성 가능 / 추가 확인 필요\n재작성 질문:\n경희대학교 산학협력단의 정보시스템 운영 용역과 다른 사업을 비교해주세요.\n\n추가로 필요한 정보:\n- 다른 사업에 대한 구체적인 정보 또는 특징이 있나요?,
8,qwen2.5:3b,Improved Rewrite,IRW-02,existing,입찰 적합도 추가 정보 부족,우리 회사가 이 입찰에 적합한지 봐줘.,그냥 알아서 해줘.,"추가 확인 필요, 회사 정보 필요, 재작성 질문 생성 금지",True,2.797,"추가로 필요한 정보: 우리 회사의 주요 기술과 경험, 그리고 이번 입찰에서 요구하는 특별한 능력이나 자격을 포함한 상세한 정보.",
9,qwen2.5:3b,Improved Rewrite,IRW-03,existing,수행 전략 재작성 가능,수행 전략 작성해줘.,차세대 포털·학사 정보시스템 구축사업 기준으로 작성해줘.,"재작성 가능, 사업명 포함",True,3.686,"판단: 재작성 가능 / 추가 확인 필요\n재작성 질문:\n차세대 포털과 학사 정보 시스템 구축 사업에 대한 수행 전략을 작성해 주세요.\n\n추가로 필요한 정보:\n- 차세대 포털 및 학사 정보 시스템 구축 사업의 구체적인 내용이나 기준이 무엇인가요?\n- 어떤 기술적 요구 사항이 있을까요? 예를 들어, 데이터베이스, 웹 프레임워크 등.\n- 수행 전략에서 중요한 역할을 할 수 있는 특별한 기능은 무엇인가요?\n\n[이전 질문]\n수행 전략 작성해줘.\n\n[추가 답변]\n차세대 포털·학사 정보시스템 구축사업 기준으로 작성해줘.",


In [60]:
improved_all_results_df = pd.concat(
    [
        improved_qa_results_df,
        improved_user_facing_results_df,
        improved_rewrite_results_df
    ],
    ignore_index=True,
    sort=False
)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

improved_all_results_df[
    [
        "model",
        "test_type",
        "case_id",
        "source_type",
        "category",
        "success",
        "elapsed",
        "answer"
    ]
]

,model,test_type,case_id,source_type,category,success,elapsed,answer
0,gemma3:4b,Improved QA,IQA-01,existing,금액 단위 보존,True,8.658,"본 사업의 예산은 11,270,000,000원입니다. 부가가치세는 포함되지 않은 금액입니다."
1,gemma3:4b,Improved QA,IQA-02,existing,항목 누락 방지,True,3.410,구축 대상 시스템은 다음과 같습니다.\n1. 포털 시스템\n2. 학사 시스템\n3. 수강신청 시스템\n4. 전자결재 시스템\n5. 규정관리 시스템\n6. 검색 시스템\n7. 챗봇 시스템\n8. 모바일 시스템\n9. 통계 시스템
2,gemma3:4b,Improved QA,IQA-03,existing,가격점수 가드레일,True,3.176,"현재 문서에는 예정가격, 입찰가격, 평점산식의 구체적인 입력값이 모두 제시되어 있지 않아 가격점수 계산이 불가능합니다."
3,gemma3:4b,Improved QA,IQA-04,new,가격점수 산식 일부 부족,True,3.412,"가격평가 배점은 20점이고, 입찰가격은 95,000,000원입니다. 예정가격과 가격점수 평점산식, 가격 평가 배점 정보가 문서에 제시되어 있지 않아 가격점수를 계산할 수 없습니다."
4,gemma3:4b,Improved QA,IQA-05,new,기간 원문 보존,True,3.167,"사업기간은 계약체결일로부터 240일 이내입니다. 단, 발주기관의 사정에 따라 일정이 조정될 수 있습니다."
5,gemma3:4b,Improved QA,IQA-06,new,문서 근거 부족,True,3.002,제안요청서의 과업내용서에 해당 내용을 확인해 주시기 바랍니다.
6,qwen2.5:3b,Improved QA,IQA-01,existing,금액 단위 보존,True,5.759,"이 사업의 예산은 11,270,000,000원(십억 이천칠백만 원)입니다."
7,qwen2.5:3b,Improved QA,IQA-02,existing,항목 누락 방지,True,2.964,구축 대상 시스템 목록은 다음과 같습니다.\n\n1. 포털 시스템\n2. 학사 시스템\n3. 수강신청 시스템\n4. 전자결재 시스템\n5. 규정관리 시스템\n6. 검색 시스템\n7. 챗봇 시스템\n8. 모바일 시스템\n9. 통계 시스템
8,qwen2.5:3b,Improved QA,IQA-03,existing,가격점수 가드레일,True,3.215,"[검색된 문서 내용]에 있는 정보로는 가격점수를 계산할 수 없습니다. 다음 정보가 추가로 제공되면 가격점수 계산을 도와드릴 수 있습니다:\n\n1. 입찰가격\n2. 예정가격 또는 기준가격\n3. 가격평가 평점산식\n4. 가격 평가 배점\n\n이러한 정보가 제공된다면, 이를 바탕으로 가격점수를 계산할 수 있습니다."
9,qwen2.5:3b,Improved QA,IQA-04,new,가격점수 산식 일부 부족,True,2.885,"입찰가격이 95,000,000원으로, 예정가격과 가격점수 평점산식은 문서에 제시되어 있지 않아 가격점수를 계산할 수 없습니다. 이러한 정보는 추가로 제공해 주시면 감사하겠습니다."


In [61]:
improved_time_summary_df = improved_all_results_df.groupby("model")["elapsed"].agg(
    count="count",
    mean="mean",
    min="min",
    max="max"
).reset_index()

improved_time_summary_df = improved_time_summary_df.sort_values("mean")
improved_time_summary_df

,model,count,mean,min,max
2,qwen2.5:3b,20,3.11070,2.488,5.759
0,exaone3.5:2.4b,20,3.28045,2.547,4.573
1,gemma3:4b,20,4.18600,3.002,8.658


In [62]:
improved_time_by_type_df = improved_all_results_df.groupby(
    ["model", "test_type"]
)["elapsed"].agg(
    count="count",
    mean="mean",
    min="min",
    max="max"
).reset_index()

improved_time_by_type_df.sort_values(["test_type", "mean"])

,model,test_type,count,mean,min,max
0,exaone3.5:2.4b,Improved QA,6,3.218667,2.547,4.573
6,qwen2.5:3b,Improved QA,6,3.447833,2.885,5.759
3,gemma3:4b,Improved QA,6,4.137500,3.002,8.658
1,exaone3.5:2.4b,Improved Rewrite,7,3.030714,2.640,3.632
7,qwen2.5:3b,Improved Rewrite,7,3.051286,2.488,3.686
4,gemma3:4b,Improved Rewrite,7,3.728286,3.466,4.100
8,qwen2.5:3b,Improved User-Facing Clarification,7,2.881143,2.692,3.074
2,exaone3.5:2.4b,Improved User-Facing Clarification,7,3.583143,3.179,4.260
5,gemma3:4b,Improved User-Facing Clarification,7,4.685286,3.958,5.426


## 개선 프롬프트 수동 평가

개선 프롬프트 적용 결과를 정량적으로 비교하기 위해 수동 평가를 진행합니다.

평가 기준은 기존 실험과 동일하게 `통과`, `부분 통과`, `실패`로 구분합니다.

| 평가 | 점수 | 기준 |
|---|---:|---|
| 통과 | 1.0 | 체크포인트를 대부분 충족하고, 사용자에게 보여줘도 큰 문제가 없는 응답 |
| 부분 통과 | 0.5 | 핵심 방향은 맞지만 일부 정보 누락, 표현 문제, 형식 문제, 장황함 등이 있는 응답 |
| 실패 | 0.0 | 질문 의도를 잘못 해석하거나, 문서 근거와 다른 답변을 하거나, 금지된 동작을 수행한 응답 |

이번 평가는 개선 프롬프트가 기존 문제를 얼마나 완화했는지 확인하는 데 목적이 있습니다.

In [63]:
improved_manual_eval_df = improved_all_results_df[
    [
        "model",
        "test_type",
        "case_id",
        "source_type",
        "category",
        "answer"
    ]
].copy()

improved_manual_eval_df["manual_result"] = ""
improved_manual_eval_df["memo"] = ""

improved_manual_eval_df

,model,test_type,case_id,source_type,category,answer,manual_result,memo
0,gemma3:4b,Improved QA,IQA-01,existing,금액 단위 보존,"본 사업의 예산은 11,270,000,000원입니다. 부가가치세는 포함되지 않은 금액입니다.",,
1,gemma3:4b,Improved QA,IQA-02,existing,항목 누락 방지,구축 대상 시스템은 다음과 같습니다.\n1. 포털 시스템\n2. 학사 시스템\n3. 수강신청 시스템\n4. 전자결재 시스템\n5. 규정관리 시스템\n6. 검색 시스템\n7. 챗봇 시스템\n8. 모바일 시스템\n9. 통계 시스템,,
2,gemma3:4b,Improved QA,IQA-03,existing,가격점수 가드레일,"현재 문서에는 예정가격, 입찰가격, 평점산식의 구체적인 입력값이 모두 제시되어 있지 않아 가격점수 계산이 불가능합니다.",,
3,gemma3:4b,Improved QA,IQA-04,new,가격점수 산식 일부 부족,"가격평가 배점은 20점이고, 입찰가격은 95,000,000원입니다. 예정가격과 가격점수 평점산식, 가격 평가 배점 정보가 문서에 제시되어 있지 않아 가격점수를 계산할 수 없습니다.",,
4,gemma3:4b,Improved QA,IQA-05,new,기간 원문 보존,"사업기간은 계약체결일로부터 240일 이내입니다. 단, 발주기관의 사정에 따라 일정이 조정될 수 있습니다.",,
5,gemma3:4b,Improved QA,IQA-06,new,문서 근거 부족,제안요청서의 과업내용서에 해당 내용을 확인해 주시기 바랍니다.,,
6,qwen2.5:3b,Improved QA,IQA-01,existing,금액 단위 보존,"이 사업의 예산은 11,270,000,000원(십억 이천칠백만 원)입니다.",,
7,qwen2.5:3b,Improved QA,IQA-02,existing,항목 누락 방지,구축 대상 시스템 목록은 다음과 같습니다.\n\n1. 포털 시스템\n2. 학사 시스템\n3. 수강신청 시스템\n4. 전자결재 시스템\n5. 규정관리 시스템\n6. 검색 시스템\n7. 챗봇 시스템\n8. 모바일 시스템\n9. 통계 시스템,,
8,qwen2.5:3b,Improved QA,IQA-03,existing,가격점수 가드레일,"[검색된 문서 내용]에 있는 정보로는 가격점수를 계산할 수 없습니다. 다음 정보가 추가로 제공되면 가격점수 계산을 도와드릴 수 있습니다:\n\n1. 입찰가격\n2. 예정가격 또는 기준가격\n3. 가격평가 평점산식\n4. 가격 평가 배점\n\n이러한 정보가 제공된다면, 이를 바탕으로 가격점수를 계산할 수 있습니다.",,
9,qwen2.5:3b,Improved QA,IQA-04,new,가격점수 산식 일부 부족,"입찰가격이 95,000,000원으로, 예정가격과 가격점수 평점산식은 문서에 제시되어 있지 않아 가격점수를 계산할 수 없습니다. 이러한 정보는 추가로 제공해 주시면 감사하겠습니다.",,


In [64]:
improved_eval_map = {
    # =========================
    # Improved QA - gemma3:4b
    # =========================
    ("gemma3:4b", "IQA-01"): ("실패", "부가가치세 포함 금액인데 포함되지 않은 금액이라고 반대로 답변함"),
    ("gemma3:4b", "IQA-02"): ("통과", "9개 구축 대상 시스템을 모두 누락 없이 제시함"),
    ("gemma3:4b", "IQA-03"): ("통과", "가격점수 계산을 수행하지 않고 예정가격, 입찰가격, 평점산식 부족을 명확히 언급함"),
    ("gemma3:4b", "IQA-04"): ("부분 통과", "계산하지 않은 점은 적절하나 문서에 있는 가격 평가 배점까지 없다고 잘못 언급함"),
    ("gemma3:4b", "IQA-05"): ("통과", "사업기간을 원문 의미에 맞게 제시함"),
    ("gemma3:4b", "IQA-06"): ("부분 통과", "추측하지 않은 점은 적절하나 문서 근거 부족을 직접적으로 설명하지 않고 과업내용서 확인으로만 안내함"),

    # =========================
    # Improved QA - qwen2.5:3b
    # =========================
    ("qwen2.5:3b", "IQA-01"): ("부분 통과", "원문 금액은 포함했으나 한글 금액 병기를 추가했고 변환값도 부정확함"),
    ("qwen2.5:3b", "IQA-02"): ("통과", "9개 구축 대상 시스템을 모두 누락 없이 제시함"),
    ("qwen2.5:3b", "IQA-03"): ("부분 통과", "부족 정보는 잘 짚었으나 [검색된 문서 내용] 같은 내부 표현을 노출함"),
    ("qwen2.5:3b", "IQA-04"): ("통과", "예정가격과 가격점수 평점산식 부족으로 계산 불가를 적절히 안내함"),
    ("qwen2.5:3b", "IQA-05"): ("실패", "계약체결일로부터 240일 이내가 명시되어 있는데 특정 기간이 없다고 잘못 답변함"),
    ("qwen2.5:3b", "IQA-06"): ("통과", "문서에 유지보수 인력 정보가 없음을 명확히 설명하고 추측하지 않음"),

    # =========================
    # Improved QA - exaone3.5:2.4b
    # =========================
    ("exaone3.5:2.4b", "IQA-01"): ("통과", "예산과 부가가치세 포함 여부를 정확히 답변함"),
    ("exaone3.5:2.4b", "IQA-02"): ("통과", "9개 구축 대상 시스템을 모두 누락 없이 제시함"),
    ("exaone3.5:2.4b", "IQA-03"): ("통과", "가격점수 계산에 필요한 입찰가격, 예정가격, 평점산식, 배점을 모두 요구함"),
    ("exaone3.5:2.4b", "IQA-04"): ("부분 통과", "최종적으로 계산 불가 판단은 맞지만 계산해 드리겠다는 시작 문장이 혼동을 줄 수 있음"),
    ("exaone3.5:2.4b", "IQA-05"): ("통과", "사업기간을 문서 근거에 맞게 답변함"),
    ("exaone3.5:2.4b", "IQA-06"): ("통과", "문서에 유지보수 인력 정보가 없음을 명확히 설명하고 추측하지 않음"),

    # =========================
    # Improved User-Facing - gemma3:4b
    # =========================
    ("gemma3:4b", "IUF-01"): ("통과", "입찰 적합도 판단에 필요한 회사 정보, 유사 실적, 보유 인력, 기술 역량을 요구함"),
    ("gemma3:4b", "IUF-02"): ("통과", "제안서 작성에 필요한 대상 RFP, 회사 소개, 강점, 실적, 수행 전략을 요구함"),
    ("gemma3:4b", "IUF-03"): ("통과", "모호한 요약 요청에 대해 대상 문서와 요약 범위를 요구함"),
    ("gemma3:4b", "IUF-04"): ("부분 통과", "가격점수 계산에 필요한 정보는 요구했으나 예시에서 임의 평가 구조를 제시함"),
    ("gemma3:4b", "IUF-05"): ("부분 통과", "비교 기준을 요구한 점은 좋지만 비교 대상 문서보다 귀사 사업 중심으로 흐름"),
    ("gemma3:4b", "IUF-06"): ("부분 통과", "회사 강점과 유사 실적은 요구했으나 대상 RFP와 수행 전략 정보 요구가 다소 약함"),
    ("gemma3:4b", "IUF-07"): ("부분 통과", "대상 문서와 보고서 주요 항목을 요구했으나 예시 질문 제시는 부족함"),

    # =========================
    # Improved User-Facing - qwen2.5:3b
    # =========================
    ("qwen2.5:3b", "IUF-01"): ("부분 통과", "필요 정보는 일부 짚었으나 판정 필요합니다라는 내부 라벨성 표현이 남음"),
    ("qwen2.5:3b", "IUF-02"): ("부분 통과", "대상 RFP와 회사 실적을 요구했으나 답변이 짧고 수행 전략 요구가 약함"),
    ("qwen2.5:3b", "IUF-03"): ("실패", "모호한 요약 요청인데 회사 주요 사업 분야를 요구하여 질문 유형을 잘못 판단함"),
    ("qwen2.5:3b", "IUF-04"): ("부분 통과", "입찰가격과 예정가격은 요구했으나 판정 필요합니다 표현이 있고 평점산식 요구가 빠짐"),
    ("qwen2.5:3b", "IUF-05"): ("부분 통과", "비교에 필요한 추가 정보를 요구했으나 비교 대상 문서와 비교 기준 요구가 구체적이지 않음"),
    ("qwen2.5:3b", "IUF-06"): ("실패", "정보부족 안내가 아니라 제안 전략 작성 방향으로 답변을 진행함"),
    ("qwen2.5:3b", "IUF-07"): ("실패", "보고서 요약 요청인데 회사 사업 분야와 입찰 자격을 물어 질문 유형을 잘못 판단함"),

    # =========================
    # Improved User-Facing - exaone3.5:2.4b
    # =========================
    ("exaone3.5:2.4b", "IUF-01"): ("통과", "입찰 적합도 판단에 필요한 회사 정보, 유사 실적, 인력, 기술 역량을 요구함"),
    ("exaone3.5:2.4b", "IUF-02"): ("부분 통과", "제안서 작성 요소를 잘 설명했지만 정보부족 안내보다 일반 작성 가이드로 흐름"),
    ("exaone3.5:2.4b", "IUF-03"): ("실패", "모호한 요약 요청인데 RFP 영어 설명과 회사 기술 역량 요구로 흐름"),
    ("exaone3.5:2.4b", "IUF-04"): ("부분 통과", "가격점수 계산에 필요한 정보를 요구했지만 평점산식 표현이 직접적이지 않고 답변이 장황함"),
    ("exaone3.5:2.4b", "IUF-05"): ("부분 통과", "비교 대상 사업과 세부 사항을 요구했으나 예시가 회사 프로젝트 사례 중심으로 흐름"),
    ("exaone3.5:2.4b", "IUF-06"): ("부분 통과", "회사 강점과 유사 프로젝트 경험을 요구했으나 대상 RFP 정보 요구가 약함"),
    ("exaone3.5:2.4b", "IUF-07"): ("부분 통과", "대상 문서와 핵심 요소를 요구했으나 비교 대상 문서까지 요구해 질문 의도와 다소 어긋남"),

    # =========================
    # Improved Rewrite - gemma3:4b
    # =========================
    ("gemma3:4b", "IRW-01"): ("부분 통과", "비교 대상은 포함했으나 귀사 경쟁력 평가처럼 원 질문보다 범위가 확장됨"),
    ("gemma3:4b", "IRW-02"): ("실패", "그냥 알아서 해줘는 정보부족인데 재작성 가능으로 오판함"),
    ("gemma3:4b", "IRW-03"): ("통과", "사업명을 포함하여 수행 전략 질문으로 재작성함"),
    ("gemma3:4b", "IRW-04"): ("실패", "비교 대상이 부족한데 A 사업과 B 사업이라는 임의 재작성 질문을 생성함"),
    ("gemma3:4b", "IRW-05"): ("실패", "산식이 없다고 했는데 재작성 가능으로 오판하고 재작성 질문을 생성함"),
    ("gemma3:4b", "IRW-06"): ("통과", "고려대학교 사업명, 사업 목적, 주요 요구사항 중심의 제안서 초안 질문으로 재작성함"),
    ("gemma3:4b", "IRW-07"): ("통과", "공공기관 정보시스템 구축 경험과 Java/Spring 인력 정보를 포함하여 적합도 질문으로 재작성함"),

    # =========================
    # Improved Rewrite - qwen2.5:3b
    # =========================
    ("qwen2.5:3b", "IRW-01"): ("부분 통과", "비교 대상은 포함했으나 판단이 재작성 가능과 추가 확인 필요로 혼재됨"),
    ("qwen2.5:3b", "IRW-02"): ("부분 통과", "재작성 질문을 만들지 않은 점은 좋지만 판단 형식이 빠지고 필요한 정보가 다소 일반적임"),
    ("qwen2.5:3b", "IRW-03"): ("부분 통과", "재작성 질문은 만들었지만 판단 혼재와 내부 입력 구조 노출 문제가 있음"),
    ("qwen2.5:3b", "IRW-04"): ("통과", "비교 대상 부족을 인식하고 추가 정보를 요구함"),
    ("qwen2.5:3b", "IRW-05"): ("부분 통과", "추가 확인 필요 판단은 맞지만 재작성 질문을 생성했고 예정가격 요구가 빠짐"),
    ("qwen2.5:3b", "IRW-06"): ("실패", "재작성 가능 케이스인데 추가 확인 필요로 판단함"),
    ("qwen2.5:3b", "IRW-07"): ("부분 통과", "재작성 가능 판단은 맞지만 외국어 문자 혼입과 추가 정보 요구가 남음"),

    # =========================
    # Improved Rewrite - exaone3.5:2.4b
    # =========================
    ("exaone3.5:2.4b", "IRW-01"): ("부분 통과", "재작성 가능 판단은 맞지만 비교 대상인 경희대 사업명이 명확히 반영되지 않음"),
    ("exaone3.5:2.4b", "IRW-02"): ("통과", "정보부족 케이스를 재작성 불가능으로 판단하고 추가 확인을 요구함"),
    ("exaone3.5:2.4b", "IRW-03"): ("통과", "사업 기준 수행 전략 질문으로 재작성함"),
    ("exaone3.5:2.4b", "IRW-04"): ("통과", "비교 대상 부족을 재작성 불가능으로 판단하고 필요한 정보를 요구함"),
    ("exaone3.5:2.4b", "IRW-05"): ("부분 통과", "재작성 불가능 판단은 맞지만 가격점수 계산에 필요한 정보가 RFP 가격평가 맥락과 다소 어긋남"),
    ("exaone3.5:2.4b", "IRW-06"): ("부분 통과", "재작성 가능 판단은 맞지만 RFP 영어 설명과 과도한 항목 확장이 있음"),
    ("exaone3.5:2.4b", "IRW-07"): ("부분 통과", "재작성 가능 판단은 맞지만 RFP 영어 설명이 포함되고 추가 정보 요구가 다소 장황함"),
}

def apply_improved_manual_eval(row):
    key = (row["model"], row["case_id"])
    return pd.Series(improved_eval_map.get(key, ("", "")))

improved_manual_eval_df[["manual_result", "memo"]] = improved_manual_eval_df.apply(
    apply_improved_manual_eval,
    axis=1
)

improved_manual_eval_df

,model,test_type,case_id,source_type,category,answer,manual_result,memo
0,gemma3:4b,Improved QA,IQA-01,existing,금액 단위 보존,"본 사업의 예산은 11,270,000,000원입니다. 부가가치세는 포함되지 않은 금액입니다.",실패,부가가치세 포함 금액인데 포함되지 않은 금액이라고 반대로 답변함
1,gemma3:4b,Improved QA,IQA-02,existing,항목 누락 방지,구축 대상 시스템은 다음과 같습니다.\n1. 포털 시스템\n2. 학사 시스템\n3. 수강신청 시스템\n4. 전자결재 시스템\n5. 규정관리 시스템\n6. 검색 시스템\n7. 챗봇 시스템\n8. 모바일 시스템\n9. 통계 시스템,통과,9개 구축 대상 시스템을 모두 누락 없이 제시함
2,gemma3:4b,Improved QA,IQA-03,existing,가격점수 가드레일,"현재 문서에는 예정가격, 입찰가격, 평점산식의 구체적인 입력값이 모두 제시되어 있지 않아 가격점수 계산이 불가능합니다.",통과,"가격점수 계산을 수행하지 않고 예정가격, 입찰가격, 평점산식 부족을 명확히 언급함"
3,gemma3:4b,Improved QA,IQA-04,new,가격점수 산식 일부 부족,"가격평가 배점은 20점이고, 입찰가격은 95,000,000원입니다. 예정가격과 가격점수 평점산식, 가격 평가 배점 정보가 문서에 제시되어 있지 않아 가격점수를 계산할 수 없습니다.",부분 통과,계산하지 않은 점은 적절하나 문서에 있는 가격 평가 배점까지 없다고 잘못 언급함
4,gemma3:4b,Improved QA,IQA-05,new,기간 원문 보존,"사업기간은 계약체결일로부터 240일 이내입니다. 단, 발주기관의 사정에 따라 일정이 조정될 수 있습니다.",통과,사업기간을 원문 의미에 맞게 제시함
5,gemma3:4b,Improved QA,IQA-06,new,문서 근거 부족,제안요청서의 과업내용서에 해당 내용을 확인해 주시기 바랍니다.,부분 통과,추측하지 않은 점은 적절하나 문서 근거 부족을 직접적으로 설명하지 않고 과업내용서 확인으로만 안내함
6,qwen2.5:3b,Improved QA,IQA-01,existing,금액 단위 보존,"이 사업의 예산은 11,270,000,000원(십억 이천칠백만 원)입니다.",부분 통과,원문 금액은 포함했으나 한글 금액 병기를 추가했고 변환값도 부정확함
7,qwen2.5:3b,Improved QA,IQA-02,existing,항목 누락 방지,구축 대상 시스템 목록은 다음과 같습니다.\n\n1. 포털 시스템\n2. 학사 시스템\n3. 수강신청 시스템\n4. 전자결재 시스템\n5. 규정관리 시스템\n6. 검색 시스템\n7. 챗봇 시스템\n8. 모바일 시스템\n9. 통계 시스템,통과,9개 구축 대상 시스템을 모두 누락 없이 제시함
8,qwen2.5:3b,Improved QA,IQA-03,existing,가격점수 가드레일,"[검색된 문서 내용]에 있는 정보로는 가격점수를 계산할 수 없습니다. 다음 정보가 추가로 제공되면 가격점수 계산을 도와드릴 수 있습니다:\n\n1. 입찰가격\n2. 예정가격 또는 기준가격\n3. 가격평가 평점산식\n4. 가격 평가 배점\n\n이러한 정보가 제공된다면, 이를 바탕으로 가격점수를 계산할 수 있습니다.",부분 통과,부족 정보는 잘 짚었으나 [검색된 문서 내용] 같은 내부 표현을 노출함
9,qwen2.5:3b,Improved QA,IQA-04,new,가격점수 산식 일부 부족,"입찰가격이 95,000,000원으로, 예정가격과 가격점수 평점산식은 문서에 제시되어 있지 않아 가격점수를 계산할 수 없습니다. 이러한 정보는 추가로 제공해 주시면 감사하겠습니다.",통과,예정가격과 가격점수 평점산식 부족으로 계산 불가를 적절히 안내함


In [65]:
score_map = {
    "통과": 1.0,
    "부분 통과": 0.5,
    "실패": 0.0
}

improved_manual_eval_df["score"] = improved_manual_eval_df["manual_result"].map(score_map)

improved_manual_eval_df[
    [
        "model",
        "test_type",
        "case_id",
        "source_type",
        "category",
        "manual_result",
        "score",
        "memo"
    ]
]

,model,test_type,case_id,source_type,category,manual_result,score,memo
0,gemma3:4b,Improved QA,IQA-01,existing,금액 단위 보존,실패,0.0,부가가치세 포함 금액인데 포함되지 않은 금액이라고 반대로 답변함
1,gemma3:4b,Improved QA,IQA-02,existing,항목 누락 방지,통과,1.0,9개 구축 대상 시스템을 모두 누락 없이 제시함
2,gemma3:4b,Improved QA,IQA-03,existing,가격점수 가드레일,통과,1.0,"가격점수 계산을 수행하지 않고 예정가격, 입찰가격, 평점산식 부족을 명확히 언급함"
3,gemma3:4b,Improved QA,IQA-04,new,가격점수 산식 일부 부족,부분 통과,0.5,계산하지 않은 점은 적절하나 문서에 있는 가격 평가 배점까지 없다고 잘못 언급함
4,gemma3:4b,Improved QA,IQA-05,new,기간 원문 보존,통과,1.0,사업기간을 원문 의미에 맞게 제시함
5,gemma3:4b,Improved QA,IQA-06,new,문서 근거 부족,부분 통과,0.5,추측하지 않은 점은 적절하나 문서 근거 부족을 직접적으로 설명하지 않고 과업내용서 확인으로만 안내함
6,qwen2.5:3b,Improved QA,IQA-01,existing,금액 단위 보존,부분 통과,0.5,원문 금액은 포함했으나 한글 금액 병기를 추가했고 변환값도 부정확함
7,qwen2.5:3b,Improved QA,IQA-02,existing,항목 누락 방지,통과,1.0,9개 구축 대상 시스템을 모두 누락 없이 제시함
8,qwen2.5:3b,Improved QA,IQA-03,existing,가격점수 가드레일,부분 통과,0.5,부족 정보는 잘 짚었으나 [검색된 문서 내용] 같은 내부 표현을 노출함
9,qwen2.5:3b,Improved QA,IQA-04,new,가격점수 산식 일부 부족,통과,1.0,예정가격과 가격점수 평점산식 부족으로 계산 불가를 적절히 안내함


In [66]:
improved_type_score_summary_df = improved_manual_eval_df.groupby(
    ["model", "test_type"]
).agg(
    total_cases=("case_id", "count"),
    pass_count=("manual_result", lambda x: (x == "통과").sum()),
    partial_count=("manual_result", lambda x: (x == "부분 통과").sum()),
    fail_count=("manual_result", lambda x: (x == "실패").sum()),
    total_score=("score", "sum"),
    avg_score=("score", "mean")
).reset_index()

improved_type_score_summary_df.sort_values(["test_type", "avg_score"], ascending=[True, False])

,model,test_type,total_cases,pass_count,partial_count,fail_count,total_score,avg_score
0,exaone3.5:2.4b,Improved QA,6,5,1,0,5.5,0.916667
3,gemma3:4b,Improved QA,6,3,2,1,4.0,0.666667
6,qwen2.5:3b,Improved QA,6,3,2,1,4.0,0.666667
1,exaone3.5:2.4b,Improved Rewrite,7,3,4,0,5.0,0.714286
4,gemma3:4b,Improved Rewrite,7,3,1,3,3.5,0.500000
7,qwen2.5:3b,Improved Rewrite,7,1,5,1,3.5,0.500000
5,gemma3:4b,Improved User-Facing Clarification,7,3,4,0,5.0,0.714286
2,exaone3.5:2.4b,Improved User-Facing Clarification,7,1,5,1,3.5,0.500000
8,qwen2.5:3b,Improved User-Facing Clarification,7,0,4,3,2.0,0.285714


In [67]:
improved_overall_score_summary_df = improved_manual_eval_df.groupby("model").agg(
    total_cases=("case_id", "count"),
    pass_count=("manual_result", lambda x: (x == "통과").sum()),
    partial_count=("manual_result", lambda x: (x == "부분 통과").sum()),
    fail_count=("manual_result", lambda x: (x == "실패").sum()),
    total_score=("score", "sum"),
    avg_score=("score", "mean")
).reset_index()

improved_overall_score_summary_df = improved_overall_score_summary_df.sort_values(
    "avg_score",
    ascending=False
)

improved_overall_score_summary_df

,model,total_cases,pass_count,partial_count,fail_count,total_score,avg_score
0,exaone3.5:2.4b,20,9,10,1,14.0,0.700
1,gemma3:4b,20,9,7,4,12.5,0.625
2,qwen2.5:3b,20,4,11,5,9.5,0.475


## 기능별 모델 적합성 정리

개선 프롬프트 적용 후 모델별 성능을 비교한 결과, 단일 모델이 모든 기능에서 가장 우수한 것은 아니었습니다. 기능별로 강점이 다르게 나타났습니다.

| 기능 | 가장 적합한 모델 | 근거 |
|---|---|---|
| 문서 기반 QA | `exaone3.5:2.4b` | 문서 근거 기반 답변, 금액 보존, 정보부족 판단에서 가장 안정적 |
| 사용자 응답형 정보부족 안내 | `gemma3:4b` | 사용자에게 보여줄 자연스러운 한국어 안내 품질이 가장 좋음 |
| 질문 재작성 및 정보부족 판단 | `exaone3.5:2.4b` | 비교 대상 부족, 산식 부족, 회사 정보 부족 상황을 비교적 안정적으로 판단 |
| 속도 중심 대안 | `qwen2.5:3b` | 응답 속도는 빠르지만 사용자 응답 품질과 형식 안정성은 낮음 |

따라서 실제 시스템에서 기능별 모델 분기가 가능하다면, 문서 기반 QA와 질문 재작성에는 `exaone3.5:2.4b`, 사용자 응답형 안내에는 `gemma3:4b`를 사용하는 방안을 고려할 수 있습니다.

다만 프로젝트 구조가 복잡해지는 것을 피하려면, 전체 평균 점수가 가장 높은 `exaone3.5:2.4b`를 개선 프롬프트 적용 후의 단일 후보로 우선 검토할 수 있습니다.

## 최종 모델 선정 방향

개선 프롬프트 적용 전에는 `gemma3:4b`가 사용자 응답 품질 측면에서 가장 안정적인 후보로 보였습니다. 그러나 개선 프롬프트 적용 후 기존 질문과 신규 질문을 함께 평가한 결과, 전체 평균 점수는 `exaone3.5:2.4b`가 가장 높게 나타났습니다.

`exaone3.5:2.4b`는 문서 기반 QA와 질문 재작성에서 강점을 보였습니다. 특히 문서에 근거가 부족한 경우 추측하지 않고 정보부족을 안내하는 능력이 비교적 안정적이었습니다. 또한 비교 대상 부족, 가격점수 산식 부족, 회사 정보 부족과 같은 상황에서 재작성 가능 여부를 더 신중하게 판단했습니다.

반면 `gemma3:4b`는 사용자 응답형 정보부족 안내에서 가장 자연스러운 답변을 생성했습니다. 입찰 적합도 판단, 제안서 작성, 요약 요청, 가격점수 계산 안내에서 사용자에게 필요한 정보를 비교적 부드럽게 요청했습니다. 그러나 문서 기반 QA에서 부가가치세 포함 여부를 반대로 답변하는 오류가 있었고, 질문 재작성에서는 정보부족 케이스를 재작성 가능으로 오판하는 문제가 반복되었습니다.

`qwen2.5:3b`는 응답 속도 측면에서는 장점이 있었지만, 사용자 응답형 안내와 질문 재작성에서 출력 형식 혼재, 내부 라벨성 표현, 질문 유형 오판, 외국어 문자 혼입 문제가 확인되었습니다. 따라서 최종 주력 모델보다는 속도 중심 대안 후보로 보는 것이 적절합니다.

최종적으로 개선 프롬프트 기준에서는 다음과 같이 정리할 수 있습니다.

| 모델 | 최종 판단 |
|---|---|
| `exaone3.5:2.4b` | 개선 프롬프트 적용 후 전체 성능 1위, 단일 후보로 우선 검토 |
| `gemma3:4b` | 사용자 응답형 안내 품질 우수, 사용자-facing 응답 후보 |
| `qwen2.5:3b` | 속도 중심 대안 후보 |

## RAG 연동 시 적용 기준

본 실험은 실제 Retriever와 완전히 연결되기 전, Generation 파트의 모델 후보와 프롬프트 안정성을 확인하기 위한 실험입니다. 이후 팀원이 구축한 Retriever와 Chroma DB가 연결되면, 검색된 문서 chunk를 context로 입력하여 동일한 기준으로 재검증해야 합니다.

RAG 연동 시 Generation 파트의 입력은 다음과 같이 구성하는 것이 적절합니다.

| 입력 항목 | 설명 |
|---|---|
| 사용자 질문 | 사용자가 입력한 원본 질문 |
| 검색된 문서 chunk | Retriever가 반환한 관련 문서 내용 |
| 문서명 | 답변 근거가 되는 RFP 문서명 |
| 페이지 또는 chunk id | 답변 근거 추적을 위한 위치 정보 |
| 질문 유형 | QA, 요약, 비교, 적합도 판단, 제안서 작성, 질문 재작성 등 |
| 추가 사용자 정보 | 회사 정보, 비교 대상, 산식, 가격 정보 등 |

RAG 연동 후에는 다음 항목을 다시 확인해야 합니다.

- 검색된 context에 답이 있을 때 정확히 답변하는가
- 검색된 context에 답이 없을 때 추측하지 않는가
- 금액, 기간, 배점, 비율, 산식 등을 원문 그대로 유지하는가
- 목록형 요구사항을 누락하지 않는가
- 가격점수 계산 시 입찰가격, 예정가격, 평점산식, 배점이 모두 있는 경우에만 계산하는가
- 문서 비교 시 비교 대상 문서를 혼동하지 않는가
- 사용자에게 내부 라벨이나 프롬프트 구조를 노출하지 않는가

현재 평가 결과만 기준으로 하면, 단일 모델을 사용할 경우 `exaone3.5:2.4b`를 우선 후보로 검토할 수 있습니다. 다만 사용자에게 직접 보여주는 안내 응답의 자연스러움은 `gemma3:4b`가 더 나았으므로, 실제 서비스 구조에서 기능별 모델 분기가 가능한지도 함께 고려할 필요가 있습니다.

## 최종 결론

본 실험에서는 공공 RFP 기반 RAG 시스템의 Generation 파트에 사용할 로컬 LLM 후보를 비교하고, 프롬프트 개선 전후의 응답 품질을 평가하였습니다. 모델 선별은 단순히 응답이 자연스러운지를 보는 것이 아니라, 실제 RFP 분석 서비스에서 필요한 기능을 안정적으로 수행할 수 있는지를 기준으로 진행하였습니다.

모델 선별 기준은 다음과 같이 설정하였습니다.

| 선별 기준             | 설명                                                                  |
| ----------------- | ------------------------------------------------------------------- |
| 문서 근거 기반 답변 정확성   | 문서에 있는 금액, 기간, 배점, 산식, 목록형 요구사항을 임의로 바꾸지 않고 답변하는지 확인                |
| 정보부족 판단 능력        | 가격점수 계산, 입찰 적합도 평가, 문서 비교처럼 추가 정보가 필요한 상황에서 추측하지 않고 부족 정보를 요구하는지 확인 |
| 질문 재작성 안정성        | 추가 답변이 충분할 때만 질문을 재작성하고, 정보가 부족한 경우에는 재작성하지 않는지 확인                  |
| 사용자 응답형 안내 품질     | 사용자가 이해하기 쉬운 한국어 존댓말로 필요한 추가 정보를 자연스럽게 안내하는지 확인                     |
| 출력 형식 안정성         | 내부 라벨, 프롬프트 구조, 외국어 혼입, 깨진 문자 등이 사용자에게 노출되지 않는지 확인                  |
| 응답 속도 및 로컬 실행 가능성 | 실제 로컬 환경에서 실행 가능한 크기인지, 응답 속도가 서비스 흐름에 사용 가능한 수준인지 확인               |
| RAG 연동 가능성        | 추후 검색된 chunk와 metadata를 context로 입력했을 때 문서 기반 답변에 활용할 수 있는지 확인      |

초기 실험에서는 `gemma3:4b`, `qwen2.5:3b`, `llama3.2:3b`, `exaone3.5:2.4b`, `phi3.5`를 대상으로 기본 QA, 정보부족 판단, 질문 재작성, 사용자 응답형 안내 테스트를 수행하였습니다. 이 과정에서 `llama3.2:3b`는 외국어 혼입과 출력 형식 불안정 문제가 반복되었고, `phi3.5`는 내부 라벨 반복, 깨진 문자, 부자연스러운 한국어 응답이 확인되어 최종 후보군에서 제외하였습니다.

이후 최종 후보군은 `gemma3:4b`, `qwen2.5:3b`, `exaone3.5:2.4b`로 압축하였습니다. 세 모델에는 동일한 개선 프롬프트와 동일한 테스트셋을 적용하여 비교하였습니다. 이는 모델별로 다른 프롬프트를 적용할 경우 성능 차이가 모델 자체의 차이인지, 프롬프트 차이인지 구분하기 어려워지기 때문입니다.

후처리 함수는 내부 라벨 제거, 외국어 문자 감지, 빈 응답 대체와 같은 표면적인 문제를 완화하는 데는 도움이 되었습니다. 그러나 문서 근거와 다른 답변 생성, 가격점수 계산 정보 누락, 비교 대상 부족 판단, 정보부족 상황 오판과 같은 내용적 오류는 후처리만으로 해결하기 어려웠습니다. 따라서 핵심 품질 개선은 후처리보다 프롬프트 설계와 RAG context 품질에 더 크게 의존한다고 판단하였습니다.

개선 프롬프트 적용 후에는 기존 질문과 신규 질문을 함께 사용하여 회귀 테스트와 일반화 테스트를 수행하였습니다. 그 결과 전체 평균 점수는 `exaone3.5:2.4b`가 가장 높게 나타났습니다. `exaone3.5:2.4b`는 문서 기반 QA와 질문 재작성에서 가장 안정적인 결과를 보였으며, 특히 문서 근거가 부족한 경우 추측하지 않고 추가 정보가 필요하다고 판단하는 능력이 상대적으로 우수했습니다.

`gemma3:4b`는 사용자 응답형 정보부족 안내에서 가장 자연스러운 한국어 응답을 생성했습니다. 입찰 적합도 판단, 제안서 작성, 요약 요청, 가격점수 계산 안내에서 사용자에게 필요한 정보를 부드럽게 요청하는 강점이 있었습니다. 다만 문서 기반 QA에서 부가가치세 포함 여부를 반대로 답변하거나, 질문 재작성에서 정보부족 상황을 재작성 가능으로 오판하는 문제가 확인되었습니다.

`qwen2.5:3b`는 응답 속도 측면에서는 장점이 있었지만, 사용자 응답형 안내와 질문 재작성에서 출력 형식 혼재, 내부 라벨성 표현, 질문 유형 오판, 일부 외국어 문자 혼입 문제가 남아 있었습니다. 따라서 최종 주력 모델보다는 속도 중심 대안 후보로 보는 것이 적절하다고 판단하였습니다.

최종적으로 본 실험의 모델 선별 결과는 다음과 같습니다.

| 구분                  | 모델                      | 선정 이유                                   |
| ------------------- | ----------------------- | --------------------------------------- |
| 개선 프롬프트 기준 단일 우선 후보 | `exaone3.5:2.4b`        | 전체 평균 점수가 가장 높고, 문서 기반 QA와 질문 재작성에서 안정적 |
| 사용자 응답형 안내 후보       | `gemma3:4b`             | 사용자에게 보여줄 한국어 안내 문장이 가장 자연스러움           |
| 속도 중심 대안 후보         | `qwen2.5:3b`            | 응답 속도는 빠르지만 품질과 형식 안정성 보완 필요            |
| 제외 후보               | `llama3.2:3b`, `phi3.5` | 외국어 혼입, 내부 라벨 노출, 출력 형식 불안정 문제가 반복됨     |

따라서 현재 Generation 단독 실험 기준으로는 `exaone3.5:2.4b`를 단일 우선 후보로 검토하는 것이 가장 적절합니다. 다만 사용자에게 직접 노출되는 안내 응답에서는 `gemma3:4b`의 품질이 더 자연스러웠으므로, 실제 서비스 구조에서 기능별 모델 분리가 가능하다면 `exaone3.5:2.4b`와 `gemma3:4b`를 역할별로 나누는 방안도 고려할 수 있습니다.

다만 본 실험은 실제 Retriever가 완전히 연결되기 전의 Generation 단독 실험입니다. 따라서 최종 모델 확정은 실제 RAG 검색 결과, 문서 chunk, 문서명, 사업명, 발주기관, 페이지 또는 section 정보, metadata가 포함된 context를 입력한 뒤 동일한 평가 기준으로 재검증한 후 결정하는 것이 적절합니다.

또한 현재 실험은 2.4B~4B급 경량 모델 중심으로 진행되었으므로, 추후 로컬 실행 가능 범위 안에서 더 큰 모델을 추가 테스트할 경우 모델 규모 확대가 문서 기반 QA, 정보부족 판단, 질문 재작성 안정성에 어떤 영향을 주는지도 함께 확인할 필요가 있습니다. 따라서 본 결론은 실제 RAG context 연결 및 추가 모델 검증 전까지의 1차 모델 선별 결과로 정리합니다.


---

In [3]:
from pathlib import Path

# 공유받은 Chroma DB가 들어 있어야 하는 경로
chroma_dir = Path("./data/chroma_db")

print("현재 작업 경로:", Path.cwd())
print("Chroma DB 경로 존재 여부:", chroma_dir.exists())

if chroma_dir.exists():
    print("\nChroma DB 내부 파일 목록:")
    for p in chroma_dir.rglob("*"):
        print(p)
else:
    print("\n./data/chroma_db 경로가 없습니다.")
    print("공유받은 chroma_db 폴더를 ./data/chroma_db 위치에 넣어야 합니다.")

현재 작업 경로: c:\Users\User\Desktop\[AI]스프린트미션\프로젝트 2번
Chroma DB 경로 존재 여부: True

Chroma DB 내부 파일 목록:
data\chroma_db\.ipynb_checkpoints
data\chroma_db\chroma.sqlite3
data\chroma_db\e170e134-d973-417b-a405-b18fb375a1d0
data\chroma_db\e170e134-d973-417b-a405-b18fb375a1d0\data_level0.bin
data\chroma_db\e170e134-d973-417b-a405-b18fb375a1d0\header.bin
data\chroma_db\e170e134-d973-417b-a405-b18fb375a1d0\index_metadata.pickle
data\chroma_db\e170e134-d973-417b-a405-b18fb375a1d0\length.bin
data\chroma_db\e170e134-d973-417b-a405-b18fb375a1d0\link_lists.bin


In [4]:
# 2단계: Vector DB 및 Retriever 로드

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

CHROMA_DIR = "./data/chroma_db"
COLLECTION_NAME = "rfp_docs"

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)

vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_model,
    persist_directory=CHROMA_DIR,
)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

print("Vector DB 로드 완료")
print("Retriever 생성 완료")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Vector DB 로드 완료
Retriever 생성 완료


In [5]:
# 3단계: Retriever 검색 결과 확인

query = "이 사업의 예산은 얼마인가요?"

docs = retriever.invoke(query)

print(f"검색 결과 수: {len(docs)}")

for i, doc in enumerate(docs, start=1):
    print("=" * 80)
    print(f"[검색 결과 {i}]")
    print("\nmetadata:")
    print(doc.metadata)
    print("\npage_content 앞부분:")
    print(doc.page_content[:1000])

검색 결과 수: 5
[검색 결과 1]

metadata:
{'재공고여부': False, '사업요약': '- 사업 개요: 통합정보시스템 고도화 용역, 사업기간 5개월 이내, 추정가격 140,000,000원 이내, 협상에 의한 계약\n- 사업추진 방안: 기존 시스템과의 연계를 고려한 신규 시스템 설계, 자료 표준화 및 체계화, 사용자 편의성 고려한 UI/UX 시스템 구현, 기존 시스템 및 인프라 활용한 운영환경 구성\n- 사업 범위: 기관생명윤리, 동물실험윤리, 국가연구개발사업 연구비 사용기준 준수, 연구계획 관리, 전자결재서식 연동, 기능개선(연구노트 관리)\n- 추진 체계: 추진조직 구성(사업추진 총괄, 사업추진 전담, 사업 수행팀)\n- 목표시스템 구성도: 사업 범위에 대한 시스템 구성도\n- 추진 일정: 사업 시작 후 5개월간의 일정\n- 공공 SW 사업 법제도 점검항목별 적용 여부: 공공 SW 사업 법제도에 따른 적용 여부\n- 주관부서 및 담당자: 입찰관련 문의(재무회계팀 담당자), 과업내용 및 제안서 관련 문의(총무전산팀 담당자)\n- 요구사항 구성: 기능 요구사항, 성능 요구사항 등 총 24개의 요구사항 분류', '사업명': '통합정보시스템 고도화 용역', '공고차수': 0.0, '입찰참여마감일': '2024-06-11 00:00:00', '입찰참여시작일': '2024-05-30 00:00:00', '공고번호': '<unknown>', 'file_name': '국가과학기술지식정보서비스_통합정보시스템 고도화 용역.hwp', '공개일자': '2024-05-30 00:00:00', 'table_type': '', '사업유형': '고도화', '사업금액': 140000000.0, 'section_id': 'S44', '발주기관': '국가과학기술지식정보서비스', 'source_format': 'hwp', 'block_type': 'text', 'block_id': 'S44-B01', 'doc_id': 'D016', 'linked_doc_id': '', 'hea

In [6]:
# 4단계: RAG context builder 작성

def build_rag_context(docs, max_content_chars=1200):
    """
    retriever가 반환한 Document 리스트를
    LLM 프롬프트에 넣기 좋은 context 문자열로 변환합니다.
    
    page_content만 사용하지 않고,
    사업명, 발주기관, 사업금액, 사업요약, 문서 위치 정보를 함께 포함합니다.
    """
    context_blocks = []

    for i, doc in enumerate(docs, start=1):
        meta = doc.metadata or {}

        block = f"""
[검색 결과 {i}]
문서명: {meta.get("file_name", "")}
문서 ID: {meta.get("doc_id", "")}
사업명: {meta.get("사업명", "")}
발주기관: {meta.get("발주기관", "")}
사업유형: {meta.get("사업유형", "")}
사업금액: {meta.get("사업금액", "")}
공개일자: {meta.get("공개일자", "")}
입찰참여시작일: {meta.get("입찰참여시작일", "")}
입찰참여마감일: {meta.get("입찰참여마감일", "")}
문서 위치: {meta.get("header_path", "")}
section_id: {meta.get("section_id", "")}
block_id: {meta.get("block_id", "")}

사업요약:
{meta.get("사업요약", "")}

본문:
{doc.page_content[:max_content_chars]}
"""
        context_blocks.append(block.strip())

    return "\n\n".join(context_blocks)

In [7]:
context = build_rag_context(docs)

print(context[:4000])

[검색 결과 1]
문서명: 국가과학기술지식정보서비스_통합정보시스템 고도화 용역.hwp
문서 ID: D016
사업명: 통합정보시스템 고도화 용역
발주기관: 국가과학기술지식정보서비스
사업유형: 고도화
사업금액: 140000000.0
공개일자: 2024-05-30 00:00:00
입찰참여시작일: 2024-05-30 00:00:00
입찰참여마감일: 2024-06-11 00:00:00
문서 위치: Ⅳ. 제안서 평가 및 사업자 선정 41 > 1. 본인은 사업과 관련된 업무가 보안관련 국가기밀 사항임을 인정하고 제반 보안관계규정 및 지침을 성실히 수행한다.
section_id: S44
block_id: S44-B01

사업요약:
- 사업 개요: 통합정보시스템 고도화 용역, 사업기간 5개월 이내, 추정가격 140,000,000원 이내, 협상에 의한 계약
- 사업추진 방안: 기존 시스템과의 연계를 고려한 신규 시스템 설계, 자료 표준화 및 체계화, 사용자 편의성 고려한 UI/UX 시스템 구현, 기존 시스템 및 인프라 활용한 운영환경 구성
- 사업 범위: 기관생명윤리, 동물실험윤리, 국가연구개발사업 연구비 사용기준 준수, 연구계획 관리, 전자결재서식 연동, 기능개선(연구노트 관리)
- 추진 체계: 추진조직 구성(사업추진 총괄, 사업추진 전담, 사업 수행팀)
- 목표시스템 구성도: 사업 범위에 대한 시스템 구성도
- 추진 일정: 사업 시작 후 5개월간의 일정
- 공공 SW 사업 법제도 점검항목별 적용 여부: 공공 SW 사업 법제도에 따른 적용 여부
- 주관부서 및 담당자: 입찰관련 문의(재무회계팀 담당자), 과업내용 및 제안서 관련 문의(총무전산팀 담당자)
- 요구사항 구성: 기능 요구사항, 성능 요구사항 등 총 24개의 요구사항 분류

본문:
[사업명] 통합정보시스템 고도화 용역
[발주기관] 국가과학기술지식정보서비스
[요약] - 사업 개요: 통합정보시스템 고도화 용역, 사업기간 5개월 이내, 추정가격 140,000,000원 이내, 협상에 의한 계약
- 사업추진 방안: 기존 시스

In [8]:
def build_rag_context(docs, max_content_chars=800):
    """
    retriever 결과를 Generation 프롬프트용 context로 변환합니다.

    핵심 원칙:
    1. page_content만 사용하지 않고 metadata를 함께 포함합니다.
    2. 사업명, 발주기관, 사업금액, 사업요약을 상단에 배치합니다.
    3. 검색된 본문 chunk가 질문과 직접 관련 없을 수 있으므로 문서 위치를 함께 제공합니다.
    4. context 길이 증가를 막기 위해 본문은 일부만 사용합니다.
    """
    context_blocks = []

    for i, doc in enumerate(docs, start=1):
        meta = doc.metadata or {}

        block = f"""
[검색 결과 {i}]
문서명: {meta.get("file_name", "")}
문서 ID: {meta.get("doc_id", "")}
사업명: {meta.get("사업명", "")}
발주기관: {meta.get("발주기관", "")}
사업유형: {meta.get("사업유형", "")}
사업금액: {meta.get("사업금액", "")}
공개일자: {meta.get("공개일자", "")}
입찰참여기간: {meta.get("입찰참여시작일", "")} ~ {meta.get("입찰참여마감일", "")}
문서 위치: {meta.get("header_path", "")}
section_id: {meta.get("section_id", "")}
block_id: {meta.get("block_id", "")}

사업요약:
{meta.get("사업요약", "")}

검색된 본문 일부:
{doc.page_content[:max_content_chars]}
"""
        context_blocks.append(block.strip())

    return "\n\n".join(context_blocks)

In [9]:
context = build_rag_context(docs)
print(context[:4000])

[검색 결과 1]
문서명: 국가과학기술지식정보서비스_통합정보시스템 고도화 용역.hwp
문서 ID: D016
사업명: 통합정보시스템 고도화 용역
발주기관: 국가과학기술지식정보서비스
사업유형: 고도화
사업금액: 140000000.0
공개일자: 2024-05-30 00:00:00
입찰참여기간: 2024-05-30 00:00:00 ~ 2024-06-11 00:00:00
문서 위치: Ⅳ. 제안서 평가 및 사업자 선정 41 > 1. 본인은 사업과 관련된 업무가 보안관련 국가기밀 사항임을 인정하고 제반 보안관계규정 및 지침을 성실히 수행한다.
section_id: S44
block_id: S44-B01

사업요약:
- 사업 개요: 통합정보시스템 고도화 용역, 사업기간 5개월 이내, 추정가격 140,000,000원 이내, 협상에 의한 계약
- 사업추진 방안: 기존 시스템과의 연계를 고려한 신규 시스템 설계, 자료 표준화 및 체계화, 사용자 편의성 고려한 UI/UX 시스템 구현, 기존 시스템 및 인프라 활용한 운영환경 구성
- 사업 범위: 기관생명윤리, 동물실험윤리, 국가연구개발사업 연구비 사용기준 준수, 연구계획 관리, 전자결재서식 연동, 기능개선(연구노트 관리)
- 추진 체계: 추진조직 구성(사업추진 총괄, 사업추진 전담, 사업 수행팀)
- 목표시스템 구성도: 사업 범위에 대한 시스템 구성도
- 추진 일정: 사업 시작 후 5개월간의 일정
- 공공 SW 사업 법제도 점검항목별 적용 여부: 공공 SW 사업 법제도에 따른 적용 여부
- 주관부서 및 담당자: 입찰관련 문의(재무회계팀 담당자), 과업내용 및 제안서 관련 문의(총무전산팀 담당자)
- 요구사항 구성: 기능 요구사항, 성능 요구사항 등 총 24개의 요구사항 분류

검색된 본문 일부:
[사업명] 통합정보시스템 고도화 용역
[발주기관] 국가과학기술지식정보서비스
[요약] - 사업 개요: 통합정보시스템 고도화 용역, 사업기간 5개월 이내, 추정가격 140,000,000원 이내, 협상에 의한 계약
- 사업추진 방안: 기존 시스템

In [10]:
import requests
import time

def ask_ollama(prompt, model, temperature=0.0, timeout=180):
    url = "http://localhost:11434/api/generate"
    
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": temperature
        }
    }
    
    start_time = time.time()
    
    try:
        response = requests.post(url, json=payload, timeout=timeout)
        elapsed_time = time.time() - start_time
        
        if response.status_code == 200:
            result = response.json()
            return {
                "success": True,
                "model": model,
                "answer": result.get("response", "").strip(),
                "elapsed_time": elapsed_time,
                "error": None
            }
        else:
            return {
                "success": False,
                "model": model,
                "answer": "",
                "elapsed_time": elapsed_time,
                "error": f"HTTP {response.status_code}: {response.text}"
            }
    except Exception as e:
        elapsed_time = time.time() - start_time
        return {
            "success": False,
            "model": model,
            "answer": "",
            "elapsed_time": elapsed_time,
            "error": str(e)
        }

In [11]:
def build_rag_qa_prompt(question, context):
    return f"""
당신은 공공 RFP 문서를 분석하는 AI입니다.

아래 [검색 결과 context]만 근거로 답변하세요.

규칙:
1. 문서에 있는 정보만 사용하세요.
2. 문서에 없는 내용은 추측하지 마세요.
3. 금액은 원문 숫자를 임의로 억/만원 단위로 바꾸지 마세요.
4. 사업금액 metadata와 사업요약이 있으면 함께 참고하세요.
5. 검색된 본문이 질문과 직접 관련 없더라도, metadata에 명시된 정보는 사용할 수 있습니다.
6. 한국어 존댓말로 간결하게 답변하세요.
7. 답변 마지막에 사용한 근거의 문서명 또는 사업명을 간단히 적어주세요.

[검색 결과 context]
{context}

[사용자 질문]
{question}

[답변]
""".strip()

In [12]:
query = "이 사업의 예산은 얼마인가요?"

docs = retriever.invoke(query)
context = build_rag_context(docs)
prompt = build_rag_qa_prompt(query, context)

result = ask_ollama(
    prompt=prompt,
    model="exaone3.5:2.4b",
    temperature=0.0,
    timeout=180
)

print("success:", result["success"])
print("elapsed_time:", round(result["elapsed_time"], 2))
print("error:", result["error"])
print("\nanswer:")
print(result["answer"])

success: True
elapsed_time: 6.51
error: None

answer:
이 사업의 예산은 **1억 4천만 원**입니다 (140,000,000 원).


### RAG 답변 생성 1차 테스트 결과

공유받은 Chroma DB를 로드하고 retriever 검색 결과를 Generation용 context로 변환한 뒤, `exaone3.5:2.4b` 모델에 입력하여 RAG 답변 생성을 테스트하였습니다.

테스트 질문은 “이 사업의 예산은 얼마인가요?”였으며, retriever는 `통합정보시스템 고도화 용역` 문서를 검색하였습니다. 검색 결과의 본문 chunk는 보안서약서나 계약내역 양식 등 질문과 직접 관련이 약한 내용도 포함하고 있었지만, metadata의 `사업요약`과 `사업금액`에는 예산 정보가 포함되어 있었습니다.

모델은 `140,000,000원`이라는 예산 정보를 정확히 답변하였으나, 프롬프트에서 원문 금액 표기를 유지하도록 지시했음에도 `1억 4천만 원`이라는 환산 표현을 함께 사용하였습니다. 또한 답변 마지막에 근거 문서명 또는 사업명을 표시하라는 지시를 완전히 따르지는 않았습니다.

따라서 본 테스트는 RAG 연결과 예산 정보 추출은 성공했지만, 금액 표기 형식과 근거 표시 형식은 추가 프롬프트 보완이 필요한 부분 통과 사례로 판단하였습니다.

In [13]:
def build_rag_qa_prompt(question, context):
    return f"""
당신은 공공 RFP 문서를 분석하는 AI입니다.

아래 [검색 결과 context]만 근거로 답변하세요.

규칙:
1. 문서에 있는 정보만 사용하세요.
2. 문서에 없는 내용은 추측하지 마세요.
3. 금액은 반드시 context에 나온 원문 숫자 표기 그대로 답변하세요.
4. 금액을 억 원, 만 원, 1억 4천만 원처럼 환산해서 표현하지 마세요.
5. 사업금액 metadata와 사업요약이 있으면 함께 참고하세요.
6. 검색된 본문이 질문과 직접 관련 없더라도, metadata에 명시된 정보는 사용할 수 있습니다.
7. 한국어 존댓말로 간결하게 답변하세요.
8. 답변 마지막에는 반드시 "근거: 사업명 / 문서명" 형식으로 근거를 표시하세요.

[검색 결과 context]
{context}

[사용자 질문]
{question}

[답변 형식]
답변: ...
근거: 사업명 / 문서명

[답변]
""".strip()

In [14]:
prompt = build_rag_qa_prompt(query, context)

result = ask_ollama(
    prompt=prompt,
    model="exaone3.5:2.4b",
    temperature=0.0,
    timeout=180
)

print("success:", result["success"])
print("elapsed_time:", round(result["elapsed_time"], 2))
print("error:", result["error"])
print("\nanswer:")
print(result["answer"])

success: True
elapsed_time: 5.11
error: None

answer:
답변: 이 사업의 예산은 **1억 4천만 원**입니다.

근거: **문서명**: 국가과학기술지식정보서비스_통합정보시스템 고도화 용역  
**문서 ID**: D016  
**공개일자**: 2024년 5월 30일  
**사업금액**: 140,000,000원  

이 정보는 제공된 문서에서 직접 추출되었습니다. 사업금액이 1억 4천만 원으로 명시되어 있습니다.


In [15]:
def format_amount(value):
    """
    metadata의 사업금액이 140000000.0처럼 float로 들어오는 경우
    LLM이 읽기 좋은 원문형 숫자 표기로 변환합니다.
    """
    if value is None or value == "":
        return ""

    try:
        return f"{int(float(value)):,}원"
    except:
        return str(value)

In [16]:
def build_rag_context(docs, max_content_chars=800):
    """
    retriever 결과를 Generation 프롬프트용 context로 변환합니다.
    금액은 LLM이 환산하지 않도록 140,000,000원 형태로 정리합니다.
    """
    context_blocks = []

    for i, doc in enumerate(docs, start=1):
        meta = doc.metadata or {}

        raw_amount = meta.get("사업금액", "")
        formatted_amount = format_amount(raw_amount)

        block = f"""
[검색 결과 {i}]
문서명: {meta.get("file_name", "")}
문서 ID: {meta.get("doc_id", "")}
사업명: {meta.get("사업명", "")}
발주기관: {meta.get("발주기관", "")}
사업유형: {meta.get("사업유형", "")}
사업금액_원문표기: {formatted_amount}
공개일자: {meta.get("공개일자", "")}
입찰참여기간: {meta.get("입찰참여시작일", "")} ~ {meta.get("입찰참여마감일", "")}
문서 위치: {meta.get("header_path", "")}
section_id: {meta.get("section_id", "")}
block_id: {meta.get("block_id", "")}

사업요약:
{meta.get("사업요약", "")}

검색된 본문 일부:
{doc.page_content[:max_content_chars]}
"""
        context_blocks.append(block.strip())

    return "\n\n".join(context_blocks)

In [17]:
def build_rag_qa_prompt(question, context):
    return f"""
당신은 공공 RFP 문서를 분석하는 AI입니다.

아래 [검색 결과 context]만 근거로 답변하세요.

규칙:
1. 문서에 있는 정보만 사용하세요.
2. 문서에 없는 내용은 추측하지 마세요.
3. 금액은 반드시 `사업금액_원문표기`에 있는 값을 그대로 복사해서 답변하세요.
4. 금액을 억 원, 만 원, 1억 4천만 원처럼 환산해서 표현하지 마세요.
5. `1억`, `천만`, `만원`, `억 원` 같은 환산 표현은 절대 사용하지 마세요.
6. 한국어 존댓말로 간결하게 답변하세요.
7. 답변은 반드시 아래 형식을 지키세요.

[검색 결과 context]
{context}

[사용자 질문]
{question}

[답변 형식]
답변: 이 사업의 예산은 {{사업금액_원문표기}}입니다.
근거: {{사업명}} / {{문서명}}

[답변]
""".strip()

In [18]:
query = "이 사업의 예산은 얼마인가요?"

docs = retriever.invoke(query)
context = build_rag_context(docs)
prompt = build_rag_qa_prompt(query, context)

result = ask_ollama(
    prompt=prompt,
    model="exaone3.5:2.4b",
    temperature=0.0,
    timeout=180
)

print("success:", result["success"])
print("elapsed_time:", round(result["elapsed_time"], 2))
print("error:", result["error"])
print("\nanswer:")
print(result["answer"])

success: True
elapsed_time: 5.06
error: None

answer:
답변: 이 사업의 예산은 **140,000,000원**입니다.
근거: **통합정보시스템 고도화 용역** / **국가과학기술지식정보서비스_통합정보시스템 고도화 용역.hwp**


### RAG 답변 생성 프롬프트 보완 결과

초기 RAG 답변 생성 테스트에서는 모델이 `140,000,000원`을 이해했지만, `1억 4천만 원`처럼 환산 표현을 우선 사용하여 금액 원문 표기 규칙을 완전히 따르지 못했습니다.

이에 따라 context builder에서 metadata의 `사업금액` 값을 `140,000,000원` 형태의 문자열로 변환한 `사업금액_원문표기` 필드를 추가하였고, 프롬프트에서도 해당 값을 그대로 복사해 답변하도록 지시를 강화하였습니다.

재실행 결과, 모델은 예산을 `140,000,000원`으로 답변하였고, 근거로 사업명과 문서명을 함께 제시하였습니다. 따라서 수정된 context builder와 RAG QA 프롬프트는 예산 질의에 대해 정상적으로 동작하는 것으로 확인되었습니다.

In [21]:
# 특정 문서 doc_id로 검색 범위를 고정한 retriever
target_doc_id = "D016"

filtered_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5,
        "filter": {"doc_id": target_doc_id}
    }
)

print("Filtered retriever 생성 완료")

Filtered retriever 생성 완료


In [22]:
def build_rag_qa_prompt(question, context):
    return f"""
당신은 공공 RFP 문서를 분석하는 AI입니다.

아래 [검색 결과 context]만 근거로 사용자 질문에 답변하세요.

규칙:
1. 문서에 있는 정보만 사용하세요.
2. 문서에 없는 내용은 추측하지 마세요.
3. 사용자의 질문에 직접 답하세요.
4. 질문이 예산/금액에 관한 경우, 금액은 반드시 context의 원문 숫자 표기를 그대로 사용하세요.
5. 금액을 억 원, 만 원, 1억 4천만 원처럼 환산해서 표현하지 마세요.
6. 질문이 기간에 관한 경우, 사업요약이나 문서에 있는 기간 표현을 그대로 사용하세요.
7. 질문이 계약 방식에 관한 경우, 계약 방식만 간결하게 답하세요.
8. 질문이 사업 범위에 관한 경우, 사업 범위에 해당하는 항목만 정리하세요.
9. 한국어 존댓말로 간결하게 답변하세요.
10. 답변 마지막에는 반드시 "근거: 사업명 / 문서명" 형식으로 표시하세요.

[검색 결과 context]
{context}

[사용자 질문]
{question}

[답변 형식]
답변: ...
근거: 사업명 / 문서명

[답변]
""".strip()

In [23]:
rag_test_cases = [
    {
        "case_id": "RAG-01",
        "question": "통합정보시스템 고도화 용역의 예산은 얼마인가요?",
        "expected": "140,000,000원",
        "check_point": "사업금액 원문 표기 유지"
    },
    {
        "case_id": "RAG-02",
        "question": "통합정보시스템 고도화 용역의 사업기간은 어떻게 되나요?",
        "expected": "5개월 이내",
        "check_point": "사업기간 추출"
    },
    {
        "case_id": "RAG-03",
        "question": "통합정보시스템 고도화 용역의 계약 방식은 무엇인가요?",
        "expected": "협상에 의한 계약",
        "check_point": "계약 방식 추출"
    },
    {
        "case_id": "RAG-04",
        "question": "통합정보시스템 고도화 용역의 주요 사업 범위는 무엇인가요?",
        "expected": "기관생명윤리, 동물실험윤리, 국가연구개발사업 연구비 사용기준 준수 등",
        "check_point": "사업 범위 요약"
    },
    {
        "case_id": "RAG-05",
        "question": "통합정보시스템 고도화 용역의 요구사항은 몇 개로 구성되어 있나요?",
        "expected": "총 24개",
        "check_point": "요구사항 수 추출"
    },
]

In [24]:
rag_results = []

model = "exaone3.5:2.4b"

for case in rag_test_cases:
    query = case["question"]
    
    docs = filtered_retriever.invoke(query)
    context = build_rag_context(docs)
    prompt = build_rag_qa_prompt(query, context)
    
    result = ask_ollama(
        prompt=prompt,
        model=model,
        temperature=0.0,
        timeout=180
    )
    
    rag_results.append({
        "case_id": case["case_id"],
        "model": model,
        "question": query,
        "expected": case["expected"],
        "check_point": case["check_point"],
        "success": result["success"],
        "elapsed_time": round(result["elapsed_time"], 2),
        "answer": result["answer"],
        "error": result["error"]
    })

rag_results_df = pd.DataFrame(rag_results)
rag_results_df

,case_id,model,question,expected,check_point,success,elapsed_time,answer,error
0,RAG-01,exaone3.5:2.4b,통합정보시스템 고도화 용역의 예산은 얼마인가요?,"140,000,000원",사업금액 원문 표기 유지,True,5.34,"답변: **통합정보시스템 고도화 용역의 예산은 140,000,000원입니다.**\n...",None
1,RAG-02,exaone3.5:2.4b,통합정보시스템 고도화 용역의 사업기간은 어떻게 되나요?,5개월 이내,사업기간 추출,True,3.33,답변: 통합정보시스템 고도화 용역의 사업기간은 **5개월**입니다.\n\n근거: \...,None
2,RAG-03,exaone3.5:2.4b,통합정보시스템 고도화 용역의 계약 방식은 무엇인가요?,협상에 의한 계약,계약 방식 추출,True,3.59,답변: 통합정보시스템 고도화 용역의 계약 방식은 **협상에 의한 계약**입니다.\n...,None
3,RAG-04,exaone3.5:2.4b,통합정보시스템 고도화 용역의 주요 사업 범위는 무엇인가요?,"기관생명윤리, 동물실험윤리, 국가연구개발사업 연구비 사용기준 준수 등",사업 범위 요약,True,4.91,답변: **통합정보시스템 고도화 용역의 주요 사업 범위는 다음과 같습니다:**\n\...,None
4,RAG-05,exaone3.5:2.4b,통합정보시스템 고도화 용역의 요구사항은 몇 개로 구성되어 있나요?,총 24개,요구사항 수 추출,True,3.91,답변: 통합정보시스템 고도화 용역의 요구사항은 총 **24개**로 구성되어 있습니다...,None


### RAG 반복 테스트 2차 결과 해석

1차 반복 테스트에서는 “이 사업”처럼 사업명이 명시되지 않은 질문으로 인해 검색 대상 문서가 흔들리는 문제가 확인되었습니다. 또한 예산 질의에 맞춰 작성한 답변 형식이 다른 질문 유형에도 영향을 주어, 사업기간이나 사업 범위 질문에서도 예산 중심 답변이 생성되는 문제가 있었습니다.

이를 보완하기 위해 `doc_id`를 기준으로 검색 대상을 `통합정보시스템 고도화 용역(D016)` 문서로 고정하고, 테스트 질문에도 사업명을 명시하였습니다. 또한 예산 전용 답변 형식을 제거하고, 예산·기간·계약 방식·사업 범위·요구사항 수 등 다양한 질문 유형에 대응할 수 있는 일반 RAG QA 프롬프트로 수정하였습니다.

수정 후 반복 테스트 결과, 예산, 계약 방식, 주요 사업 범위, 요구사항 수는 정상적으로 추출되었습니다. 특히 예산 질의에서는 `140,000,000원`이라는 원문 금액 표기를 유지하였고, 계약 방식도 `협상에 의한 계약`으로 올바르게 답변하였습니다.

다만 사업기간 질의에서는 문서의 원문 표현인 `5개월 이내` 대신 `5개월`로 답변하여, 기간 표현의 원문 보존 측면에서는 부분 통과로 판단하였습니다. 따라서 이후 프롬프트에서는 금액뿐 아니라 기간 표현도 원문 그대로 유지하도록 지시를 강화할 필요가 있습니다.

전체적으로는 검색 대상 고정과 일반 QA 프롬프트 수정 이후 RAG 답변 품질이 크게 개선되었으며, `context builder → retriever 검색 결과 → LLM 답변 생성` 흐름이 정상적으로 연결되는 것을 확인하였습니다.


In [25]:
def build_rag_qa_prompt(question, context):
    return f"""
당신은 공공 RFP 문서를 분석하는 AI입니다.

아래 [검색 결과 context]만 근거로 사용자 질문에 답변하세요.

규칙:
1. 문서에 있는 정보만 사용하세요.
2. 문서에 없는 내용은 추측하지 마세요.
3. 사용자의 질문에 직접 답하세요.
4. 질문이 예산/금액에 관한 경우, 금액은 반드시 context의 원문 숫자 표기를 그대로 사용하세요.
5. 금액을 억 원, 만 원, 1억 4천만 원처럼 환산해서 표현하지 마세요.
6. 질문이 기간에 관한 경우, `5개월 이내`, `계약일로부터 24개월 이내`처럼 context에 나온 기간 표현을 그대로 사용하세요. `이내`, `이상`, `미만`, `계약일로부터` 같은 표현을 생략하지 마세요.
7. 질문이 계약 방식에 관한 경우, 계약 방식만 간결하게 답하세요.
8. 질문이 사업 범위에 관한 경우, 사업 범위에 해당하는 항목만 정리하세요.
9. 한국어 존댓말로 간결하게 답변하세요.
10. 답변 마지막에는 반드시 "근거: 사업명 / 문서명" 형식으로 표시하세요.

[검색 결과 context]
{context}

[사용자 질문]
{question}

[답변 형식]
답변: ...
근거: 사업명 / 문서명

[답변]
""".strip()

In [26]:
rag_candidate_models = [
    "exaone3.5:2.4b",
    "gemma3:4b",
    "qwen2.5:3b",
]

In [27]:
rag_compare_results = []

for model in rag_candidate_models:
    print(f"테스트 중: {model}")

    for case in rag_test_cases:
        query = case["question"]

        docs = filtered_retriever.invoke(query)
        context = build_rag_context(docs)
        prompt = build_rag_qa_prompt(query, context)

        result = ask_ollama(
            prompt=prompt,
            model=model,
            temperature=0.0,
            timeout=180
        )

        rag_compare_results.append({
            "case_id": case["case_id"],
            "model": model,
            "question": query,
            "expected": case["expected"],
            "check_point": case["check_point"],
            "success": result["success"],
            "elapsed_time": round(result["elapsed_time"], 2),
            "answer": result["answer"],
            "error": result["error"]
        })

rag_compare_df = pd.DataFrame(rag_compare_results)
rag_compare_df

테스트 중: exaone3.5:2.4b
테스트 중: gemma3:4b
테스트 중: qwen2.5:3b


,case_id,model,question,expected,check_point,success,elapsed_time,answer,error
0,RAG-01,exaone3.5:2.4b,통합정보시스템 고도화 용역의 예산은 얼마인가요?,"140,000,000원",사업금액 원문 표기 유지,True,5.69,"답변: **통합정보시스템 고도화 용역의 예산은 140,000,000원입니다.**\n...",None
1,RAG-02,exaone3.5:2.4b,통합정보시스템 고도화 용역의 사업기간은 어떻게 되나요?,5개월 이내,사업기간 추출,True,3.46,답변: 통합정보시스템 고도화 용역의 사업기간은 **5개월**입니다.\n\n근거: \...,None
2,RAG-03,exaone3.5:2.4b,통합정보시스템 고도화 용역의 계약 방식은 무엇인가요?,협상에 의한 계약,계약 방식 추출,True,3.67,답변: 통합정보시스템 고도화 용역의 계약 방식은 **협상에 의한 계약**입니다.\n...,None
3,RAG-04,exaone3.5:2.4b,통합정보시스템 고도화 용역의 주요 사업 범위는 무엇인가요?,"기관생명윤리, 동물실험윤리, 국가연구개발사업 연구비 사용기준 준수 등",사업 범위 요약,True,5.05,답변: **통합정보시스템 고도화 용역의 주요 사업 범위는 다음과 같습니다:**\n\...,None
4,RAG-05,exaone3.5:2.4b,통합정보시스템 고도화 용역의 요구사항은 몇 개로 구성되어 있나요?,총 24개,요구사항 수 추출,True,4.05,답변: 통합정보시스템 고도화 용역의 요구사항은 총 **24개**로 구성되어 있습니다...,None
5,RAG-01,gemma3:4b,통합정보시스템 고도화 용역의 예산은 얼마인가요?,"140,000,000원",사업금액 원문 표기 유지,True,8.61,답,None
6,RAG-02,gemma3:4b,통합정보시스템 고도화 용역의 사업기간은 어떻게 되나요?,5개월 이내,사업기간 추출,True,3.38,답,None
7,RAG-03,gemma3:4b,통합정보시스템 고도화 용역의 계약 방식은 무엇인가요?,협상에 의한 계약,계약 방식 추출,True,3.52,답,None
8,RAG-04,gemma3:4b,통합정보시스템 고도화 용역의 주요 사업 범위는 무엇인가요?,"기관생명윤리, 동물실험윤리, 국가연구개발사업 연구비 사용기준 준수 등",사업 범위 요약,True,3.46,답,None
9,RAG-05,gemma3:4b,통합정보시스템 고도화 용역의 요구사항은 몇 개로 구성되어 있나요?,총 24개,요구사항 수 추출,True,3.45,답,None


In [28]:
rag_compare_df[
    [
        "case_id",
        "model",
        "question",
        "expected",
        "check_point",
        "success",
        "elapsed_time",
        "answer"
    ]
]

,case_id,model,question,expected,check_point,success,elapsed_time,answer
0,RAG-01,exaone3.5:2.4b,통합정보시스템 고도화 용역의 예산은 얼마인가요?,"140,000,000원",사업금액 원문 표기 유지,True,5.69,"답변: **통합정보시스템 고도화 용역의 예산은 140,000,000원입니다.**\n..."
1,RAG-02,exaone3.5:2.4b,통합정보시스템 고도화 용역의 사업기간은 어떻게 되나요?,5개월 이내,사업기간 추출,True,3.46,답변: 통합정보시스템 고도화 용역의 사업기간은 **5개월**입니다.\n\n근거: \...
2,RAG-03,exaone3.5:2.4b,통합정보시스템 고도화 용역의 계약 방식은 무엇인가요?,협상에 의한 계약,계약 방식 추출,True,3.67,답변: 통합정보시스템 고도화 용역의 계약 방식은 **협상에 의한 계약**입니다.\n...
3,RAG-04,exaone3.5:2.4b,통합정보시스템 고도화 용역의 주요 사업 범위는 무엇인가요?,"기관생명윤리, 동물실험윤리, 국가연구개발사업 연구비 사용기준 준수 등",사업 범위 요약,True,5.05,답변: **통합정보시스템 고도화 용역의 주요 사업 범위는 다음과 같습니다:**\n\...
4,RAG-05,exaone3.5:2.4b,통합정보시스템 고도화 용역의 요구사항은 몇 개로 구성되어 있나요?,총 24개,요구사항 수 추출,True,4.05,답변: 통합정보시스템 고도화 용역의 요구사항은 총 **24개**로 구성되어 있습니다...
5,RAG-01,gemma3:4b,통합정보시스템 고도화 용역의 예산은 얼마인가요?,"140,000,000원",사업금액 원문 표기 유지,True,8.61,답
6,RAG-02,gemma3:4b,통합정보시스템 고도화 용역의 사업기간은 어떻게 되나요?,5개월 이내,사업기간 추출,True,3.38,답
7,RAG-03,gemma3:4b,통합정보시스템 고도화 용역의 계약 방식은 무엇인가요?,협상에 의한 계약,계약 방식 추출,True,3.52,답
8,RAG-04,gemma3:4b,통합정보시스템 고도화 용역의 주요 사업 범위는 무엇인가요?,"기관생명윤리, 동물실험윤리, 국가연구개발사업 연구비 사용기준 준수 등",사업 범위 요약,True,3.46,답
9,RAG-05,gemma3:4b,통합정보시스템 고도화 용역의 요구사항은 몇 개로 구성되어 있나요?,총 24개,요구사항 수 추출,True,3.45,답


In [29]:
rag_speed_summary = (
    rag_compare_df
    .groupby("model")
    .agg(
        test_count=("case_id", "count"),
        success_count=("success", "sum"),
        avg_elapsed_time=("elapsed_time", "mean"),
        max_elapsed_time=("elapsed_time", "max"),
        min_elapsed_time=("elapsed_time", "min")
    )
    .reset_index()
)

rag_speed_summary["avg_elapsed_time"] = rag_speed_summary["avg_elapsed_time"].round(2)
rag_speed_summary

,model,test_count,success_count,avg_elapsed_time,max_elapsed_time,min_elapsed_time
0,exaone3.5:2.4b,5,5,4.38,5.69,3.46
1,gemma3:4b,5,5,4.48,8.61,3.38
2,qwen2.5:3b,5,5,4.56,8.00,3.51


In [31]:
def build_rag_qa_prompt(question, context):
    return f"""
당신은 공공 RFP 문서를 분석하는 AI입니다.

아래 검색 결과만 근거로 사용자 질문에 답변하세요.

규칙:
- 문서에 있는 정보만 사용하세요.
- 문서에 없는 내용은 추측하지 마세요.
- 사용자의 질문에 직접 답하세요.
- 금액은 원문 숫자 표기 그대로 사용하세요.
- 금액을 억 원, 만 원, 1억 4천만 원처럼 환산하지 마세요.
- 기간은 context에 나온 표현을 그대로 사용하세요. 예: 5개월 이내
- 한국어 존댓말로 간결하게 답변하세요.
- 마지막 줄에는 근거를 `근거: 사업명 / 문서명` 형태로 작성하세요.

검색 결과:
{context}

사용자 질문:
{question}

위 질문에 대한 답변을 작성하세요.
""".strip()

In [32]:
gemma_retry_results = []

model = "gemma3:4b"

for case in rag_test_cases:
    query = case["question"]

    docs = filtered_retriever.invoke(query)
    context = build_rag_context(docs)
    prompt = build_rag_qa_prompt(query, context)

    result = ask_ollama(
        prompt=prompt,
        model=model,
        temperature=0.0,
        timeout=180
    )

    gemma_retry_results.append({
        "case_id": case["case_id"],
        "model": model,
        "question": query,
        "expected": case["expected"],
        "check_point": case["check_point"],
        "success": result["success"],
        "elapsed_time": round(result["elapsed_time"], 2),
        "answer": result["answer"],
        "error": result["error"]
    })

gemma_retry_df = pd.DataFrame(gemma_retry_results)
gemma_retry_df

,case_id,model,question,expected,check_point,success,elapsed_time,answer,error
0,RAG-01,gemma3:4b,통합정보시스템 고도화 용역의 예산은 얼마인가요?,"140,000,000원",사업금액 원문 표기 유지,True,8.19,통,None
1,RAG-02,gemma3:4b,통합정보시스템 고도화 용역의 사업기간은 어떻게 되나요?,5개월 이내,사업기간 추출,True,3.22,통,None
2,RAG-03,gemma3:4b,통합정보시스템 고도화 용역의 계약 방식은 무엇인가요?,협상에 의한 계약,계약 방식 추출,True,3.24,통,None
3,RAG-04,gemma3:4b,통합정보시스템 고도화 용역의 주요 사업 범위는 무엇인가요?,"기관생명윤리, 동물실험윤리, 국가연구개발사업 연구비 사용기준 준수 등",사업 범위 요약,True,3.41,통,None
4,RAG-05,gemma3:4b,통합정보시스템 고도화 용역의 요구사항은 몇 개로 구성되어 있나요?,총 24개,요구사항 수 추출,True,3.33,통,None


In [38]:
def build_short_rag_context(docs, max_content_chars=300):
    """
    gemma3:4b처럼 긴 context에서 출력이 불안정한 모델을 위한
    짧은 RAG context builder입니다.
    """
    context_blocks = []

    for i, doc in enumerate(docs, start=1):
        meta = doc.metadata or {}

        raw_amount = meta.get("사업금액", "")
        formatted_amount = format_amount(raw_amount)

        block = f"""
[검색 결과 {i}]
문서명: {meta.get("file_name", "")}
사업명: {meta.get("사업명", "")}
발주기관: {meta.get("발주기관", "")}
사업금액: {formatted_amount}
사업요약:
{meta.get("사업요약", "")}

문서 위치: {meta.get("header_path", "")}

본문 일부:
{doc.page_content[:max_content_chars]}
"""
        context_blocks.append(block.strip())

    return "\n\n".join(context_blocks)

In [39]:
gemma_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 2,
        "filter": {"doc_id": "D016"}
    }
)

print("gemma용 retriever 생성 완료")

gemma용 retriever 생성 완료


In [40]:
query = "통합정보시스템 고도화 용역의 예산은 얼마인가요?"

docs = gemma_retriever.invoke(query)
context = build_short_rag_context(docs)
prompt = build_rag_qa_prompt(query, context)

gemma_rag_test = ask_ollama(
    prompt=prompt,
    model="gemma3:4b",
    temperature=0.0,
    timeout=180,
    num_predict=512
)

print("success:", gemma_rag_test["success"])
print("elapsed_time:", round(gemma_rag_test["elapsed_time"], 2))
print("error:", gemma_rag_test["error"])
print("answer repr:", repr(gemma_rag_test["answer"]))
print("\nanswer:")
print(gemma_rag_test["answer"])

success: True
elapsed_time: 3.47
error: None
answer repr: '통합정보시스템 고도화 용역의 예산은 140,000,000원입니다. \n근거: 사업명 / 통합정보시스템 고도화 용역'

answer:
통합정보시스템 고도화 용역의 예산은 140,000,000원입니다. 
근거: 사업명 / 통합정보시스템 고도화 용역


In [41]:
gemma_short_results = []

model = "gemma3:4b"

for case in rag_test_cases:
    query = case["question"]

    docs = gemma_retriever.invoke(query)
    context = build_short_rag_context(docs)
    prompt = build_rag_qa_prompt(query, context)

    result = ask_ollama(
        prompt=prompt,
        model=model,
        temperature=0.0,
        timeout=180,
        num_predict=512
    )

    gemma_short_results.append({
        "case_id": case["case_id"],
        "model": model,
        "question": query,
        "expected": case["expected"],
        "check_point": case["check_point"],
        "success": result["success"],
        "elapsed_time": round(result["elapsed_time"], 2),
        "answer": result["answer"],
        "error": result["error"]
    })

gemma_short_df = pd.DataFrame(gemma_short_results)
gemma_short_df

,case_id,model,question,expected,check_point,success,elapsed_time,answer,error
0,RAG-01,gemma3:4b,통합정보시스템 고도화 용역의 예산은 얼마인가요?,"140,000,000원",사업금액 원문 표기 유지,True,3.27,"통합정보시스템 고도화 용역의 예산은 140,000,000원입니다. \n근거: 사업명...",None
1,RAG-02,gemma3:4b,통합정보시스템 고도화 용역의 사업기간은 어떻게 되나요?,5개월 이내,사업기간 추출,True,3.40,사업 기간은 5개월 이내입니다.\n\n근거: 사업명 / 통합정보시스템 고도화 용역,None
2,RAG-03,gemma3:4b,통합정보시스템 고도화 용역의 계약 방식은 무엇인가요?,협상에 의한 계약,계약 방식 추출,True,3.31,협상에 의한 계약입니다. \n근거: 사업명 / 통합정보시스템 고도화 용역,None
3,RAG-04,gemma3:4b,통합정보시스템 고도화 용역의 주요 사업 범위는 무엇인가요?,"기관생명윤리, 동물실험윤리, 국가연구개발사업 연구비 사용기준 준수 등",사업 범위 요약,True,3.95,"통합정보시스템 고도화 용역의 주요 사업 범위는 다음과 같습니다. 기관생명윤리, 동물...",None
4,RAG-05,gemma3:4b,통합정보시스템 고도화 용역의 요구사항은 몇 개로 구성되어 있나요?,총 24개,요구사항 수 추출,True,3.61,요구사항은 총 24개의 요구사항으로 구성되어 있습니다. \n근거: 통합정보시스템 고...,None


### gemma3:4b 재실험 결과

초기 RAG 비교에서 `gemma3:4b`는 응답이 한 글자 수준으로 출력되어 정상적인 평가가 어려웠습니다. 단순 질의에서는 정상적으로 응답했기 때문에, 모델 자체 오류보다는 긴 RAG context 또는 복잡한 프롬프트 구조에 따른 출력 불안정으로 판단하였습니다.

이에 따라 검색 결과 수를 `k=2`로 줄이고, metadata 중심의 짧은 context builder를 적용하여 재실험하였습니다. 재실험 결과, 예산, 사업기간, 계약 방식, 사업 범위, 요구사항 수에 대해 모두 정상적인 답변을 생성하였습니다. 특히 예산 질의에서는 `140,000,000원`, 사업기간 질의에서는 `5개월 이내`처럼 원문 표현을 유지하였습니다.

다만 근거 표시에서는 문서명까지 완전하게 제시하지 못하고 `사업명 / 통합정보시스템 고도화 용역`처럼 일부 형식이 불완전한 사례가 있었습니다. 따라서 `gemma3:4b`는 짧은 context 조건에서는 RAG QA에 활용 가능하지만, 긴 context 입력과 근거 형식 준수 측면에서는 추가 보완이 필요한 모델로 판단하였습니다.


In [42]:
# 기존 3개 모델 비교 결과에서 gemma3:4b의 비정상 결과 제거
rag_compare_fixed_df = rag_compare_df[rag_compare_df["model"] != "gemma3:4b"].copy()

# 짧은 context로 재실험한 gemma3:4b 결과 추가
rag_compare_fixed_df = pd.concat(
    [rag_compare_fixed_df, gemma_short_df],
    ignore_index=True
)

# 보기 좋게 정렬
rag_compare_fixed_df = rag_compare_fixed_df.sort_values(
    by=["model", "case_id"]
).reset_index(drop=True)

rag_compare_fixed_df[
    [
        "case_id",
        "model",
        "question",
        "expected",
        "check_point",
        "success",
        "elapsed_time",
        "answer"
    ]
]

,case_id,model,question,expected,check_point,success,elapsed_time,answer
0,RAG-01,exaone3.5:2.4b,통합정보시스템 고도화 용역의 예산은 얼마인가요?,"140,000,000원",사업금액 원문 표기 유지,True,5.69,"답변: **통합정보시스템 고도화 용역의 예산은 140,000,000원입니다.**\n..."
1,RAG-02,exaone3.5:2.4b,통합정보시스템 고도화 용역의 사업기간은 어떻게 되나요?,5개월 이내,사업기간 추출,True,3.46,답변: 통합정보시스템 고도화 용역의 사업기간은 **5개월**입니다.\n\n근거: \...
2,RAG-03,exaone3.5:2.4b,통합정보시스템 고도화 용역의 계약 방식은 무엇인가요?,협상에 의한 계약,계약 방식 추출,True,3.67,답변: 통합정보시스템 고도화 용역의 계약 방식은 **협상에 의한 계약**입니다.\n...
3,RAG-04,exaone3.5:2.4b,통합정보시스템 고도화 용역의 주요 사업 범위는 무엇인가요?,"기관생명윤리, 동물실험윤리, 국가연구개발사업 연구비 사용기준 준수 등",사업 범위 요약,True,5.05,답변: **통합정보시스템 고도화 용역의 주요 사업 범위는 다음과 같습니다:**\n\...
4,RAG-05,exaone3.5:2.4b,통합정보시스템 고도화 용역의 요구사항은 몇 개로 구성되어 있나요?,총 24개,요구사항 수 추출,True,4.05,답변: 통합정보시스템 고도화 용역의 요구사항은 총 **24개**로 구성되어 있습니다...
5,RAG-01,gemma3:4b,통합정보시스템 고도화 용역의 예산은 얼마인가요?,"140,000,000원",사업금액 원문 표기 유지,True,3.27,"통합정보시스템 고도화 용역의 예산은 140,000,000원입니다. \n근거: 사업명..."
6,RAG-02,gemma3:4b,통합정보시스템 고도화 용역의 사업기간은 어떻게 되나요?,5개월 이내,사업기간 추출,True,3.40,사업 기간은 5개월 이내입니다.\n\n근거: 사업명 / 통합정보시스템 고도화 용역
7,RAG-03,gemma3:4b,통합정보시스템 고도화 용역의 계약 방식은 무엇인가요?,협상에 의한 계약,계약 방식 추출,True,3.31,협상에 의한 계약입니다. \n근거: 사업명 / 통합정보시스템 고도화 용역
8,RAG-04,gemma3:4b,통합정보시스템 고도화 용역의 주요 사업 범위는 무엇인가요?,"기관생명윤리, 동물실험윤리, 국가연구개발사업 연구비 사용기준 준수 등",사업 범위 요약,True,3.95,"통합정보시스템 고도화 용역의 주요 사업 범위는 다음과 같습니다. 기관생명윤리, 동물..."
9,RAG-05,gemma3:4b,통합정보시스템 고도화 용역의 요구사항은 몇 개로 구성되어 있나요?,총 24개,요구사항 수 추출,True,3.61,요구사항은 총 24개의 요구사항으로 구성되어 있습니다. \n근거: 통합정보시스템 고...


In [43]:
rag_speed_summary_fixed = (
    rag_compare_fixed_df
    .groupby("model")
    .agg(
        test_count=("case_id", "count"),
        success_count=("success", "sum"),
        avg_elapsed_time=("elapsed_time", "mean"),
        max_elapsed_time=("elapsed_time", "max"),
        min_elapsed_time=("elapsed_time", "min")
    )
    .reset_index()
)

rag_speed_summary_fixed["avg_elapsed_time"] = rag_speed_summary_fixed["avg_elapsed_time"].round(2)
rag_speed_summary_fixed

,model,test_count,success_count,avg_elapsed_time,max_elapsed_time,min_elapsed_time
0,exaone3.5:2.4b,5,5,4.38,5.69,3.46
1,gemma3:4b,5,5,3.51,3.95,3.27
2,qwen2.5:3b,5,5,4.56,8.00,3.51


In [44]:
# 기존 조건: k=5
gemma_default_retriever_k5 = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5,
        "filter": {"doc_id": "D016"}
    }
)

# 중간 조건: k=2
gemma_default_retriever_k2 = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 2,
        "filter": {"doc_id": "D016"}
    }
)

print("gemma 재테스트용 retriever 준비 완료")

gemma 재테스트용 retriever 준비 완료


In [45]:
def run_gemma_rag_retest(case, retriever_obj, context_builder, context_setting):
    query = case["question"]

    docs = retriever_obj.invoke(query)
    context = context_builder(docs)
    prompt = build_rag_qa_prompt(query, context)

    result = ask_ollama(
        prompt=prompt,
        model="gemma3:4b",
        temperature=0.0,
        timeout=180,
        num_predict=512
    )

    return {
        "case_id": case["case_id"],
        "model": "gemma3:4b",
        "context_setting": context_setting,
        "question": query,
        "expected": case["expected"],
        "check_point": case["check_point"],
        "success": result["success"],
        "elapsed_time": round(result["elapsed_time"], 2),
        "answer": result["answer"],
        "answer_len": len(result["answer"]),
        "error": result["error"]
    }

In [46]:
gemma_retest_results = []

for case in rag_test_cases:
    # A. 기존 context + k=5
    gemma_retest_results.append(
        run_gemma_rag_retest(
            case=case,
            retriever_obj=gemma_default_retriever_k5,
            context_builder=build_rag_context,
            context_setting="default_context_k5"
        )
    )

    # B. 기존 context + k=2
    gemma_retest_results.append(
        run_gemma_rag_retest(
            case=case,
            retriever_obj=gemma_default_retriever_k2,
            context_builder=build_rag_context,
            context_setting="default_context_k2"
        )
    )

    # C. 짧은 context + k=2
    gemma_retest_results.append(
        run_gemma_rag_retest(
            case=case,
            retriever_obj=gemma_default_retriever_k2,
            context_builder=build_short_rag_context,
            context_setting="short_context_k2"
        )
    )

gemma_retest_df = pd.DataFrame(gemma_retest_results)

gemma_retest_df[
    [
        "case_id",
        "context_setting",
        "expected",
        "elapsed_time",
        "answer_len",
        "answer"
    ]
]

,case_id,context_setting,expected,elapsed_time,answer_len,answer
0,RAG-01,default_context_k5,"140,000,000원",7.23,1,통
1,RAG-01,default_context_k2,"140,000,000원",3.53,66,"통합정보시스템 고도화 용역의 추정가격은 140,000,000원 이내입니다. 근거: ..."
2,RAG-01,short_context_k2,"140,000,000원",3.54,62,"통합정보시스템 고도화 용역의 예산은 140,000,000원입니다. \n근거: 사업명..."
3,RAG-02,default_context_k5,5개월 이내,3.31,1,통
4,RAG-02,default_context_k2,5개월 이내,3.54,56,사업 기간은 5개월 이내입니다.\n근거: 사업명 / 통합정보시스템 고도화 용역 / ...
5,RAG-02,short_context_k2,5개월 이내,3.37,43,사업 기간은 5개월 이내입니다.\n\n근거: 사업명 / 통합정보시스템 고도화 용역
6,RAG-03,default_context_k5,협상에 의한 계약,3.39,1,통
7,RAG-03,default_context_k2,협상에 의한 계약,3.39,63,통합정보시스템 고도화 용역의 계약 방식은 협상에 의한 계약입니다. `근거: 사업명 ...
8,RAG-03,short_context_k2,협상에 의한 계약,3.47,39,협상에 의한 계약입니다. \n근거: 사업명 / 통합정보시스템 고도화 용역
9,RAG-04,default_context_k5,"기관생명윤리, 동물실험윤리, 국가연구개발사업 연구비 사용기준 준수 등",3.28,1,통


In [47]:
gemma_retest_summary = (
    gemma_retest_df
    .groupby("context_setting")
    .agg(
        test_count=("case_id", "count"),
        avg_elapsed_time=("elapsed_time", "mean"),
        avg_answer_len=("answer_len", "mean"),
        min_answer_len=("answer_len", "min"),
        max_answer_len=("answer_len", "max")
    )
    .reset_index()
)

gemma_retest_summary[
    ["avg_elapsed_time", "avg_answer_len", "min_answer_len", "max_answer_len"]
] = gemma_retest_summary[
    ["avg_elapsed_time", "avg_answer_len", "min_answer_len", "max_answer_len"]
].round(2)

gemma_retest_summary

,context_setting,test_count,avg_elapsed_time,avg_answer_len,min_answer_len,max_answer_len
0,default_context_k2,5,3.66,78.0,56,139
1,default_context_k5,5,4.12,1.0,1,1
2,short_context_k2,5,3.59,71.4,39,137


### gemma3:4b context 조건별 재테스트 결과

초기 RAG 비교에서 `gemma3:4b`는 모든 답변이 한 글자 수준으로 출력되어 정상적인 평가가 어려웠습니다. 단순 질의에서는 정상적으로 응답했기 때문에, 모델 자체의 실행 오류가 아니라 RAG context 입력 조건에 따른 출력 불안정 가능성이 있다고 판단하였습니다.

이를 확인하기 위해 `gemma3:4b`를 대상으로 세 가지 조건을 비교하였습니다. 첫 번째는 기존 context builder와 검색 결과 수 `k=5`를 사용하는 `default_context_k5`, 두 번째는 기존 context builder를 유지하되 검색 결과 수를 `k=2`로 줄인 `default_context_k2`, 세 번째는 metadata 중심의 짧은 context builder와 `k=2`를 사용하는 `short_context_k2` 조건입니다.

재테스트 결과, `default_context_k5` 조건에서는 5개 케이스 모두 응답 길이가 1글자였고, 답변이 `"통"`으로만 출력되었습니다. 반면 `default_context_k2`와 `short_context_k2` 조건에서는 예산, 사업기간, 계약 방식, 사업 범위, 요구사항 수에 대해 정상적인 답변을 생성하였습니다. 특히 `default_context_k2`에서도 정상 답변이 생성된 점을 고려하면, 문제의 핵심은 context builder 자체보다는 검색 결과 수가 많아지면서 context가 길고 복잡해진 조건에 있는 것으로 판단됩니다.

따라서 `gemma3:4b`는 RAG QA에 사용할 수 없는 모델이라기보다는, 긴 context 또는 다수 검색 결과 입력에 민감한 모델로 평가하였습니다. 실제 적용 시에는 검색 결과 수를 줄이거나, metadata 중심으로 context를 압축하는 방식이 필요합니다.


## 기존 QA 기반 RAG 회귀 테스트

앞선 실험에서는 `통합정보시스템 고도화 용역(D016)` 문서를 기준으로 RAG 연결 상태와 모델별 응답 경향을 확인하였습니다. 이번 단계에서는 기존 프롬프트 실험에서 사용했던 QA 유형을 RAG 환경에 맞게 변환하여, 실제 retriever context 입력 조건에서도 모델들이 기존 검증 항목을 안정적으로 처리하는지 확인합니다.

단, 기존 QA 테스트셋의 context는 직접 작성한 가상 문맥이므로, 해당 expected 값을 그대로 사용하지 않습니다. 대신 기존 QA의 검증 의도인 금액 원문 보존, 항목 누락 방지, 가격점수 계산 가드레일, 기간 원문 보존, 문서 근거 부족 판단을 유지하되, expected 값은 현재 RAG 대상 문서인 `통합정보시스템 고도화 용역(D016)` 기준으로 재정의하였습니다.

따라서 본 테스트는 최종 Golden QA 평가가 아니라, RAG 연결 이후 기존 프롬프트 검증 항목이 유지되는지 확인하기 위한 회귀 테스트로 해석합니다.


In [48]:
existing_qa_rag_cases = [
    {
        "case_id": "IQA-RAG-01",
        "original_case_id": "IQA-01",
        "source_type": "existing_qa_adapted",
        "category": "금액 단위 보존",
        "question": "통합정보시스템 고도화 용역의 예산은 얼마인가요?",
        "expected": "140,000,000원",
        "check_points": ["140,000,000원", "억/조/만 원 단위 임의 환산 금지"]
    },
    {
        "case_id": "IQA-RAG-02",
        "original_case_id": "IQA-02",
        "source_type": "existing_qa_adapted",
        "category": "항목 누락 방지",
        "question": "통합정보시스템 고도화 용역의 주요 사업 범위를 모두 알려주세요.",
        "expected": "기관생명윤리, 동물실험윤리, 국가연구개발사업 연구비 사용기준 준수, 연구계획 관리, 전자결재서식 연동, 연구노트 관리",
        "check_points": ["주요 사업 범위", "항목 누락 최소화", "항목명 왜곡 없음"]
    },
    {
        "case_id": "IQA-RAG-03",
        "original_case_id": "IQA-03",
        "source_type": "existing_qa_adapted",
        "category": "가격점수 가드레일",
        "question": "통합정보시스템 고도화 용역의 가격점수를 계산해주세요.",
        "expected": "계산 불가 또는 정보 부족",
        "check_points": ["계산하지 않음", "입찰가격", "예정가격", "평점산식", "정보 부족"]
    },
    {
        "case_id": "IQA-RAG-04",
        "original_case_id": "IQA-04",
        "source_type": "existing_qa_adapted",
        "category": "가격점수 산식 일부 부족",
        "question": "통합정보시스템 고도화 용역의 입찰가격이 95,000,000원이라고 할 때 가격점수를 계산해주세요.",
        "expected": "계산 불가 또는 정보 부족",
        "check_points": ["계산하지 않음", "예정가격", "평점산식", "정보 부족"]
    },
    {
        "case_id": "IQA-RAG-05",
        "original_case_id": "IQA-05",
        "source_type": "existing_qa_adapted",
        "category": "기간 원문 보존",
        "question": "통합정보시스템 고도화 용역의 사업기간은 어떻게 되나요?",
        "expected": "5개월 이내",
        "check_points": ["5개월 이내", "이내 표현 생략 금지"]
    },
    {
        "case_id": "IQA-RAG-06",
        "original_case_id": "IQA-06",
        "source_type": "existing_qa_adapted",
        "category": "문서 근거 부족",
        "question": "통합정보시스템 고도화 용역의 유지보수 인력은 몇 명 투입해야 하나요?",
        "expected": "문서 근거 부족",
        "check_points": ["문서 근거 부족", "추측하지 않음", "인원 임의 생성 금지"]
    },
]

existing_qa_rag_df = pd.DataFrame(existing_qa_rag_cases)
existing_qa_rag_df

,case_id,original_case_id,source_type,category,question,expected,check_points
0,IQA-RAG-01,IQA-01,existing_qa_adapted,금액 단위 보존,통합정보시스템 고도화 용역의 예산은 얼마인가요?,"140,000,000원","[140,000,000원, 억/조/만 원 단위 임의 환산 금지]"
1,IQA-RAG-02,IQA-02,existing_qa_adapted,항목 누락 방지,통합정보시스템 고도화 용역의 주요 사업 범위를 모두 알려주세요.,"기관생명윤리, 동물실험윤리, 국가연구개발사업 연구비 사용기준 준수, 연구계획 관리,...","[주요 사업 범위, 항목 누락 최소화, 항목명 왜곡 없음]"
2,IQA-RAG-03,IQA-03,existing_qa_adapted,가격점수 가드레일,통합정보시스템 고도화 용역의 가격점수를 계산해주세요.,계산 불가 또는 정보 부족,"[계산하지 않음, 입찰가격, 예정가격, 평점산식, 정보 부족]"
3,IQA-RAG-04,IQA-04,existing_qa_adapted,가격점수 산식 일부 부족,"통합정보시스템 고도화 용역의 입찰가격이 95,000,000원이라고 할 때 가격점수를...",계산 불가 또는 정보 부족,"[계산하지 않음, 예정가격, 평점산식, 정보 부족]"
4,IQA-RAG-05,IQA-05,existing_qa_adapted,기간 원문 보존,통합정보시스템 고도화 용역의 사업기간은 어떻게 되나요?,5개월 이내,"[5개월 이내, 이내 표현 생략 금지]"
5,IQA-RAG-06,IQA-06,existing_qa_adapted,문서 근거 부족,통합정보시스템 고도화 용역의 유지보수 인력은 몇 명 투입해야 하나요?,문서 근거 부족,"[문서 근거 부족, 추측하지 않음, 인원 임의 생성 금지]"


In [49]:
existing_qa_rag_models = [
    "exaone3.5:2.4b",
    "gemma3:4b",
    "qwen2.5:3b",
]

existing_qa_rag_results = []

for model in existing_qa_rag_models:
    print(f"기존 QA 기반 RAG 테스트 실행 중: {model}")

    for case in existing_qa_rag_cases:
        query = case["question"]

        if model == "gemma3:4b":
            # gemma3:4b는 k=5에서 출력이 한 글자로 끊기는 현상이 있어 k=2 조건 사용
            docs = gemma_default_retriever_k2.invoke(query)
            context = build_rag_context(docs)
            context_setting = "default_context_k2"
        else:
            docs = filtered_retriever.invoke(query)
            context = build_rag_context(docs)
            context_setting = "default_context_k5"

        prompt = build_rag_qa_prompt(query, context)

        result = ask_ollama(
            prompt=prompt,
            model=model,
            temperature=0.0,
            timeout=180,
            num_predict=512
        )

        existing_qa_rag_results.append({
            "case_id": case["case_id"],
            "original_case_id": case["original_case_id"],
            "source_type": case["source_type"],
            "category": case["category"],
            "model": model,
            "context_setting": context_setting,
            "question": query,
            "expected": case["expected"],
            "check_points": case["check_points"],
            "success": result["success"],
            "elapsed_time": round(result["elapsed_time"], 2),
            "answer": result["answer"],
            "answer_len": len(result["answer"]),
            "error": result["error"]
        })

existing_qa_rag_results_df = pd.DataFrame(existing_qa_rag_results)

existing_qa_rag_results_df[
    [
        "case_id",
        "original_case_id",
        "category",
        "model",
        "context_setting",
        "expected",
        "elapsed_time",
        "answer_len",
        "answer"
    ]
]

기존 QA 기반 RAG 테스트 실행 중: exaone3.5:2.4b
기존 QA 기반 RAG 테스트 실행 중: gemma3:4b
기존 QA 기반 RAG 테스트 실행 중: qwen2.5:3b


,case_id,original_case_id,category,model,context_setting,expected,elapsed_time,answer_len,answer
0,IQA-RAG-01,IQA-01,금액 단위 보존,exaone3.5:2.4b,default_context_k5,"140,000,000원",7.02,238,통합정보시스템 고도화 용역의 예산은 **1억 4천만 원**입니다 (사업금액 원문 표...
1,IQA-RAG-02,IQA-02,항목 누락 방지,exaone3.5:2.4b,default_context_k5,"기관생명윤리, 동물실험윤리, 국가연구개발사업 연구비 사용기준 준수, 연구계획 관리,...",4.60,637,통합정보시스템 고도화 용역의 주요 사업 범위는 다음과 같습니다:\n\n1. **기관...
2,IQA-RAG-03,IQA-03,가격점수 가드레일,exaone3.5:2.4b,default_context_k5,계산 불가 또는 정보 부족,4.51,485,통합정보시스템 고도화 용역의 가격점수를 계산하기 위해서는 주어진 정보와 평가 기준을...
3,IQA-RAG-04,IQA-04,가격점수 산식 일부 부족,exaone3.5:2.4b,default_context_k5,계산 불가 또는 정보 부족,6.10,946,통합정보시스템 고도화 용역의 입찰 가격 평가 점수 계산을 위해 주어진 정보를 바탕으...
4,IQA-RAG-05,IQA-05,기간 원문 보존,exaone3.5:2.4b,default_context_k5,5개월 이내,3.24,153,통합정보시스템 고도화 용역의 사업기간은 **5개월**입니다. 구체적으로는 **사업 ...
5,IQA-RAG-06,IQA-06,문서 근거 부족,exaone3.5:2.4b,default_context_k5,문서 근거 부족,6.04,1048,통합정보시스템 고도화 용역의 유지보수 인력 규모에 대한 구체적인 숫자는 제공된 문서...
6,IQA-RAG-01,IQA-01,금액 단위 보존,gemma3:4b,default_context_k2,"140,000,000원",8.83,66,"통합정보시스템 고도화 용역의 추정가격은 140,000,000원 이내입니다. 근거: ..."
7,IQA-RAG-02,IQA-02,항목 누락 방지,gemma3:4b,default_context_k2,"기관생명윤리, 동물실험윤리, 국가연구개발사업 연구비 사용기준 준수, 연구계획 관리,...",3.84,128,"통합정보시스템 고도화 용역의 주요 사업 범위는 다음과 같습니다. 기관생명윤리, 동물..."
8,IQA-RAG-03,IQA-03,가격점수 가드레일,gemma3:4b,default_context_k2,계산 불가 또는 정보 부족,3.63,63,"통합정보시스템 고도화 용역의 추정가격은 140,000,000원입니다. 근거: 사업명..."
9,IQA-RAG-04,IQA-04,가격점수 산식 일부 부족,gemma3:4b,default_context_k2,계산 불가 또는 정보 부족,3.68,66,"통합정보시스템 고도화 용역의 추정가격은 140,000,000원 이내입니다. 근거: ..."


In [50]:
existing_qa_rag_speed_summary_df = (
    existing_qa_rag_results_df
    .groupby(["model", "context_setting"])
    .agg(
        test_count=("case_id", "count"),
        success_count=("success", "sum"),
        avg_elapsed_time=("elapsed_time", "mean"),
        max_elapsed_time=("elapsed_time", "max"),
        min_elapsed_time=("elapsed_time", "min"),
        avg_answer_len=("answer_len", "mean")
    )
    .reset_index()
)

existing_qa_rag_speed_summary_df[
    ["avg_elapsed_time", "max_elapsed_time", "min_elapsed_time", "avg_answer_len"]
] = existing_qa_rag_speed_summary_df[
    ["avg_elapsed_time", "max_elapsed_time", "min_elapsed_time", "avg_answer_len"]
].round(2)

existing_qa_rag_speed_summary_df

,model,context_setting,test_count,success_count,avg_elapsed_time,max_elapsed_time,min_elapsed_time,avg_answer_len
0,exaone3.5:2.4b,default_context_k5,6,6,5.25,7.02,3.24,584.50
1,gemma3:4b,default_context_k2,6,6,4.57,8.83,3.59,79.33
2,qwen2.5:3b,default_context_k5,6,6,5.20,7.22,3.09,320.00


In [52]:
# 세종테크노파크 문서 doc_id 확인
sejong_docs = vectorstore.similarity_search("세종테크노파크 인사정보 전산시스템 구축 용역", k=10)

print("세종테크노파크 검색 결과")
for i, doc in enumerate(sejong_docs, start=1):
    meta = doc.metadata or {}
    print("=" * 80)
    print(f"[{i}]")
    print("doc_id:", meta.get("doc_id"))
    print("file_name:", meta.get("file_name"))
    print("사업명:", meta.get("사업명"))
    print("발주기관:", meta.get("발주기관"))

세종테크노파크 검색 결과
[1]
doc_id: D088
file_name: 세종테크노파크_세종테크노파크 인사정보 전산시스템 구축 용역 입찰공.hwp
사업명: 세종테크노파크 인사정보 전산시스템 구축 용역 입찰공고[긴급]
발주기관: 세종테크노파크
[2]
doc_id: D088
file_name: 세종테크노파크_세종테크노파크 인사정보 전산시스템 구축 용역 입찰공.hwp
사업명: 세종테크노파크 인사정보 전산시스템 구축 용역 입찰공고[긴급]
발주기관: 세종테크노파크
[3]
doc_id: D088
file_name: 세종테크노파크_세종테크노파크 인사정보 전산시스템 구축 용역 입찰공.hwp
사업명: 세종테크노파크 인사정보 전산시스템 구축 용역 입찰공고[긴급]
발주기관: 세종테크노파크
[4]
doc_id: D088
file_name: 세종테크노파크_세종테크노파크 인사정보 전산시스템 구축 용역 입찰공.hwp
사업명: 세종테크노파크 인사정보 전산시스템 구축 용역 입찰공고[긴급]
발주기관: 세종테크노파크
[5]
doc_id: D088
file_name: 세종테크노파크_세종테크노파크 인사정보 전산시스템 구축 용역 입찰공.hwp
사업명: 세종테크노파크 인사정보 전산시스템 구축 용역 입찰공고[긴급]
발주기관: 세종테크노파크
[6]
doc_id: D088
file_name: 세종테크노파크_세종테크노파크 인사정보 전산시스템 구축 용역 입찰공.hwp
사업명: 세종테크노파크 인사정보 전산시스템 구축 용역 입찰공고[긴급]
발주기관: 세종테크노파크
[7]
doc_id: D088
file_name: 세종테크노파크_세종테크노파크 인사정보 전산시스템 구축 용역 입찰공.hwp
사업명: 세종테크노파크 인사정보 전산시스템 구축 용역 입찰공고[긴급]
발주기관: 세종테크노파크
[8]
doc_id: D088
file_name: 세종테크노파크_세종테크노파크 인사정보 전산시스템 구축 용역 입찰공.hwp
사업명: 세종테크노파크 인사정보 전산시스템 구축 용역 입찰공고[긴급]
발주기관: 세종테크노파크
[9]
doc_id

In [53]:
# 경희대학교 문서 doc_id 확인
kyunghee_docs = vectorstore.similarity_search("경희대학교 산학협력단 정보시스템 운영 용역업체 선정", k=10)

print("경희대학교 검색 결과")
for i, doc in enumerate(kyunghee_docs, start=1):
    meta = doc.metadata or {}
    print("=" * 80)
    print(f"[{i}]")
    print("doc_id:", meta.get("doc_id"))
    print("file_name:", meta.get("file_name"))
    print("사업명:", meta.get("사업명"))
    print("발주기관:", meta.get("발주기관"))

경희대학교 검색 결과
[1]
doc_id: D014
file_name: 경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp
사업명: [입찰공고] 산학협력단 정보시스템 운영 용역업체 선정
발주기관: 경희대학교
[2]
doc_id: D014
file_name: 경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp
사업명: [입찰공고] 산학협력단 정보시스템 운영 용역업체 선정
발주기관: 경희대학교
[3]
doc_id: D014
file_name: 경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp
사업명: [입찰공고] 산학협력단 정보시스템 운영 용역업체 선정
발주기관: 경희대학교
[4]
doc_id: D014
file_name: 경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp
사업명: [입찰공고] 산학협력단 정보시스템 운영 용역업체 선정
발주기관: 경희대학교
[5]
doc_id: D014
file_name: 경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp
사업명: [입찰공고] 산학협력단 정보시스템 운영 용역업체 선정
발주기관: 경희대학교
[6]
doc_id: D014
file_name: 경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp
사업명: [입찰공고] 산학협력단 정보시스템 운영 용역업체 선정
발주기관: 경희대학교
[7]
doc_id: D014
file_name: 경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp
사업명: [입찰공고] 산학협력단 정보시스템 운영 용역업체 선정
발주기관: 경희대학교
[8]
doc_id: D014
file_name: 경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp
사업명: [입찰공고] 산학협력단 정보시스템 운영 용역업체 선정
발주기관: 경희대학교
[9]
doc_id: D014
file_name: 경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp


In [54]:
# 고려대학교 문서 doc_id 확인
korea_docs = vectorstore.similarity_search("고려대학교 차세대 포털 학사 정보시스템 구축사업", k=10)

print("고려대학교 검색 결과")
for i, doc in enumerate(korea_docs, start=1):
    meta = doc.metadata or {}
    print("=" * 80)
    print(f"[{i}]")
    print("doc_id:", meta.get("doc_id"))
    print("file_name:", meta.get("file_name"))
    print("사업명:", meta.get("사업명"))
    print("발주기관:", meta.get("발주기관"))

고려대학교 검색 결과
[1]
doc_id: D008
file_name: 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf
사업명: 차세대 포털·학사 정보시스템 구축사업 
발주기관: 고려대학교
[2]
doc_id: D008
file_name: 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf
사업명: 차세대 포털·학사 정보시스템 구축사업 
발주기관: 고려대학교
[3]
doc_id: D008
file_name: 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf
사업명: 차세대 포털·학사 정보시스템 구축사업 
발주기관: 고려대학교
[4]
doc_id: D008
file_name: 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf
사업명: 차세대 포털·학사 정보시스템 구축사업 
발주기관: 고려대학교
[5]
doc_id: D008
file_name: 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf
사업명: 차세대 포털·학사 정보시스템 구축사업 
발주기관: 고려대학교
[6]
doc_id: D008
file_name: 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf
사업명: 차세대 포털·학사 정보시스템 구축사업 
발주기관: 고려대학교
[7]
doc_id: D008
file_name: 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf
사업명: 차세대 포털·학사 정보시스템 구축사업 
발주기관: 고려대학교
[8]
doc_id: D008
file_name: 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf
사업명: 차세대 포털·학사 정보시스템 구축사업 
발주기관: 고려대학교
[9]
doc_id: D008
file_name: 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf
사업명: 차세대 포털·학사 정보시스템 구축사업 
발주기관: 고려대학교
[10]
doc_id: D008
file_name: 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf
사업명: 차세대 포털·학사 정보시스템 구축사업 
발주기관: 고려대학교


In [55]:
# Vector DB에 들어있는 고유 문서 목록 확인
all_data = vectorstore.get(include=["metadatas"])

unique_docs = {}

for meta in all_data["metadatas"]:
    doc_id = meta.get("doc_id")
    if doc_id not in unique_docs:
        unique_docs[doc_id] = {
            "doc_id": doc_id,
            "file_name": meta.get("file_name"),
            "사업명": meta.get("사업명"),
            "발주기관": meta.get("발주기관"),
            "사업유형": meta.get("사업유형"),
            "사업금액": meta.get("사업금액"),
        }

unique_docs_df = pd.DataFrame(unique_docs.values()).sort_values("doc_id").reset_index(drop=True)
unique_docs_df

,doc_id,file_name,사업명,발주기관,사업유형,사업금액
0,D001,한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp,한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화,한영대학,고도화,1.300000e+08
1,D002,한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp,2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선,한국연구재단,개선,1.293000e+08
2,D003,한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp,EIP3.0 고압가스 안전관리 시스템 구축 용역,한국생산기술연구원,구축,4.000000e+07
3,D004,인천광역시_도시계획위원회 통합관리시스템 구축용역.hwp,도시계획위원회 통합관리시스템 구축용역,인천광역시,통합,1.500000e+08
4,D005,경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp,봉화군 재난통합관리시스템 고도화 사업(협상)(긴급),경상북도 봉화군,고도화,9.000000e+08
...,...,...,...,...,...,...
92,D095,대한적십자사 의료원_적십자병원 병원정보 재해복구시스템 구축 용역 .hwp,적십자병원 병원정보 재해복구시스템 구축 용역 재공고입찰,대한적십자사 의료원,구축,5.000000e+08
93,D096,한국수자원조사기술원_수문자료정보관리시스템(HDIMS) 재구축 용역(3단계.hwp,수문자료정보관리시스템(HDIMS) 재구축 용역(3단계),한국수자원조사기술원,재구축,4.932704e+08
94,D097,(재)예술경영지원센터_통합 정보시스템 구축 사전 컨설팅.hwp,통합 정보시스템 구축 사전 컨설팅,(재)예술경영지원센터,통합,5.000000e+07
95,D098,재단법인 광주광역시 광주문화재단_2024년 광주문화예술통합플랫폼 시스.hwp,2024년 광주문화예술통합플랫폼 시스템 유지 및 보수 용역 제안서 제출 안내 재공고,재단법인 광주광역시 광주문화재단,통합,6.189000e+07


In [57]:
doc_id_map = {
    "sejong": "D088",
    "kyunghee": "D014",
    "korea": "D008",
}

doc_retrievers = {
    doc_key: vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={
            "k": 5,
            "filter": {"doc_id": doc_id}
        }
    )
    for doc_key, doc_id in doc_id_map.items()
}

# gemma3:4b는 k=5에서 출력 불안정이 확인되어 k=2 retriever를 별도로 생성
doc_retrievers_k2 = {
    doc_key: vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={
            "k": 2,
            "filter": {"doc_id": doc_id}
        }
    )
    for doc_key, doc_id in doc_id_map.items()
}

print("3개 임시 Golden QA 문서 retriever 생성 완료")

3개 임시 Golden QA 문서 retriever 생성 완료


In [58]:
temp_golden_rag_cases = [
    # 세종테크노파크
    {
        "case_id": "TG-RAG-01",
        "doc_key": "sejong",
        "source_type": "temp_golden_rag",
        "category": "기본 정보 추출",
        "question": "세종테크노파크 인사정보 전산시스템 구축 용역의 용역명은 무엇인가요?",
        "expected": "세종테크노파크 인사정보 전산시스템 구축 용역",
        "check_points": ["용역명", "사업명"]
    },
    {
        "case_id": "TG-RAG-02",
        "doc_key": "sejong",
        "source_type": "temp_golden_rag",
        "category": "기간/금액 추출",
        "question": "세종테크노파크 인사정보 전산시스템 구축 용역의 사업기간과 사업비는 어떻게 되나요?",
        "expected": "사업기간, 사업비",
        "check_points": ["사업기간", "사업비", "금액 원문 표기"]
    },
    {
        "case_id": "TG-RAG-03",
        "doc_key": "sejong",
        "source_type": "temp_golden_rag",
        "category": "계약 방식 추출",
        "question": "세종테크노파크 인사정보 전산시스템 구축 용역의 계약방법은 무엇인가요?",
        "expected": "계약방법",
        "check_points": ["계약방법", "협상", "입찰"]
    },

    # 경희대학교
    {
        "case_id": "TG-RAG-04",
        "doc_key": "kyunghee",
        "source_type": "temp_golden_rag",
        "category": "기간/예산 추출",
        "question": "경희대학교 산학협력단 정보시스템 운영 용역의 사업기간과 예산은 어떻게 되나요?",
        "expected": "사업기간, 예산",
        "check_points": ["사업기간", "예산", "금액 원문 표기"]
    },
    {
        "case_id": "TG-RAG-05",
        "doc_key": "kyunghee",
        "source_type": "temp_golden_rag",
        "category": "운영/유지관리 범위",
        "question": "경희대학교 산학협력단 정보시스템 운영 용역의 주요 운영 및 유지관리 범위는 무엇인가요?",
        "expected": "정보시스템 운영 및 유지관리 범위",
        "check_points": ["운영", "유지관리", "업무 범위"]
    },

    # 고려대학교
    {
        "case_id": "TG-RAG-06",
        "doc_key": "korea",
        "source_type": "temp_golden_rag",
        "category": "기본 정보 추출",
        "question": "고려대학교 차세대 포털·학사 정보시스템 구축사업의 사업명은 무엇인가요?",
        "expected": "차세대 포털·학사 정보시스템 구축사업",
        "check_points": ["사업명"]
    },
    {
        "case_id": "TG-RAG-07",
        "doc_key": "korea",
        "source_type": "temp_golden_rag",
        "category": "구축 대상 추출",
        "question": "고려대학교 차세대 포털·학사 정보시스템 구축사업의 구축 대상 시스템 목록을 알려주세요.",
        "expected": "포털, 학사 정보시스템 등",
        "check_points": ["구축 대상", "시스템 목록", "항목 누락 방지"]
    },
    {
        "case_id": "TG-RAG-08",
        "doc_key": "korea",
        "source_type": "temp_golden_rag",
        "category": "금액 단위 보존",
        "question": "고려대학교 차세대 포털·학사 정보시스템 구축사업의 예산은 얼마인가요?",
        "expected": "금액 원문 표기",
        "check_points": ["금액", "원문 표기", "억/조 단위 임의 환산 금지"]
    },
]

temp_golden_rag_cases_df = pd.DataFrame(temp_golden_rag_cases)
temp_golden_rag_cases_df

,case_id,doc_key,source_type,category,question,expected,check_points
0,TG-RAG-01,sejong,temp_golden_rag,기본 정보 추출,세종테크노파크 인사정보 전산시스템 구축 용역의 용역명은 무엇인가요?,세종테크노파크 인사정보 전산시스템 구축 용역,"[용역명, 사업명]"
1,TG-RAG-02,sejong,temp_golden_rag,기간/금액 추출,세종테크노파크 인사정보 전산시스템 구축 용역의 사업기간과 사업비는 어떻게 되나요?,"사업기간, 사업비","[사업기간, 사업비, 금액 원문 표기]"
2,TG-RAG-03,sejong,temp_golden_rag,계약 방식 추출,세종테크노파크 인사정보 전산시스템 구축 용역의 계약방법은 무엇인가요?,계약방법,"[계약방법, 협상, 입찰]"
3,TG-RAG-04,kyunghee,temp_golden_rag,기간/예산 추출,경희대학교 산학협력단 정보시스템 운영 용역의 사업기간과 예산은 어떻게 되나요?,"사업기간, 예산","[사업기간, 예산, 금액 원문 표기]"
4,TG-RAG-05,kyunghee,temp_golden_rag,운영/유지관리 범위,경희대학교 산학협력단 정보시스템 운영 용역의 주요 운영 및 유지관리 범위는 무엇인가요?,정보시스템 운영 및 유지관리 범위,"[운영, 유지관리, 업무 범위]"
5,TG-RAG-06,korea,temp_golden_rag,기본 정보 추출,고려대학교 차세대 포털·학사 정보시스템 구축사업의 사업명은 무엇인가요?,차세대 포털·학사 정보시스템 구축사업,[사업명]
6,TG-RAG-07,korea,temp_golden_rag,구축 대상 추출,고려대학교 차세대 포털·학사 정보시스템 구축사업의 구축 대상 시스템 목록을 알려주세요.,"포털, 학사 정보시스템 등","[구축 대상, 시스템 목록, 항목 누락 방지]"
7,TG-RAG-08,korea,temp_golden_rag,금액 단위 보존,고려대학교 차세대 포털·학사 정보시스템 구축사업의 예산은 얼마인가요?,금액 원문 표기,"[금액, 원문 표기, 억/조 단위 임의 환산 금지]"


In [59]:
temp_golden_rag_models = [
    "exaone3.5:2.4b",
    "gemma3:4b",
    "qwen2.5:3b",
]

temp_golden_rag_results = []

for model in temp_golden_rag_models:
    print(f"임시 Golden QA RAG 테스트 실행 중: {model}")

    for case in temp_golden_rag_cases:
        query = case["question"]
        doc_key = case["doc_key"]

        if model == "gemma3:4b":
            retriever_obj = doc_retrievers_k2[doc_key]
            context_setting = "doc_filtered_k2"
        else:
            retriever_obj = doc_retrievers[doc_key]
            context_setting = "doc_filtered_k5"

        docs = retriever_obj.invoke(query)
        context = build_rag_context(docs)
        prompt = build_rag_qa_prompt(query, context)

        result = ask_ollama(
            prompt=prompt,
            model=model,
            temperature=0.0,
            timeout=180,
            num_predict=512
        )

        temp_golden_rag_results.append({
            "case_id": case["case_id"],
            "doc_key": doc_key,
            "source_type": case["source_type"],
            "category": case["category"],
            "model": model,
            "context_setting": context_setting,
            "question": query,
            "expected": case["expected"],
            "check_points": case["check_points"],
            "success": result["success"],
            "elapsed_time": round(result["elapsed_time"], 2),
            "answer": result["answer"],
            "answer_len": len(result["answer"]),
            "error": result["error"]
        })

temp_golden_rag_results_df = pd.DataFrame(temp_golden_rag_results)

temp_golden_rag_results_df[
    [
        "case_id",
        "doc_key",
        "category",
        "model",
        "context_setting",
        "expected",
        "elapsed_time",
        "answer_len",
        "answer"
    ]
]

임시 Golden QA RAG 테스트 실행 중: exaone3.5:2.4b
임시 Golden QA RAG 테스트 실행 중: gemma3:4b
임시 Golden QA RAG 테스트 실행 중: qwen2.5:3b


,case_id,doc_key,category,model,context_setting,expected,elapsed_time,answer_len,answer
0,TG-RAG-01,sejong,기본 정보 추출,exaone3.5:2.4b,doc_filtered_k5,세종테크노파크 인사정보 전산시스템 구축 용역,5.41,108,세종테크노파크 인사정보 전산시스템 구축 용역의 용역명은 **세종테크노파크 인사정보 ...
1,TG-RAG-02,sejong,기간/금액 추출,exaone3.5:2.4b,doc_filtered_k5,"사업기간, 사업비",2.96,132,세종테크노파크 인사정보 전산시스템 구축 용역의 사업기간은 계약 체결 후 **3개월 ...
2,TG-RAG-03,sejong,계약 방식 추출,exaone3.5:2.4b,doc_filtered_k5,계약방법,3.05,126,세종테크노파크 인사정보 전산시스템 구축 용역의 계약방법은 협상에 의한 계약 체결입니...
3,TG-RAG-04,kyunghee,기간/예산 추출,exaone3.5:2.4b,doc_filtered_k5,"사업기간, 예산",3.80,342,경희대학교 산학협력단 정보시스템 운영 용역의 사업기간과 예산에 대해 답변 드리겠습니...
4,TG-RAG-05,kyunghee,운영/유지관리 범위,exaone3.5:2.4b,doc_filtered_k5,정보시스템 운영 및 유지관리 범위,5.04,791,경희대학교 산학협력단 정보시스템 운영 용역의 주요 운영 및 유지관리 범위는 다음과 ...
5,TG-RAG-06,korea,기본 정보 추출,exaone3.5:2.4b,doc_filtered_k5,차세대 포털·학사 정보시스템 구축사업,2.93,83,고려대학교 차세대 포털·학사 정보시스템 구축사업의 사업명은 **차세대 포털·학사 정...
6,TG-RAG-07,korea,구축 대상 추출,exaone3.5:2.4b,doc_filtered_k5,"포털, 학사 정보시스템 등",3.46,303,고려대학교 차세대 포털·학사 정보시스템 구축사업의 구축 대상 시스템 목록은 다음과 ...
7,TG-RAG-08,korea,금액 단위 보존,exaone3.5:2.4b,doc_filtered_k5,금액 원문 표기,3.08,125,"고려대학교 차세대 포털·학사 정보시스템 구축사업의 예산은 **11,270,000,0..."
8,TG-RAG-01,sejong,기본 정보 추출,gemma3:4b,doc_filtered_k2,세종테크노파크 인사정보 전산시스템 구축 용역,8.37,89,세종테크노파크 인사정보 전산시스템 구축 용역 입찰공고[긴급]입니다.\n\n근거: 사...
9,TG-RAG-02,sejong,기간/금액 추출,gemma3:4b,doc_filtered_k2,"사업기간, 사업비",3.71,98,"사업 기간은 계약 체결 후 3개월 이내이며, 사업 비는 90,000,000원입니다...."


In [61]:
for _, row in temp_golden_rag_results_df.iterrows():
    print("=" * 100)
    print("case_id:", row["case_id"])
    print("doc_key:", row["doc_key"])
    print("model:", row["model"])
    print("context_setting:", row["context_setting"])
    print("category:", row["category"])
    print("question:", row["question"])
    print("expected:", row["expected"])
    print("elapsed_time:", row["elapsed_time"])
    print("answer_len:", row["answer_len"])
    print("\nanswer:")
    print(row["answer"])
    print()

case_id: TG-RAG-01
doc_key: sejong
model: exaone3.5:2.4b
context_setting: doc_filtered_k5
category: 기본 정보 추출
question: 세종테크노파크 인사정보 전산시스템 구축 용역의 용역명은 무엇인가요?
expected: 세종테크노파크 인사정보 전산시스템 구축 용역
elapsed_time: 5.41
answer_len: 108

answer:
세종테크노파크 인사정보 전산시스템 구축 용역의 용역명은 **세종테크노파크 인사정보 전산시스템 구축 용역**입니다.

근거: 사업명 / 세종테크노파크 인사정보 전산시스템 구축 용역 입찰공고[긴급]

case_id: TG-RAG-02
doc_key: sejong
model: exaone3.5:2.4b
context_setting: doc_filtered_k5
category: 기간/금액 추출
question: 세종테크노파크 인사정보 전산시스템 구축 용역의 사업기간과 사업비는 어떻게 되나요?
expected: 사업기간, 사업비
elapsed_time: 2.96
answer_len: 132

answer:
세종테크노파크 인사정보 전산시스템 구축 용역의 사업기간은 계약 체결 후 **3개월 이내**로 명시되어 있습니다. 사업비는 **90,000,000원**입니다.

근거: 사업명 / 세종테크노파크 인사정보 전산시스템 구축 용역 입찰공고[긴급]

case_id: TG-RAG-03
doc_key: sejong
model: exaone3.5:2.4b
context_setting: doc_filtered_k5
category: 계약 방식 추출
question: 세종테크노파크 인사정보 전산시스템 구축 용역의 계약방법은 무엇인가요?
expected: 계약방법
elapsed_time: 3.05
answer_len: 126

answer:
세종테크노파크 인사정보 전산시스템 구축 용역의 계약방법은 협상에 의한 계약 체결입니다. 근거: 세종테크노파크 인사정보 전산시스템 구축

In [60]:
temp_golden_rag_speed_summary_df = (
    temp_golden_rag_results_df
    .groupby(["model", "context_setting"])
    .agg(
        test_count=("case_id", "count"),
        success_count=("success", "sum"),
        avg_elapsed_time=("elapsed_time", "mean"),
        max_elapsed_time=("elapsed_time", "max"),
        min_elapsed_time=("elapsed_time", "min"),
        avg_answer_len=("answer_len", "mean")
    )
    .reset_index()
)

temp_golden_rag_speed_summary_df[
    ["avg_elapsed_time", "max_elapsed_time", "min_elapsed_time", "avg_answer_len"]
] = temp_golden_rag_speed_summary_df[
    ["avg_elapsed_time", "max_elapsed_time", "min_elapsed_time", "avg_answer_len"]
].round(2)

temp_golden_rag_speed_summary_df

,model,context_setting,test_count,success_count,avg_elapsed_time,max_elapsed_time,min_elapsed_time,avg_answer_len
0,exaone3.5:2.4b,doc_filtered_k5,8,8,3.72,5.41,2.93,251.25
1,gemma3:4b,doc_filtered_k2,8,8,4.38,8.37,3.44,118.38
2,qwen2.5:3b,doc_filtered_k5,8,8,4.36,7.08,3.21,208.50


### metadata 금액값 오류 확인 및 context builder 보완

임시 Golden QA 기반 실제 RAG context 테스트 중, 경희대학교 산학협력단 정보시스템 운영 용역 문서에서 일부 모델이 사업예산을 `0원`으로 답변하는 문제가 확인되었습니다. 그러나 문서 원문에는 사업예산이 `금400,000,000원(부가세 포함)`으로 명시되어 있으며, 1차년도와 2차년도 예산도 각각 `금200,000,000원 이내`로 제시되어 있습니다.

이 문제는 문서 metadata의 `사업금액` 값이 `0`으로 저장되어 있었고, context builder가 해당 값을 그대로 LLM 입력에 포함하면서 모델이 이를 실제 예산으로 오인한 사례로 판단하였습니다. 즉, 모델이 문서 본문보다 metadata 값을 우선적으로 신뢰하면서 잘못된 답변이 생성된 것입니다.

따라서 metadata의 사업금액이 `0`, `0.0`, 빈값, 결측값인 경우에는 이를 실제 예산으로 사용하지 않고 `metadata 미확인`으로 처리하도록 context builder를 보완하였습니다. 또한 프롬프트에도 metadata 금액이 미확인 상태일 경우 본문이나 사업요약의 `사업예산`, `사업비`, `추정가격` 표현을 우선 확인하도록 규칙을 추가하였습니다.


In [62]:
def format_amount(value):
    """
    metadata의 사업금액을 원문형 숫자 표기로 변환합니다.
    단, 0 또는 결측값은 실제 사업금액이 아니라 추출 실패일 수 있으므로
    금액으로 단정하지 않고 'metadata 미확인'으로 처리합니다.
    """
    if value is None or value == "":
        return "metadata 미확인"

    try:
        amount = int(float(value))
        if amount == 0:
            return "metadata 미확인"
        return f"{amount:,}원"
    except:
        return str(value)

In [63]:
def build_rag_context(docs, max_content_chars=800):
    """
    retriever 결과를 Generation 프롬프트용 context로 변환합니다.
    metadata의 사업금액이 0 또는 결측값이면 실제 예산으로 단정하지 않습니다.
    """
    context_blocks = []

    for i, doc in enumerate(docs, start=1):
        meta = doc.metadata or {}

        raw_amount = meta.get("사업금액", "")
        formatted_amount = format_amount(raw_amount)

        block = f"""
[검색 결과 {i}]
문서명: {meta.get("file_name", "")}
문서 ID: {meta.get("doc_id", "")}
사업명: {meta.get("사업명", "")}
발주기관: {meta.get("발주기관", "")}
사업유형: {meta.get("사업유형", "")}
사업금액_원문표기: {formatted_amount}
공개일자: {meta.get("공개일자", "")}
입찰참여기간: {meta.get("입찰참여시작일", "")} ~ {meta.get("입찰참여마감일", "")}
문서 위치: {meta.get("header_path", "")}
section_id: {meta.get("section_id", "")}
block_id: {meta.get("block_id", "")}

사업요약:
{meta.get("사업요약", "")}

검색된 본문 일부:
{doc.page_content[:max_content_chars]}
"""
        context_blocks.append(block.strip())

    return "\n\n".join(context_blocks)

In [64]:
def build_rag_qa_prompt(question, context):
    return f"""
당신은 공공 RFP 문서를 분석하는 AI입니다.

아래 검색 결과만 근거로 사용자 질문에 답변하세요.

규칙:
- 문서에 있는 정보만 사용하세요.
- 문서에 없는 내용은 추측하지 마세요.
- 사용자의 질문에 직접 답하세요.
- 금액은 원문 숫자 표기 그대로 사용하세요.
- 금액을 억 원, 만 원, 1억 4천만 원처럼 환산하지 마세요.
- metadata의 사업금액이 `metadata 미확인`, `0`, `0원`이면 이를 실제 예산으로 단정하지 마세요.
- metadata 금액이 미확인인 경우, 본문 또는 사업요약에 명시된 `사업예산`, `사업비`, `추정가격` 표현을 우선 확인하세요.
- 기간은 context에 나온 표현을 그대로 사용하세요. 예: 5개월 이내
- 한국어 존댓말로 간결하게 답변하세요.
- 마지막 줄에는 근거를 `근거: 사업명 / 문서명` 형태로 작성하세요.

검색 결과:
{context}

사용자 질문:
{question}

위 질문에 대한 답변을 작성하세요.
""".strip()

In [65]:
target_case = [case for case in temp_golden_rag_cases if case["case_id"] == "TG-RAG-04"][0]

target_models = [
    "exaone3.5:2.4b",
    "gemma3:4b",
    "qwen2.5:3b",
]

metadata_fix_check_results = []

for model in target_models:
    query = target_case["question"]
    doc_key = target_case["doc_key"]

    if model == "gemma3:4b":
        retriever_obj = doc_retrievers_k2[doc_key]
        context_setting = "doc_filtered_k2"
    else:
        retriever_obj = doc_retrievers[doc_key]
        context_setting = "doc_filtered_k5"

    docs = retriever_obj.invoke(query)
    context = build_rag_context(docs)
    prompt = build_rag_qa_prompt(query, context)

    result = ask_ollama(
        prompt=prompt,
        model=model,
        temperature=0.0,
        timeout=180,
        num_predict=512
    )

    metadata_fix_check_results.append({
        "case_id": target_case["case_id"],
        "model": model,
        "context_setting": context_setting,
        "question": query,
        "expected": target_case["expected"],
        "answer": result["answer"],
        "elapsed_time": round(result["elapsed_time"], 2),
        "answer_len": len(result["answer"]),
        "error": result["error"]
    })

metadata_fix_check_df = pd.DataFrame(metadata_fix_check_results)
metadata_fix_check_df

,case_id,model,context_setting,question,expected,answer,elapsed_time,answer_len,error
0,TG-RAG-04,exaone3.5:2.4b,doc_filtered_k5,경희대학교 산학협력단 정보시스템 운영 용역의 사업기간과 예산은 어떻게 되나요?,"사업기간, 예산",경희대학교 산학협력단 정보시스템 운영 용역의 사업기간과 예산에 대한 구체적인 정보는...,6.03,259,None
1,TG-RAG-04,gemma3:4b,doc_filtered_k2,경희대학교 산학협력단 정보시스템 운영 용역의 사업기간과 예산은 어떻게 되나요?,"사업기간, 예산","경희대학교 산학협력단 정보시스템 운영 용역사업의 추진목표는 최적의 운영상태 유지, ...",9.50,305,None
2,TG-RAG-04,qwen2.5:3b,doc_filtered_k5,경희대학교 산학협력단 정보시스템 운영 용역의 사업기간과 예산은 어떻게 되나요?,"사업기간, 예산",경희대학교 산학협력단 정보시스템 운영 용역의 사업기간과 예산에 대해 현재 공개된 문...,6.17,258,None


In [66]:
for _, row in metadata_fix_check_df.iterrows():
    print("=" * 100)
    print("model:", row["model"])
    print("context_setting:", row["context_setting"])
    print("question:", row["question"])
    print("expected:", row["expected"])
    print("elapsed_time:", row["elapsed_time"])
    print("\nanswer:")
    print(row["answer"])
    print()

model: exaone3.5:2.4b
context_setting: doc_filtered_k5
question: 경희대학교 산학협력단 정보시스템 운영 용역의 사업기간과 예산은 어떻게 되나요?
expected: 사업기간, 예산
elapsed_time: 6.03

answer:
경희대학교 산학협력단 정보시스템 운영 용역의 사업기간과 예산에 대한 구체적인 정보는 제공된 문서들에서 명시되지 않았습니다. 사업금액이 `metadata 미확인` 상태이며, 사업기간 역시 `<unknown> ~ <unknown>`으로 표시되어 있어 정확한 답변을 드리기 어렵습니다. 따라서 현재로서는 사업기간과 예산에 대한 확실한 답변을 제공할 수 없습니다.

근거: 사업명 / 문서명: [입찰공고] 산학협력단 정보시스템 운영 용역업체 선정 / 문서 ID: D014

model: gemma3:4b
context_setting: doc_filtered_k2
question: 경희대학교 산학협력단 정보시스템 운영 용역의 사업기간과 예산은 어떻게 되나요?
expected: 사업기간, 예산
elapsed_time: 9.5

answer:
경희대학교 산학협력단 정보시스템 운영 용역사업의 추진목표는 최적의 운영상태 유지, 서비스 연속성 향상, 사업의 성공적인 수행입니다. 다만, 본 사업은 『소프트웨어사업 관리감독에 관한 일반기준』 제4조의 2 및 제6조에 의거 소프트웨어 유지관리 사업으로 적정 사업기간 산정 기준을 적용하지 않음 (근거: 문서 ID: D014 / 사업명: [입찰공고] 산학협력단 정보시스템 운영 용역업체 선정). 사업금액_원문표기 정보는 metadata 미확인입니다. (근거: 문서 ID: D014 / 사업명: [입찰공고] 산학협력단 정보시스템 운영 용역업체 선정)

model: qwen2.5:3b
context_setting: doc_filtered_k5
question: 경희대학교 산학협력단 정보시스템 운영 용역의 사업기간과 예산은 어떻게 되나요?
expected: 사업기간, 예산
el

### metadata 보완 후 TG-RAG-04 재실험 결과

metadata의 `사업금액` 값이 `0`인 경우 이를 실제 예산으로 단정하지 않도록 context builder와 프롬프트를 보완한 뒤, 문제가 발생했던 `TG-RAG-04` 케이스를 재실험하였습니다.

재실험 결과, 이전처럼 `사업금액이 0원`이라고 단정하는 오답은 발생하지 않았습니다. 이는 metadata의 0 값을 실제 예산으로 사용하지 않도록 처리한 효과로 볼 수 있습니다.

다만 세 모델 모두 경희대학교 산학협력단 정보시스템 운영 용역의 실제 사업기간과 예산을 정확히 답변하지 못했습니다. `exaone3.5:2.4b`와 `qwen2.5:3b`는 정보가 명시되어 있지 않다고 답변하였고, `gemma3:4b`는 유지관리 사업기간 산정 기준과 추진목표를 중심으로 답변하였습니다.

따라서 이 결과는 metadata 오류값 보정은 성공했지만, 해당 질문에 필요한 사업기간과 예산 정보가 retriever 검색 결과에 충분히 포함되지 않았을 가능성을 보여줍니다. 즉, TG-RAG-04는 모델 생성 문제만이 아니라 검색 질의와 retrieval 결과 품질을 함께 점검해야 하는 사례로 판단하였습니다.


In [67]:
# TG-RAG-04에서 실제로 검색된 context 확인
target_case = [case for case in temp_golden_rag_cases if case["case_id"] == "TG-RAG-04"][0]

query = target_case["question"]
doc_key = target_case["doc_key"]

docs = doc_retrievers[doc_key].invoke(query)
context = build_rag_context(docs)

print(context[:6000])

[검색 결과 1]
문서명: 경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp
문서 ID: D014
사업명: [입찰공고] 산학협력단 정보시스템 운영 용역업체 선정
발주기관: 경희대학교
사업유형: 운영
사업금액_원문표기: metadata 미확인
공개일자: 2024-05-02 00:00:00
입찰참여기간: <unknown> ~ <unknown>
문서 위치: Ⅵ. 제안서 관련서식 > 2. 입찰․계약체결 및 계약이행 과정에서 관계교직원에게 직․간접적으로 금품․향응 등의 부당한 이익을 제공하지 않겠습니다.
section_id: S52
block_id: S52-B28

사업요약:
- 사업명: 경희대학교 산학협력단 정보시스템 운영 용역사업
- 추진배경: 정보시스템 운영 효율화 및 안정적인 서비스 제공을 위한 최적상태 유지
- 사업범위: 정보시스템 운영 및 유지관리, 내/외부 연계시스템 운영/유지관리, 정보시스템 개발/운영 관련 제반 업무
- 기대효과: 사용자 편의성 향상, 신속한 요구사항 처리 및 시스템 반영, 연계시스템 관련 지원 및 개발 등
- 추진목표: 최적의 운영상태 유지, 서비스 연속성 향상, 사업의 성공적인 수행

검색된 본문 일부:
[사업명] [입찰공고] 산학협력단 정보시스템 운영 용역업체 선정
[발주기관] 경희대학교
[요약] - 사업명: 경희대학교 산학협력단 정보시스템 운영 용역사업
- 추진배경: 정보시스템 운영 효율화 및 안정적인 서비스 제공을 위한 최적상태 유지
- 사업범위: 정보시스템 운영 및 유지관리, 내/외부 연계시스템 운영/유지관리, 정보시스템 개발/운영 관련 제반 업무
- 기대효과: 사용자 편의성 향상, 신속한 요구사항 처리 및 시스템 반영, 연계시스템 관련 지원 및 개발 등
- 추진목표: 최적의 운영상태 유지, 서비스 연속성 향상, 사업의 성공적인 수행

[섹션: Ⅵ. 제안서 관련서식 > 2. 입찰․계약체결 및 계약이행 과정에서 관계교직원에게 직․간접적으로 금품․향응 등의 부당한 이익을 제공하지 않겠습니다.]

| 재정상황(

In [68]:
# D014 경희대학교 문서 전체 chunk에서 예산/기간 관련 키워드 검색

kyunghee_data = vectorstore.get(
    where={"doc_id": "D014"},
    include=["documents", "metadatas"]
)

keywords = [
    "400,000,000",
    "200,000,000",
    "2026.04.30",
    "2026. 04. 30",
    "사업예산",
    "예산",
    "사업기간",
    "계약체결일",
    "부가세",
    "부가가치세"
]

matched_chunks = []

for doc_text, meta in zip(kyunghee_data["documents"], kyunghee_data["metadatas"]):
    text = doc_text or ""
    hit_keywords = [kw for kw in keywords if kw in text]
    
    if hit_keywords:
        matched_chunks.append({
            "hit_keywords": hit_keywords,
            "section_id": meta.get("section_id"),
            "block_id": meta.get("block_id"),
            "header_path": meta.get("header_path"),
            "text": text[:1500]
        })

print("매칭 chunk 수:", len(matched_chunks))

for i, item in enumerate(matched_chunks[:20], start=1):
    print("=" * 100)
    print(f"[매칭 결과 {i}]")
    print("hit_keywords:", item["hit_keywords"])
    print("section_id:", item["section_id"])
    print("block_id:", item["block_id"])
    print("header_path:", item["header_path"])
    print("\ntext:")
    print(item["text"])

매칭 chunk 수: 9
[매칭 결과 1]
hit_keywords: ['사업기간']
section_id: S05
block_id: S05-B20
header_path: Ⅰ. 개요 > 6. 과업내용 > 2) 운영/유지관리 업무내역

text:
[사업명] [입찰공고] 산학협력단 정보시스템 운영 용역업체 선정
[발주기관] 경희대학교
[요약] - 사업명: 경희대학교 산학협력단 정보시스템 운영 용역사업
- 추진배경: 정보시스템 운영 효율화 및 안정적인 서비스 제공을 위한 최적상태 유지
- 사업범위: 정보시스템 운영 및 유지관리, 내/외부 연계시스템 운영/유지관리, 정보시스템 개발/운영 관련 제반 업무
- 기대효과: 사용자 편의성 향상, 신속한 요구사항 처리 및 시스템 반영, 연계시스템 관련 지원 및 개발 등
- 추진목표: 최적의 운영상태 유지, 서비스 연속성 향상, 사업의 성공적인 수행

[섹션: Ⅰ. 개요 > 6. 과업내용 > 2) 운영/유지관리 업무내역]

가) 정보시스템 상시 운영/유지보수

- 사용자(학생, 교원, 직원 등)의 요구사항 처리 및 시스템 반영

- 정보시스템 업무범위에 대한 자료처리 및 시스템 오류에 대한 보수

나) 정보시스템을 사용하는 산학협력단 업무의 일정별 운영 지원

다) 연계시스템과의 연동관련 지원 및 개발

- 연동 업무관련 상시운영/유지보수

- 소프트웨어 변경, DATA변경, 기타 운영환경의 변경에 따른 시스템간 연동지원 및 유지보수

- 연계시스템 추가 도입 시 시스템간 연동관련 지원 및 개발

라) 사업기간 내 산학협력단이 인정하여 신규 또는 추가 개발된 부분의 운영/유지관리

마) 법규/제도/양식/절차/조직 등 업무처리 절차상의 변경으로 인하여 발생하는 응용프로그램의 변경 또는 신규 개발

바) 기능 보완 및 UI 개선, 성능 개선, 데이터의 보안성 및 정확도 개선, 사용자 편의성 개선 등을 위한 유지보수

사) 정보시스템 관련 개발/운영 환경 및 개발플랫폼의 최적화 관리

아) 정보시스템 최적 상태 유지를 위한 선제적 예방 

In [69]:
# D014 경희대학교 문서 전체 chunk에서 더 넓은 키워드로 재검색

kyunghee_data = vectorstore.get(
    where={"doc_id": "D014"},
    include=["documents", "metadatas"]
)

keywords = [
    "400",
    "400,000",
    "400000",
    "400,000,000",
    "금400",
    "금 400",
    "200,000",
    "200000",
    "200,000,000",
    "금200",
    "금 200",
    "2026",
    "2026.04",
    "2026. 04",
    "2026.04.30",
    "2026. 04. 30",
    "계약체결",
    "계약 체결",
    "용역기간",
    "사업기간",
    "사업예산",
    "예산액",
    "부가세 포함",
    "부가가치세 포함",
]

matched_chunks_wide = []

for doc_text, meta in zip(kyunghee_data["documents"], kyunghee_data["metadatas"]):
    text = doc_text or ""
    hit_keywords = [kw for kw in keywords if kw in text]
    
    if hit_keywords:
        matched_chunks_wide.append({
            "hit_keywords": hit_keywords,
            "section_id": meta.get("section_id"),
            "block_id": meta.get("block_id"),
            "header_path": meta.get("header_path"),
            "text": text[:2000]
        })

print("넓은 키워드 매칭 chunk 수:", len(matched_chunks_wide))

for i, item in enumerate(matched_chunks_wide[:30], start=1):
    print("=" * 100)
    print(f"[매칭 결과 {i}]")
    print("hit_keywords:", item["hit_keywords"])
    print("section_id:", item["section_id"])
    print("block_id:", item["block_id"])
    print("header_path:", item["header_path"])
    print("\ntext:")
    print(item["text"])

넓은 키워드 매칭 chunk 수: 26
[매칭 결과 1]
hit_keywords: ['사업기간']
section_id: S05
block_id: S05-B20
header_path: Ⅰ. 개요 > 6. 과업내용 > 2) 운영/유지관리 업무내역

text:
[사업명] [입찰공고] 산학협력단 정보시스템 운영 용역업체 선정
[발주기관] 경희대학교
[요약] - 사업명: 경희대학교 산학협력단 정보시스템 운영 용역사업
- 추진배경: 정보시스템 운영 효율화 및 안정적인 서비스 제공을 위한 최적상태 유지
- 사업범위: 정보시스템 운영 및 유지관리, 내/외부 연계시스템 운영/유지관리, 정보시스템 개발/운영 관련 제반 업무
- 기대효과: 사용자 편의성 향상, 신속한 요구사항 처리 및 시스템 반영, 연계시스템 관련 지원 및 개발 등
- 추진목표: 최적의 운영상태 유지, 서비스 연속성 향상, 사업의 성공적인 수행

[섹션: Ⅰ. 개요 > 6. 과업내용 > 2) 운영/유지관리 업무내역]

가) 정보시스템 상시 운영/유지보수

- 사용자(학생, 교원, 직원 등)의 요구사항 처리 및 시스템 반영

- 정보시스템 업무범위에 대한 자료처리 및 시스템 오류에 대한 보수

나) 정보시스템을 사용하는 산학협력단 업무의 일정별 운영 지원

다) 연계시스템과의 연동관련 지원 및 개발

- 연동 업무관련 상시운영/유지보수

- 소프트웨어 변경, DATA변경, 기타 운영환경의 변경에 따른 시스템간 연동지원 및 유지보수

- 연계시스템 추가 도입 시 시스템간 연동관련 지원 및 개발

라) 사업기간 내 산학협력단이 인정하여 신규 또는 추가 개발된 부분의 운영/유지관리

마) 법규/제도/양식/절차/조직 등 업무처리 절차상의 변경으로 인하여 발생하는 응용프로그램의 변경 또는 신규 개발

바) 기능 보완 및 UI 개선, 성능 개선, 데이터의 보안성 및 정확도 개선, 사용자 편의성 개선 등을 위한 유지보수

사) 정보시스템 관련 개발/운영 환경 및 개발플랫폼의 최적화 관리

아) 정보시스템 최적 상태 유지를 위한

In [70]:
# D014 문서의 전체 chunk 위치 확인
kyunghee_chunk_index = []

for doc_text, meta in zip(kyunghee_data["documents"], kyunghee_data["metadatas"]):
    kyunghee_chunk_index.append({
        "section_id": meta.get("section_id"),
        "block_id": meta.get("block_id"),
        "header_path": meta.get("header_path"),
        "text_preview": (doc_text or "")[:120]
    })

kyunghee_chunk_index_df = pd.DataFrame(kyunghee_chunk_index)

pd.set_option("display.max_colwidth", None)
kyunghee_chunk_index_df[
    ["section_id", "block_id", "header_path", "text_preview"]
].head(80)

,section_id,block_id,header_path,text_preview
0,S01,S01-B04,(서두),[사업명] [입찰공고] 산학협력단 정보시스템 운영 용역업체 선정\n[발주기관] 경희대학교\n[요약] - 사업명: 경희대학교 산학협력단 정보시스템 운영 용역사업\n- 추진배경: 정보시스템 운영 효율화 및 안정적인 서비스 제
1,S01,S01-B05,(서두),[사업명] [입찰공고] 산학협력단 정보시스템 운영 용역업체 선정\n[발주기관] 경희대학교\n[요약] - 사업명: 경희대학교 산학협력단 정보시스템 운영 용역사업\n- 추진배경: 정보시스템 운영 효율화 및 안정적인 서비스 제
2,S01,S01-B06,(서두),[사업명] [입찰공고] 산학협력단 정보시스템 운영 용역업체 선정\n[발주기관] 경희대학교\n[요약] - 사업명: 경희대학교 산학협력단 정보시스템 운영 용역사업\n- 추진배경: 정보시스템 운영 효율화 및 안정적인 서비스 제
3,S03,S03-B02,Ⅰ. 개요 > 5. 일반내용 > 1) 정보시스템 운영 및 유지관리,[사업명] [입찰공고] 산학협력단 정보시스템 운영 용역업체 선정\n[발주기관] 경희대학교\n[요약] - 사업명: 경희대학교 산학협력단 정보시스템 운영 용역사업\n- 추진배경: 정보시스템 운영 효율화 및 안정적인 서비스 제
4,S04,S04-B01,Ⅰ. 개요 > 6. 과업내용 > 1) 정보시스템 현황,[사업명] [입찰공고] 산학협력단 정보시스템 운영 용역업체 선정\n[발주기관] 경희대학교\n[요약] - 사업명: 경희대학교 산학협력단 정보시스템 운영 용역사업\n- 추진배경: 정보시스템 운영 효율화 및 안정적인 서비스 제
...,...,...,...,...
75,S47,S47-B04,Ⅵ. 제안서 관련서식 > 2. 보증기간의 만료일 : 입찰서 제출마감일 다음날부터 30일 이후일 것,[사업명] [입찰공고] 산학협력단 정보시스템 운영 용역업체 선정\n[발주기관] 경희대학교\n[요약] - 사업명: 경희대학교 산학협력단 정보시스템 운영 용역사업\n- 추진배경: 정보시스템 운영 효율화 및 안정적인 서비스 제
76,S48,S48-B09,Ⅵ. 제안서 관련서식 > 4. 기타 임ㆍ직원 여부를 확인할 수 있는 자료,[사업명] [입찰공고] 산학협력단 정보시스템 운영 용역업체 선정\n[발주기관] 경희대학교\n[요약] - 사업명: 경희대학교 산학협력단 정보시스템 운영 용역사업\n- 추진배경: 정보시스템 운영 효율화 및 안정적인 서비스 제
77,S48,S48-B15,Ⅵ. 제안서 관련서식 > 4. 기타 임ㆍ직원 여부를 확인할 수 있는 자료,[사업명] [입찰공고] 산학협력단 정보시스템 운영 용역업체 선정\n[발주기관] 경희대학교\n[요약] - 사업명: 경희대학교 산학협력단 정보시스템 운영 용역사업\n- 추진배경: 정보시스템 운영 효율화 및 안정적인 서비스 제
78,S49,S49-B01,Ⅵ. 제안서 관련서식 > 4. 담합하거나 타인의 경쟁참가를 방해 또는 관계계약담당자의 공무집행을 방해한 자의 입찰,[사업명] [입찰공고] 산학협력단 정보시스템 운영 용역업체 선정\n[발주기관] 경희대학교\n[요약] - 사업명: 경희대학교 산학협력단 정보시스템 운영 용역사업\n- 추진배경: 정보시스템 운영 효율화 및 안정적인 서비스 제


In [71]:
# 전체 문서 기준 metadata 품질 점검

all_data = vectorstore.get(include=["documents", "metadatas"])

metadata_audit = {}

for doc_text, meta in zip(all_data["documents"], all_data["metadatas"]):
    doc_id = meta.get("doc_id")
    
    if doc_id not in metadata_audit:
        raw_amount = meta.get("사업금액", None)
        
        try:
            amount_value = int(float(raw_amount)) if raw_amount not in [None, ""] else None
        except:
            amount_value = None
        
        metadata_audit[doc_id] = {
            "doc_id": doc_id,
            "file_name": meta.get("file_name"),
            "사업명": meta.get("사업명"),
            "발주기관": meta.get("발주기관"),
            "사업유형": meta.get("사업유형"),
            "사업금액": raw_amount,
            "사업금액_숫자": amount_value,
            "입찰참여시작일": meta.get("입찰참여시작일"),
            "입찰참여마감일": meta.get("입찰참여마감일"),
            "chunk_count": 0,
        }
    
    metadata_audit[doc_id]["chunk_count"] += 1

metadata_audit_df = pd.DataFrame(metadata_audit.values())

metadata_audit_df.head()

,doc_id,file_name,사업명,발주기관,사업유형,사업금액,사업금액_숫자,입찰참여시작일,입찰참여마감일,chunk_count
0,D001,한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp,한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화,한영대학,고도화,130000000.0,130000000,<unknown>,2024-10-15 17:00:00,153
1,D002,한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp,2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선,한국연구재단,개선,129300000.0,129300000,2024-10-14 10:00:00,2024-10-16 14:00:00,193
2,D003,한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp,EIP3.0 고압가스 안전관리 시스템 구축 용역,한국생산기술연구원,구축,40000000.0,40000000,2024-08-29 09:00:00,2024-09-09 10:00:00,217
3,D004,인천광역시_도시계획위원회 통합관리시스템 구축용역.hwp,도시계획위원회 통합관리시스템 구축용역,인천광역시,통합,150000000.0,150000000,2024-05-02 10:00:00,2024-05-09 16:00:00,169
4,D005,경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp,봉화군 재난통합관리시스템 고도화 사업(협상)(긴급),경상북도 봉화군,고도화,900000000.0,900000000,2024-04-26 09:00:00,2024-04-30 17:00:00,161


In [72]:
# 사업금액이 0, 결측, 미확인인 문서 확인

amount_problem_df = metadata_audit_df[
    metadata_audit_df["사업금액_숫자"].isna() |
    (metadata_audit_df["사업금액_숫자"] == 0)
].copy()

amount_problem_df[
    [
        "doc_id",
        "file_name",
        "사업명",
        "발주기관",
        "사업금액",
        "사업금액_숫자",
        "chunk_count"
    ]
].sort_values("doc_id")

,doc_id,file_name,사업명,발주기관,사업금액,사업금액_숫자,chunk_count
12,D013,서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf,[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차 고도화 용역,서울시립대학교,0.0,0,451
13,D014,경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp,[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정,경희대학교,0.0,0,100
16,D017,한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp,예약발매시스템 개량 ISMP 용역,한국철도공사 (용역),0.0,0,197
33,D035,한국철도공사 (용역)_모바일오피스 시스템 고도화 용역(총체 및 1차).hwp,모바일오피스 시스템 고도화 용역(총체 및 1차),한국철도공사 (용역),0.0,0,183
40,D042,을지대학교_을지대학교 비교과시스템 개발.hwp,을지대학교 비교과시스템 개발,을지대학교,0.0,0,137
56,D059,대한상공회의소_기업 재생에너지 지원센터 홈페이지 개편 및 시스템 고.hwp,기업 재생에너지 지원센터 홈페이지 개편 및 시스템 고도화 사업,대한상공회의소,0.0,0,150
68,D071,한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp,[재공고][긴급][협상형]운행정보기록 자동분석시스템 개량,한국철도공사 (용역),0.0,0,235


In [73]:
# 입찰참여기간이 unknown인 문서 확인

period_problem_df = metadata_audit_df[
    metadata_audit_df["입찰참여시작일"].astype(str).str.contains("unknown|None|nan", case=False, na=True) |
    metadata_audit_df["입찰참여마감일"].astype(str).str.contains("unknown|None|nan", case=False, na=True)
].copy()

period_problem_df[
    [
        "doc_id",
        "file_name",
        "사업명",
        "발주기관",
        "입찰참여시작일",
        "입찰참여마감일",
        "chunk_count"
    ]
].sort_values("doc_id")

,doc_id,file_name,사업명,발주기관,입찰참여시작일,입찰참여마감일,chunk_count
0,D001,한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp,한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화,한영대학,<unknown>,2024-10-15 17:00:00,153
12,D013,서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf,[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차 고도화 용역,서울시립대학교,<unknown>,<unknown>,451
13,D014,경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp,[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정,경희대학교,<unknown>,<unknown>,100
14,D015,한국수자원공사_건설통합시스템(CMS) 고도화.hwp,건설통합시스템(CMS) 고도화,한국수자원공사,<unknown>,<unknown>,172
17,D018,2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp,2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역,2025 구미 아시아육상경기선수권대회 조직위원회,<unknown>,2024-09-23 18:00:00,125
23,D024,수협중앙회_강릉어선안전조업국 상황관제시스템 구축.hwp,강릉어선안전조업국 상황관제시스템 구축,수협중앙회,<unknown>,2025-03-10 11:00:00,156
24,D025,한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp,용인 첨단 시스템반도체 국가산단 용수공급사업 타당성조사 및 기본계획 수립 용역,한국수자원공사,<unknown>,<unknown>,95
25,D027,KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp,[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 국회 방송시스템 구축 및 지역의회 연계 개선 PMC 용역,KOICA 전자조달,<unknown>,<unknown>,242
30,D032,인천광역시 동구_수도국산달동네박물관 전시해설 시스템 구축(협상에 .hwp,수도국산달동네박물관 전시해설 시스템 구축(협상에 의한 계약),인천광역시 동구,<unknown>,2025-02-20 18:00:00,159
37,D039,광주과학기술원_학사시스템 기능개선 사업.hwp,학사시스템 기능개선 사업,광주과학기술원,<unknown>,2024-12-20 14:00:00,123


In [74]:
import re

amount_pattern = re.compile(
    r"(\d{1,3}(,\d{3})+|\d+)\s*원|금\s*\d|사업예산|사업비|추정가격|부가세|부가가치세"
)

date_pattern = re.compile(
    r"\d{4}\.\s*\d{1,2}\.\s*\d{1,2}|계약\s*체결|계약체결|사업기간|용역기간|계약기간"
)

doc_text_audit = {}

for doc_text, meta in zip(all_data["documents"], all_data["metadatas"]):
    doc_id = meta.get("doc_id")
    text = doc_text or ""
    
    if doc_id not in doc_text_audit:
        doc_text_audit[doc_id] = {
            "doc_id": doc_id,
            "file_name": meta.get("file_name"),
            "사업명": meta.get("사업명"),
            "발주기관": meta.get("발주기관"),
            "has_amount_like_text": False,
            "has_date_like_text": False,
            "amount_hit_count": 0,
            "date_hit_count": 0,
        }
    
    if amount_pattern.search(text):
        doc_text_audit[doc_id]["has_amount_like_text"] = True
        doc_text_audit[doc_id]["amount_hit_count"] += 1
    
    if date_pattern.search(text):
        doc_text_audit[doc_id]["has_date_like_text"] = True
        doc_text_audit[doc_id]["date_hit_count"] += 1

doc_text_audit_df = pd.DataFrame(doc_text_audit.values())

doc_quality_audit_df = metadata_audit_df.merge(
    doc_text_audit_df[
        [
            "doc_id",
            "has_amount_like_text",
            "has_date_like_text",
            "amount_hit_count",
            "date_hit_count"
        ]
    ],
    on="doc_id",
    how="left"
)

doc_quality_audit_df[
    [
        "doc_id",
        "file_name",
        "사업명",
        "사업금액",
        "사업금액_숫자",
        "has_amount_like_text",
        "amount_hit_count",
        "has_date_like_text",
        "date_hit_count"
    ]
].sort_values("doc_id")

,doc_id,file_name,사업명,사업금액,사업금액_숫자,has_amount_like_text,amount_hit_count,has_date_like_text,date_hit_count
0,D001,한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp,한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화,1.300000e+08,130000000,True,1,True,114
1,D002,한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp,2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선,1.293000e+08,129300000,True,6,True,13
2,D003,한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp,EIP3.0 고압가스 안전관리 시스템 구축 용역,4.000000e+07,40000000,True,3,True,43
3,D004,인천광역시_도시계획위원회 통합관리시스템 구축용역.hwp,도시계획위원회 통합관리시스템 구축용역,1.500000e+08,150000000,True,4,True,31
4,D005,경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp,봉화군 재난통합관리시스템 고도화 사업(협상)(긴급),9.000000e+08,900000000,True,3,True,13
...,...,...,...,...,...,...,...,...,...
92,D095,대한적십자사 의료원_적십자병원 병원정보 재해복구시스템 구축 용역 .hwp,적십자병원 병원정보 재해복구시스템 구축 용역 재공고입찰,5.000000e+08,500000000,True,4,True,11
93,D096,한국수자원조사기술원_수문자료정보관리시스템(HDIMS) 재구축 용역(3단계.hwp,수문자료정보관리시스템(HDIMS) 재구축 용역(3단계),4.932704e+08,493270440,True,2,True,13
94,D097,(재)예술경영지원센터_통합 정보시스템 구축 사전 컨설팅.hwp,통합 정보시스템 구축 사전 컨설팅,5.000000e+07,50000000,True,3,True,16
95,D098,재단법인 광주광역시 광주문화재단_2024년 광주문화예술통합플랫폼 시스.hwp,2024년 광주문화예술통합플랫폼 시스템 유지 및 보수 용역 제안서 제출 안내 재공고,6.189000e+07,61890000,True,9,True,17


In [75]:
# metadata는 문제인데 본문에는 금액/기간 표현이 있는 문서
# → metadata 추출 실패 가능성이 큰 문서

suspicious_metadata_df = doc_quality_audit_df[
    (
        doc_quality_audit_df["사업금액_숫자"].isna() |
        (doc_quality_audit_df["사업금액_숫자"] == 0)
    ) &
    (doc_quality_audit_df["has_amount_like_text"] == True)
].copy()

suspicious_metadata_df[
    [
        "doc_id",
        "file_name",
        "사업명",
        "발주기관",
        "사업금액",
        "사업금액_숫자",
        "amount_hit_count",
        "date_hit_count"
    ]
].sort_values(["amount_hit_count", "date_hit_count"], ascending=False)

,doc_id,file_name,사업명,발주기관,사업금액,사업금액_숫자,amount_hit_count,date_hit_count
12,D013,서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf,[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차 고도화 용역,서울시립대학교,0.0,0,10,35
68,D071,한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp,[재공고][긴급][협상형]운행정보기록 자동분석시스템 개량,한국철도공사 (용역),0.0,0,7,19
33,D035,한국철도공사 (용역)_모바일오피스 시스템 고도화 용역(총체 및 1차).hwp,모바일오피스 시스템 고도화 용역(총체 및 1차),한국철도공사 (용역),0.0,0,6,18
13,D014,경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp,[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정,경희대학교,0.0,0,5,24
16,D017,한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp,예약발매시스템 개량 ISMP 용역,한국철도공사 (용역),0.0,0,5,20
40,D042,을지대학교_을지대학교 비교과시스템 개발.hwp,을지대학교 비교과시스템 개발,을지대학교,0.0,0,4,12
56,D059,대한상공회의소_기업 재생에너지 지원센터 홈페이지 개편 및 시스템 고.hwp,기업 재생에너지 지원센터 홈페이지 개편 및 시스템 고도화 사업,대한상공회의소,0.0,0,1,16


---

In [ ]:
from pathlib import Path
import tarfile
import time
import chromadb

# chroma_db.tar.gz가 있는 프로젝트 폴더 경로로 수정
PROJECT_DIR = Path(r"여기에_프로젝트_폴더_경로")

ARCHIVE_PATH = PROJECT_DIR / "chroma_db.tar.gz"
EXTRACT_DIR = PROJECT_DIR / "data" / f"fresh_chroma_test_{int(time.time())}"

print("ARCHIVE_PATH:", ARCHIVE_PATH)
print("압축파일 존재:", ARCHIVE_PATH.exists())

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with tarfile.open(ARCHIVE_PATH, "r:gz") as tar_ref:
    tar_ref.extractall(EXTRACT_DIR)

print("압축 해제 위치:", EXTRACT_DIR)

sqlite_paths = list(EXTRACT_DIR.rglob("chroma.sqlite3"))
print("chroma.sqlite3 개수:", len(sqlite_paths))

for p in sqlite_paths:
    print("찾은 DB:", p)

if len(sqlite_paths) == 0:
    raise FileNotFoundError("chroma.sqlite3를 찾지 못했습니다.")

CHROMA_DIR = sqlite_paths[0].parent
print("CHROMA_DIR:", CHROMA_DIR)

client = chromadb.PersistentClient(path=str(CHROMA_DIR))

collections = client.list_collections()
print("collections:", collections)

for col in collections:
    print("collection name:", col.name)

collection = client.get_collection("rfp_docs")

try:
    print("count:", collection.count())
    print("count 성공")
except Exception as e:
    print("count 실패")
    print(type(e).__name__)
    print(e)

ARCHIVE_PATH: C:\Users\User\Desktop\[AI]스프린트미션\프로젝트 2번\chroma_db.tar.gz
압축파일 존재: True


C:\Users\User\AppData\Local\Temp\ipykernel_30112\2578348771.py:18: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar_ref.extractall(EXTRACT_DIR)


압축 해제 위치: C:\Users\User\Desktop\[AI]스프린트미션\프로젝트 2번\data\fresh_chroma_test_1782200267
chroma.sqlite3 개수: 1
찾은 DB: C:\Users\User\Desktop\[AI]스프린트미션\프로젝트 2번\data\fresh_chroma_test_1782200267\chroma_db\chroma.sqlite3
CHROMA_DIR: C:\Users\User\Desktop\[AI]스프린트미션\프로젝트 2번\data\fresh_chroma_test_1782200267\chroma_db
collections: [Collection(name=rfp_docs)]
collection name: rfp_docs
count 실패
InternalError
Error executing plan: Error sending backfill request to compactor: Error constructing hnsw segment reader: Error creating hnsw segment reader: Error loading hnsw index
